<a href="https://colab.research.google.com/github/mhirschberg/competitive_vsibility_audit_bd/blob/main/competitive_visibility_audit_bd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1. Install dependencies
#@markdown Run this cell once after opening the notebook.

!pip install -q \
    requests \
    pydantic \
    pandas \
    rich \
    tldextract \
    json-repair \
    nest_asyncio

print("✓ Dependencies installed")


In [ ]:
#@title 2. Configure the audit
#@markdown Enter the company you want to analyze.
#@markdown
#@markdown The domain can be entered as:
#@markdown - `example.com`
#@markdown - `www.example.com`
#@markdown - `https://www.example.com/`
#@markdown
#@markdown Your Bright Data API token is read from the Colab secret named `BRIGHTDATA_API_TOKEN`.

COMPANY_NAME = "Rayner" #@param {type:"string"}

COMPANY_DOMAIN = "rayner.com" #@param {type:"string"}

AUDIT_FOCUS = "RayOne Galaxy" #@param {type:"string"}

COUNTRY = "US"
SEARCH_ENGINE = "auto" # @param ["auto", "google", "bing", "none"]

SERP_ZONE = "serp_api2" #@param {type:"string"}

AUTO_DOWNLOAD_REPORT = False #@param {type:"boolean"}

DEBUG_MODE = True #@param {type:"boolean"}


# ============================================================
# Validate configuration
# ============================================================

from urllib.parse import urlparse
from google.colab import userdata


def read_colab_secret(name):
    try:
        value = userdata.get(name)
        return value.strip() if value else ""
    except Exception:
        return ""


BRIGHTDATA_API_TOKEN = read_colab_secret(
    "BRIGHTDATA_API_TOKEN"
)

COMPANY_NAME = COMPANY_NAME.strip()
COMPANY_DOMAIN = COMPANY_DOMAIN.strip()
COUNTRY = COUNTRY.strip().upper()
SERP_ZONE = SERP_ZONE.strip()


if not BRIGHTDATA_API_TOKEN:
    raise ValueError(
        "The BRIGHTDATA_API_TOKEN Colab secret is missing. "
        "Add it through the key icon in the Colab sidebar and "
        "enable notebook access."
    )

if not COMPANY_NAME:
    raise ValueError(
        "COMPANY_NAME cannot be empty."
    )

if not COMPANY_DOMAIN:
    raise ValueError(
        "COMPANY_DOMAIN cannot be empty."
    )

if not COUNTRY:
    raise ValueError(
        "COUNTRY cannot be empty."
    )

if not SERP_ZONE:
    raise ValueError(
        "SERP_ZONE cannot be empty."
    )


# Accept either a domain or complete URL.
if not COMPANY_DOMAIN.lower().startswith(
    ("http://", "https://")
):
    COMPANY_URL = (
        f"https://{COMPANY_DOMAIN}"
    )
else:
    COMPANY_URL = COMPANY_DOMAIN


parsed_company_url = urlparse(
    COMPANY_URL
)

if not parsed_company_url.hostname:
    raise ValueError(
        f"Invalid company domain or URL: "
        f"{COMPANY_DOMAIN}"
    )


COMPANY_HOSTNAME = (
    parsed_company_url.hostname
    .lower()
    .removeprefix("www.")
)

COMPANY_URL = (
    f"{parsed_company_url.scheme or 'https'}://"
    f"{parsed_company_url.hostname}/"
)


# ============================================================
# Final user-facing configuration object
# ============================================================

AUDIT_SETTINGS = {
    "company_name": COMPANY_NAME,
    "company_url": COMPANY_URL,
    "company_domain": COMPANY_HOSTNAME,
    "audit_focus": AUDIT_FOCUS.strip(),
    "country": COUNTRY,
    "search_engine": SEARCH_ENGINE,
    "serp_zone": SERP_ZONE,
    "auto_download": (
        AUTO_DOWNLOAD_REPORT
    ),
    "debug": DEBUG_MODE,
}


print("✓ Audit configured")
print()
print(f"Company: {COMPANY_NAME}")
print(f"Website: {COMPANY_URL}")
print(f"Country: {COUNTRY}")
print(f"Audit focus: {AUDIT_FOCUS.strip() or 'Primary offering'}")
print(f"SERP zone: {SERP_ZONE}")
print(
    f"Debug logging: "
    f"{'enabled' if DEBUG_MODE else 'disabled'}"
)


In [ ]:
#@title 3A. Load core audit engine
#@markdown Internal API, reliable snapshot handling, parsing, models, and SERP aggregation.
#@markdown This cell normally does not need to be edited.

import os
import re
import json
import time
import shutil
import asyncio
import zipfile

from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import (
    quote_plus,
    urlparse,
    urlunparse,
)

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed,
)
from collections import (
    Counter,
    defaultdict,
)

import requests
import pandas as pd
import nest_asyncio
import tldextract

from json_repair import repair_json
from pydantic import BaseModel, Field
from rich.console import Console
from rich.markdown import Markdown

nest_asyncio.apply()
console = Console()


# ============================================================
# Constants
# ============================================================

GOOGLE_AI_MODE_DATASET_ID = (
    "gd_mcswdt6z2elth3zqr2"
)

CHATGPT_DATASET_ID = (
    "gd_m7aof0k82r803d5bjm"
)

GEMINI_DATASET_ID = (
    "gd_mbz66arm2mf9cu856y"
)

BD_REQUEST_URL = (
    "https://api.brightdata.com/request"
)

BD_SCRAPE_URL = (
    "https://api.brightdata.com/datasets/v3/scrape"
)

BD_TRIGGER_URL = (
    "https://api.brightdata.com/datasets/v3/trigger"
)

BD_PROGRESS_URL = (
    "https://api.brightdata.com/datasets/v3/progress"
)

BD_SNAPSHOT_URL = (
    "https://api.brightdata.com/datasets/v3/snapshot"
)

GOOGLE_AI_OUTPUT_FIELDS = (
    "prompt,"
    "answer_text,"
    "citations,"
    "answer_text_markdown,"
    "timestamp"
)

FAILED_STATUSES = {
    "failed",
    "error",
    "canceled",
    "cancelled",
    "aborted",
}


# ============================================================
# Exceptions
# ============================================================

class BrightDataAPIError(RuntimeError):
    pass


class SnapshotTimeoutError(TimeoutError):
    def __init__(
        self,
        snapshot_id,
        timeout_seconds,
    ):
        self.snapshot_id = snapshot_id
        self.timeout_seconds = (
            timeout_seconds
        )

        super().__init__(
            f"Snapshot {snapshot_id} did not finish "
            f"within {timeout_seconds} seconds."
        )


# ============================================================
# Data models
# ============================================================

class BuyerIntentKeyword(BaseModel):
    keyword: str
    intent: str = "commercial"
    rationale: str = ""


class BrandAnalysis(BaseModel):
    brand_name: str
    official_url: str
    domain: str
    category: str = ""
    description: str = ""
    positioning: str = ""

    target_customers: list[str] = Field(
        default_factory=list
    )

    products: list[str] = Field(
        default_factory=list
    )

    key_features: list[str] = Field(
        default_factory=list
    )

    differentiators: list[str] = Field(
        default_factory=list
    )

    confidence: float = 0.0

    evidence: list[str] = Field(
        default_factory=list
    )


class CompanyIntake(BaseModel):
    brand: BrandAnalysis

    buyer_intent_keywords: list[
        BuyerIntentKeyword
    ] = Field(default_factory=list)


class CompetitorCandidate(BaseModel):
    domain: str
    preferred_hostname: str
    homepage_url: str
    frequency: int
    keyword_coverage: float
    best_rank: int
    average_rank: float
    rank_score: float
    total_score: float

    matched_keywords: list[str] = Field(
        default_factory=list
    )

    serp_urls: list[str] = Field(
        default_factory=list
    )

    serp_titles: list[str] = Field(
        default_factory=list
    )


class SelectedCompetitor(BaseModel):
    brand_name: str
    domain: str
    official_url: str
    reason: str = ""
    confidence: float = 0.0


class BrandProfile(BaseModel):
    brand_name: str
    official_url: str
    domain: str
    category: str = ""
    positioning: str = ""

    target_customers: list[str] = Field(
        default_factory=list
    )

    relevant_products: list[str] = Field(
        default_factory=list
    )

    key_features: list[str] = Field(
        default_factory=list
    )

    differentiators: list[str] = Field(
        default_factory=list
    )

    pricing_model: str = "unknown"
    competitor_reason: str = ""
    direct_competitor: bool = True
    confidence: float = 0.0

    evidence: list[str] = Field(
        default_factory=list
    )


# ============================================================
# General helpers
# ============================================================

def model_to_dict(model):
    if hasattr(model, "model_dump"):
        return model.model_dump()

    return model.dict()


def validate_model(
    model_class,
    data,
):
    if hasattr(
        model_class,
        "model_validate",
    ):
        return model_class.model_validate(
            data
        )

    return model_class.parse_obj(data)


def ensure_string_list(value):
    if value is None:
        return []

    if isinstance(value, str):
        value = value.strip()
        return [value] if value else []

    if isinstance(value, list):
        results = []

        for item in value:
            if item is None:
                continue

            item = str(item).strip()

            if item:
                results.append(item)

        return results

    return [str(value).strip()]


def normalize_boolean(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, str):
        return value.strip().lower() in {
            "true",
            "yes",
            "1",
        }

    return bool(value)


def normalize_confidence(value):
    try:
        value = float(value)

        if 1 < value <= 100:
            value = value / 100

        return max(
            0.0,
            min(1.0, value),
        )

    except (TypeError, ValueError):
        return 0.0


def shorten(
    value,
    max_length,
):
    value = str(value or "").strip()

    if len(value) <= max_length:
        return value

    return (
        value[:max_length - 3]
        .rstrip()
        + "..."
    )


def slugify(value):
    value = re.sub(
        r"[^a-zA-Z0-9]+",
        "-",
        str(value or "").lower(),
    )

    return value.strip("-") or "audit"


def remove_ai_boilerplate(text):
    """
    Remove AI-interface boilerplate and unwrap reports captured as one
    large Markdown list item.
    """
    if not isinstance(text, str):
        return ""

    text = text.strip()

    if not text:
        return ""

    lines = text.splitlines()

    # ChatGPT scraper output can wrap the complete response in a single
    # Markdown list item:
    #
    # *   Competitive Visibility Audit
    #     ============================
    #     Executive Summary
    #
    # Remove the outer bullet and indentation.
    first_nonempty_index = next(
        (
            index
            for index, line in enumerate(lines)
            if line.strip()
        ),
        None,
    )

    if first_nonempty_index is not None:
        first_line = lines[
            first_nonempty_index
        ]

        if re.match(
            r"^\s*[*+-]\s+"
            r"Competitive Visibility Audit\s*$",
            first_line,
            flags=re.IGNORECASE,
        ):
            lines[
                first_nonempty_index
            ] = "Competitive Visibility Audit"

            for index in range(
                first_nonempty_index + 1,
                len(lines),
            ):
                if lines[index].startswith(
                    "    "
                ):
                    lines[index] = (
                        lines[index][4:]
                    )

    exact_unwanted = {
        "log in",
        "login",
        "sign up",
        "sign up for free",
        "log insign up for free",
    }

    unwanted_prefixes = (
        "log in for more personalized",
        "if you want",
        "if useful",
        "would you like",
        "below is a professional audit",
        "below is the competitive visibility audit",
        "below is the requested audit",
        "here is the requested audit",
        "here's the requested audit",
    )

    cleaned_lines = []

    for line in lines:
        normalized = re.sub(
            r"\s+",
            " ",
            line,
        ).strip().lower()

        normalized_for_check = (
            normalized.lstrip(
                "•*-–—>#_ "
            )
        )

        if (
            normalized_for_check
            in exact_unwanted
        ):
            continue

        if any(
            normalized_for_check.startswith(
                prefix
            )
            for prefix
            in unwanted_prefixes
        ):
            continue

        cleaned_lines.append(line)

    cleaned = "\n".join(
        cleaned_lines
    ).strip()

    # Convert a Setext title to a standard Markdown heading.
    cleaned = re.sub(
        r"^Competitive Visibility Audit\s*\n"
        r"=+\s*$",
        "# Competitive Visibility Audit",
        cleaned,
        count=1,
        flags=(
            re.IGNORECASE
            | re.MULTILINE
        ),
    )

    # Remove duplicate titles.
    cleaned = re.sub(
        r"^# Competitive Visibility Audit\s*\n"
        r"\s*# Competitive Visibility Audit",
        "# Competitive Visibility Audit",
        cleaned,
        count=1,
        flags=re.IGNORECASE,
    )

    return cleaned



# ============================================================
# URL and domain helpers
# ============================================================

def extract_visible_url(value):
    value = str(value or "").strip()

    markdown_match = re.search(
        r"\[(https?://[^\]]+)\]"
        r"\([^)]+\)",
        value,
    )

    if markdown_match:
        value = markdown_match.group(1)

    url_match = re.search(
        r"https?://[^\s\])\"'>]+",
        value,
    )

    if url_match:
        value = url_match.group(0)

    return value.rstrip(
        ".,;:!?)]}"
    )


def normalize_public_url(value):
    value = extract_visible_url(value)

    if not value:
        value = str(value or "").strip()

    if not value:
        return ""

    if not value.lower().startswith(
        ("http://", "https://")
    ):
        value = f"https://{value}"

    parsed = urlparse(value)

    if not parsed.hostname:
        return ""

    scheme = parsed.scheme or "https"
    hostname = parsed.hostname.lower()
    path = parsed.path or "/"

    return f"{scheme}://{hostname}{path}"


def get_hostname(value):
    normalized = normalize_public_url(
        value
    )

    if not normalized:
        return ""

    return (
        urlparse(normalized).hostname
        or ""
    ).lower().removeprefix("www.")


def get_root_domain(value):
    if not value:
        return ""

    value = str(value).strip().lower()

    if "://" in value:
        hostname = (
            urlparse(value).hostname
            or ""
        )
    else:
        hostname = value.split("/")[0]

    hostname = hostname.removeprefix(
        "www."
    )

    extracted = tldextract.extract(
        hostname
    )

    if not extracted.domain:
        return hostname

    if not extracted.suffix:
        return extracted.domain

    return (
        f"{extracted.domain}."
        f"{extracted.suffix}"
    )


def canonical_source_url(value):
    """
    Deduplicate citation URLs by removing query strings and fragments.
    """
    value = str(value or "").strip()

    if not value:
        return ""

    try:
        parsed = urlparse(value)

        return urlunparse(
            (
                parsed.scheme,
                parsed.netloc.lower(),
                parsed.path.rstrip("/"),
                "",
                "",
                "",
            )
        )

    except Exception:
        return value


# ============================================================
# JSON repair
# ============================================================

def clean_ai_json_text(text):
    text = str(text or "").strip()

    text = re.sub(
        r"^\s*```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\s*```\s*$",
        "",
        text,
    )

    replacements = {
        r"\_": "_",
        r"\[": "[",
        r"\]": "]",
        r"\{": "{",
        r"\}": "}",
        r"\#": "#",
        r"\-": "-",
        r"\+": "+",
        r"\.": ".",
    }

    for source, replacement in (
        replacements.items()
    ):
        text = text.replace(
            source,
            replacement,
        )

    text = re.sub(
        r"\[(https?://[^\]]+)\]"
        r"\([^)]+\)",
        r"\1",
        text,
    )

    return text.strip()


def parse_ai_json(text):
    cleaned = clean_ai_json_text(
        text
    )

    first_brace = cleaned.find("{")
    last_brace = cleaned.rfind("}")

    if (
        first_brace >= 0
        and last_brace > first_brace
    ):
        cleaned = cleaned[
            first_brace:last_brace + 1
        ]

    try:
        parsed = json.loads(cleaned)

        if isinstance(parsed, dict):
            return parsed

    except json.JSONDecodeError:
        pass

    repaired = repair_json(
        cleaned,
        return_objects=True,
    )

    if isinstance(repaired, dict):
        return repaired

    if isinstance(repaired, str):
        repaired = json.loads(repaired)

        if isinstance(repaired, dict):
            return repaired

    raise ValueError(
        "AI response could not be parsed "
        "as a JSON object."
    )


# ============================================================
# Bright Data client
# ============================================================

class BrightDataClient:
    def __init__(
        self,
        token,
        serp_zone,
        country="US",
        debug=False,
    ):
        self.token = token
        self.serp_zone = serp_zone
        self.country = (
            country or "US"
        ).upper()
        self.debug = bool(debug)

        self.headers = {
            "Authorization": (
                f"Bearer {self.token}"
            ),
            "Content-Type": (
                "application/json"
            ),
        }

    def log(
        self,
        message,
        style="dim",
    ):
        if self.debug:
            console.print(
                f"[{style}]{message}"
                f"[/{style}]"
            )

    @staticmethod
    def normalize_records(data):
        if data is None:
            return []

        if isinstance(data, list):
            return data

        if isinstance(data, dict):
            for key in (
                "data",
                "results",
                "records",
            ):
                if isinstance(
                    data.get(key),
                    list,
                ):
                    return data[key]

            return [data]

        return []

    @staticmethod
    def answer_text(record):
        if not isinstance(record, dict):
            return str(record or "").strip()

        value = (
            record.get(
                "answer_text_markdown"
            )
            or record.get(
                "answer_text"
            )
            or record.get(
                "answer_markdown"
            )
            or record.get("answer")
            or record.get("response")
            or record.get("text")
            or ""
        )

        if isinstance(value, str):
            return value.strip()

        if isinstance(
            value,
            (dict, list),
        ):
            return json.dumps(
                value,
                ensure_ascii=False,
            )

        return str(value or "").strip()

    def snapshot_status(
        self,
        snapshot_id,
    ):
        response = requests.get(
            f"{BD_PROGRESS_URL}/"
            f"{snapshot_id}",
            headers=self.headers,
            timeout=30,
        )

        if not response.ok:
            return {
                "status": "unknown",
                "details": (
                    response.text[:1000]
                ),
            }

        data = response.json()

        return {
            "status": str(
                data.get(
                    "status",
                    "unknown",
                )
            ).lower(),
            "details": data,
        }

    def download_snapshot(
        self,
        snapshot_id,
    ):
        response = requests.get(
            f"{BD_SNAPSHOT_URL}/"
            f"{snapshot_id}",
            headers=self.headers,
            params={"format": "json"},
            timeout=90,
        )

        if not response.ok:
            raise BrightDataAPIError(
                f"Could not download snapshot "
                f"{snapshot_id}. HTTP "
                f"{response.status_code}: "
                f"{response.text[:1500]}"
            )

        return self.normalize_records(
            response.json()
        )

    def wait_for_snapshot(
        self,
        snapshot_id,
        timeout_seconds=600,
        poll_seconds=5,
    ):
        started_at = time.monotonic()
        last_debug_log = -30

        while True:
            elapsed = (
                time.monotonic()
                - started_at
            )

            if elapsed >= timeout_seconds:
                raise SnapshotTimeoutError(
                    snapshot_id,
                    timeout_seconds,
                )

            status_result = (
                self.snapshot_status(
                    snapshot_id
                )
            )

            status = status_result[
                "status"
            ]

            if (
                self.debug
                and elapsed
                - last_debug_log
                >= 30
            ):
                self.log(
                    f"Snapshot {snapshot_id}: "
                    f"{status} — "
                    f"{elapsed:.0f}s"
                )

                last_debug_log = elapsed

            if status == "ready":
                records = (
                    self.download_snapshot(
                        snapshot_id
                    )
                )

                # A ready snapshot can briefly return
                # a materialization-status object.
                if (
                    len(records) == 1
                    and isinstance(
                        records[0],
                        dict,
                    )
                    and str(
                        records[0].get(
                            "status",
                            "",
                        )
                    ).lower()
                    in {
                        "building",
                        "collecting",
                        "digesting",
                        "running",
                    }
                ):
                    time.sleep(
                        poll_seconds
                    )
                    continue

                return records

            if status in FAILED_STATUSES:
                raise BrightDataAPIError(
                    f"Snapshot {snapshot_id} "
                    f"ended with status "
                    f"{status}."
                )

            time.sleep(poll_seconds)

    def scrape_dataset(
        self,
        dataset_id,
        payload,
        timeout_seconds=600,
        custom_output_fields=None,
    ):
        params = {
            "dataset_id": dataset_id,
            "format": "json",
            "notify": "false",
            "include_errors": "true",
        }

        if custom_output_fields:
            params[
                "custom_output_fields"
            ] = custom_output_fields

        response = requests.post(
            BD_SCRAPE_URL,
            headers=self.headers,
            params=params,
            json=payload,
            timeout=90,
        )

        if response.status_code not in {
            200,
            202,
        }:
            raise BrightDataAPIError(
                f"Dataset request failed. "
                f"HTTP {response.status_code}: "
                f"{response.text[:2000]}"
            )

        try:
            data = response.json()
        except Exception as exc:
            raise BrightDataAPIError(
                "Dataset response was not "
                "valid JSON."
            ) from exc

        if (
            isinstance(data, dict)
            and data.get("snapshot_id")
        ):
            snapshot_id = data[
                "snapshot_id"
            ]

            self.log(
                f"Continuing snapshot "
                f"{snapshot_id}"
            )

            return self.wait_for_snapshot(
                snapshot_id,
                timeout_seconds=(
                    timeout_seconds
                ),
            )

        return self.normalize_records(data)

    def trigger_dataset(
        self,
        dataset_id,
        payload,
    ):
        response = requests.post(
            BD_TRIGGER_URL,
            headers=self.headers,
            params={
                "dataset_id": dataset_id,
                "format": "json",
                "include_errors": "true",
            },
            json=payload,
            timeout=60,
        )

        if not response.ok:
            raise BrightDataAPIError(
                f"Snapshot trigger failed. "
                f"HTTP {response.status_code}: "
                f"{response.text[:2000]}"
            )

        data = response.json()

        snapshot_id = (
            data.get("snapshot_id")
            if isinstance(data, dict)
            else None
        )

        if not snapshot_id:
            raise BrightDataAPIError(
                "Snapshot trigger did not "
                "return snapshot_id."
            )

        return snapshot_id

    def google_ai_mode(
        self,
        prompt,
        timeout_seconds=720,
    ):
        payload = {
            "input": [
                {
                    "url": (
                        "https://google.com/"
                        "aimode"
                    ),
                    "prompt": prompt,
                    "country": self.country,
                }
            ]
        }

        records = self.scrape_dataset(
            dataset_id=(
                GOOGLE_AI_MODE_DATASET_ID
            ),
            payload=payload,
            timeout_seconds=(
                timeout_seconds
            ),
            custom_output_fields=(
                GOOGLE_AI_OUTPUT_FIELDS
            ),
        )

        for record in records:
            if self.answer_text(record):
                return record

        raise BrightDataAPIError(
            "Google AI Mode returned "
            "no answer text."
        )

    def google_serp(
        self,
        query,
        language="en",
        num_results=10,
    ):
        search_url = (
            "https://www.google.com/"
            "search"
            f"?q={quote_plus(query)}"
            f"&gl={self.country.lower()}"
            f"&hl={language.lower()}"
            f"&num={num_results}"
        )

        response = requests.post(
            BD_REQUEST_URL,
            headers=self.headers,
            json={
                "zone": self.serp_zone,
                "url": search_url,
                "format": "raw",
                "data_format": (
                    "parsed_light"
                ),
            },
            timeout=90,
        )

        if not response.ok:
            raise BrightDataAPIError(
                f"SERP request failed. "
                f"HTTP {response.status_code}: "
                f"{response.text[:1500]}"
            )

        data = response.json()
        organic = data.get(
            "organic",
            [],
        )

        results = []

        for position, item in enumerate(
            organic,
            start=1,
        ):
            result_url = (
                item.get("link")
                or item.get("url")
                or ""
            )

            results.append(
                {
                    "rank": (
                        item.get("rank")
                        or position
                    ),
                    "title": item.get(
                        "title",
                        "",
                    ),
                    "url": result_url,
                    "domain": get_hostname(
                        result_url
                    ),
                    "description": (
                        item.get(
                            "description"
                        )
                        or item.get(
                            "snippet"
                        )
                        or ""
                    ),
                }
            )

        return {
            "query": query,
            "results": results,
        }

    def _engine_payload(
        self,
        engine,
        prompt,
        request_index,
        web_search=True,
    ):
        if engine == "chatgpt":
            item = {
                "url": (
                    "https://chatgpt.com/"
                ),
                "prompt": prompt,
                "country": self.country,
                "index": request_index,
                "web_search": web_search,
            }

            return (
                CHATGPT_DATASET_ID,
                [item],
            )

        if engine == "gemini":
            item = {
                "url": (
                    "https://gemini.google.com/"
                ),
                "prompt": prompt,
                "country": self.country,
                "index": request_index,
            }

            return (
                GEMINI_DATASET_ID,
                {"input": [item]},
            )

        raise ValueError(
            f"Unknown AI engine: {engine}"
        )

    def race_ai_engine(
        self,
        engine,
        prompt,
        redundancy=3,
        timeout_seconds=600,
    ):
        engine_name = (
            "ChatGPT"
            if engine == "chatgpt"
            else "Gemini"
        )

        started_at = time.monotonic()

        def trigger_one(index):
            dataset_id, payload = (
                self._engine_payload(
                    engine=engine,
                    prompt=prompt,
                    request_index=index,
                    web_search=True,
                )
            )

            return {
                "request_index": index,
                "snapshot_id": (
                    self.trigger_dataset(
                        dataset_id,
                        payload,
                    )
                ),
            }

        trigger_results = []

        with ThreadPoolExecutor(
            max_workers=redundancy
        ) as executor:
            futures = [
                executor.submit(
                    trigger_one,
                    index,
                )
                for index in range(
                    1,
                    redundancy + 1,
                )
            ]

            for future in as_completed(
                futures
            ):
                try:
                    trigger_results.append(
                        future.result()
                    )
                except Exception as exc:
                    self.log(
                        f"{engine_name} trigger "
                        f"failed: {exc}",
                        "yellow",
                    )

        if not trigger_results:
            raise BrightDataAPIError(
                f"All {engine_name} "
                f"triggers failed."
            )

        invalid_snapshots = set()
        failed_snapshots = set()

        while True:
            elapsed = (
                time.monotonic()
                - started_at
            )

            if elapsed >= timeout_seconds:
                raise TimeoutError(
                    f"No valid {engine_name} "
                    f"snapshot became ready "
                    f"within {timeout_seconds}s."
                )

            for item in trigger_results:
                snapshot_id = item[
                    "snapshot_id"
                ]

                if (
                    snapshot_id
                    in invalid_snapshots
                    or snapshot_id
                    in failed_snapshots
                ):
                    continue

                status = self.snapshot_status(
                    snapshot_id
                )["status"]

                if status in FAILED_STATUSES:
                    failed_snapshots.add(
                        snapshot_id
                    )
                    continue

                if status != "ready":
                    continue

                records = (
                    self.download_snapshot(
                        snapshot_id
                    )
                )

                for record in records:
                    answer = self.answer_text(
                        record
                    )

                    if not answer:
                        continue

                    return {
                        "engine": engine,
                        "engine_name": (
                            engine_name
                        ),
                        "status": "success",
                        "winner_snapshot_id": (
                            snapshot_id
                        ),
                        "winner_request_index": (
                            item[
                                "request_index"
                            ]
                        ),
                        "all_snapshot_ids": [
                            trigger[
                                "snapshot_id"
                            ]
                            for trigger in (
                                trigger_results
                            )
                        ],
                        "duration_seconds": (
                            round(
                                elapsed,
                                2,
                            )
                        ),
                        "answer": answer,
                        "record": record,
                        "citations": (
                            record.get(
                                "citations",
                                [],
                            )
                            or record.get(
                                "search_sources",
                                [],
                            )
                            or []
                        ),
                        "web_search_triggered": (
                            record.get(
                                "web_search_triggered"
                            )
                        ),
                    }

                invalid_snapshots.add(
                    snapshot_id
                )

            if (
                len(failed_snapshots)
                + len(invalid_snapshots)
                >= len(trigger_results)
            ):
                raise BrightDataAPIError(
                    f"All {engine_name} "
                    f"snapshots failed or "
                    f"returned no answer."
                )

            time.sleep(5)

    def generate_chatgpt_report(
        self,
        prompt,
        timeout_seconds=600,
    ):
        item = {
            "url": "https://chatgpt.com/",
            "prompt": prompt,
            "country": self.country,
            "web_search": False,
        }

        # The array request shape has been
        # validated by the working notebook.
        snapshot_id = self.trigger_dataset(
            CHATGPT_DATASET_ID,
            [item],
        )

        records = self.wait_for_snapshot(
            snapshot_id,
            timeout_seconds=(
                timeout_seconds
            ),
        )

        for record in records:
            answer = self.answer_text(
                record
            )

            if answer:
                return {
                    "snapshot_id": (
                        snapshot_id
                    ),
                    "answer": (
                        remove_ai_boilerplate(
                            answer
                        )
                    ),
                    "record": record,
                }

        raise BrightDataAPIError(
            "Final ChatGPT snapshot "
            "returned no report text."
        )


# ============================================================
# SERP competitor helpers
# ============================================================

NON_COMPETITOR_DOMAINS = {
    "facebook.com",
    "instagram.com",
    "linkedin.com",
    "twitter.com",
    "x.com",
    "youtube.com",
    "tiktok.com",
    "pinterest.com",
    "reddit.com",
    "quora.com",
    "wikipedia.org",
    "stackoverflow.com",
    "stackexchange.com",
    "g2.com",
    "capterra.com",
    "trustradius.com",
    "trustpilot.com",
    "getapp.com",
    "softwareadvice.com",
    "sourceforge.net",
    "alternativeto.net",
    "saasworthy.com",
    "crunchbase.com",
    "zoominfo.com",
    "bloomberg.com",
    "pitchbook.com",
    "glassdoor.com",
    "indeed.com",
    "medium.com",
    "substack.com",
    "dev.to",
    "forbes.com",
    "techcrunch.com",
    "businessinsider.com",
    "zdnet.com",
    "venturebeat.com",
    "gartner.com",
    "forrester.com",
    "coursera.org",
    "udemy.com",
    "researchgate.net",
    "arxiv.org",
}


def is_non_competitor_domain(
    domain,
):
    root = get_root_domain(domain)

    if not root:
        return True

    return any(
        root == excluded
        or root.endswith(
            f".{excluded}"
        )
        for excluded
        in NON_COMPETITOR_DOMAINS
    )


def looks_like_irrelevant_result(
    result,
):
    result_url = str(
        result.get("url", "")
    ).lower()

    title = str(
        result.get("title", "")
    ).lower()

    url_patterns = {
        "/jobs/",
        "/careers/",
        "/job/",
        "/news/",
        "/press/",
        "/events/",
        "/webinar/",
        "/podcast/",
    }

    title_patterns = {
        "salary",
        "jobs at",
        "careers at",
        "interview questions",
    }

    return (
        any(
            pattern in result_url
            for pattern in url_patterns
        )
        or any(
            pattern in title
            for pattern in title_patterns
        )
    )


def preferred_homepage_url(
    root_domain,
    hostnames,
):
    """
    Choose a likely official product or company hostname without
    hardcoding any vendor or industry.
    """
    host_counts = Counter(
        hostname
        for hostname in hostnames
        if hostname
    )

    preferred_hostname = (
        host_counts.most_common(1)[0][0]
        if host_counts
        else root_domain
    )

    ignored_subdomains = {
        "blog",
        "blogs",
        "docs",
        "documentation",
        "developer",
        "developers",
        "help",
        "support",
        "community",
        "forum",
        "forums",
        "news",
        "careers",
        "jobs",
    }

    first_label = (
        preferred_hostname
        .split(".")[0]
        .lower()
        if preferred_hostname
        else ""
    )

    # Preserve a meaningful product subdomain when it is the hostname
    # that actually ranks. Otherwise use the root domain.
    if (
        preferred_hostname
        and preferred_hostname.endswith(
            root_domain
        )
        and first_label
        not in ignored_subdomains
    ):
        homepage_hostname = (
            preferred_hostname
        )
    else:
        homepage_hostname = (
            root_domain
        )

    return (
        homepage_hostname,
        f"https://{homepage_hostname}/",
    )



def aggregate_competitor_domains(
    keyword_serp_results,
    target_domain,
    total_keyword_count,
):
    target_root = get_root_domain(
        target_domain
    )

    domain_data = defaultdict(
        lambda: {
            "ranks": [],
            "keywords": set(),
            "urls": [],
            "titles": [],
            "hostnames": [],
        }
    )

    for keyword_result in (
        keyword_serp_results
    ):
        if not keyword_result.get(
            "success"
        ):
            continue

        keyword = keyword_result[
            "keyword"
        ]

        seen_for_keyword = set()

        for position, result in enumerate(
            keyword_result.get(
                "results",
                [],
            ),
            start=1,
        ):
            hostname = (
                get_hostname(
                    result.get(
                        "url",
                        "",
                    )
                )
                or result.get(
                    "domain",
                    "",
                )
            )

            root = get_root_domain(
                hostname
            )

            if (
                not root
                or root == target_root
                or root in seen_for_keyword
                or is_non_competitor_domain(
                    root
                )
                or looks_like_irrelevant_result(
                    result
                )
            ):
                continue

            seen_for_keyword.add(root)

            try:
                rank = int(
                    result.get(
                        "rank",
                        position,
                    )
                )
            except Exception:
                rank = position

            domain_data[root][
                "ranks"
            ].append(rank)

            domain_data[root][
                "keywords"
            ].add(keyword)

            domain_data[root][
                "urls"
            ].append(
                result.get("url", "")
            )

            domain_data[root][
                "titles"
            ].append(
                result.get("title", "")
            )

            domain_data[root][
                "hostnames"
            ].append(hostname)

    candidates = []

    for domain, data in (
        domain_data.items()
    ):
        ranks = data["ranks"]

        if not ranks:
            continue

        frequency = len(
            data["keywords"]
        )

        rank_score = sum(
            1 / max(rank, 1)
            for rank in ranks
        )

        preferred_hostname, homepage = (
            preferred_homepage_url(
                domain,
                data["hostnames"],
            )
        )

        total_score = (
            frequency * 100
            + rank_score * 25
            + max(
                0,
                11 - min(ranks),
            )
        )

        candidates.append(
            CompetitorCandidate(
                domain=domain,
                preferred_hostname=(
                    preferred_hostname
                ),
                homepage_url=homepage,
                frequency=frequency,
                keyword_coverage=round(
                    frequency
                    / total_keyword_count,
                    4,
                ),
                best_rank=min(ranks),
                average_rank=round(
                    sum(ranks)
                    / len(ranks),
                    2,
                ),
                rank_score=round(
                    rank_score,
                    4,
                ),
                total_score=round(
                    total_score,
                    2,
                ),
                matched_keywords=sorted(
                    data["keywords"]
                ),
                serp_urls=[
                    url
                    for url in data["urls"]
                    if url
                ],
                serp_titles=[
                    title
                    for title in (
                        data["titles"]
                    )
                    if title
                ],
            )
        )

    candidates.sort(
        key=lambda item: (
            -item.frequency,
            -item.total_score,
            item.average_rank,
            item.domain,
        )
    )

    return candidates


# ============================================================
# Initialize client
# ============================================================

bd_client = BrightDataClient(
    token=BRIGHTDATA_API_TOKEN,
    serp_zone=SERP_ZONE,
    country=COUNTRY,
    debug=DEBUG_MODE,
)


# ============================================================
# Integrated reliability improvements
# ============================================================

def decode_bright_data_response(
    response,
    context,
):
    """
    Decode JSON, JSON text, or NDJSON from a Bright Data response.
    """
    text = (
        response.text
        if response is not None
        else ""
    )

    text = str(
        text or ""
    ).strip()

    if not text:
        raise BrightDataAPIError(
            f"{context} returned an empty response. "
            f"HTTP status: "
            f"{getattr(response, 'status_code', 'unknown')}"
        )

    try:
        return response.json()
    except Exception:
        pass

    try:
        return json.loads(text)
    except Exception:
        pass

    # Some endpoints can return newline-delimited JSON.
    ndjson_records = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        try:
            ndjson_records.append(
                json.loads(line)
            )
        except Exception:
            ndjson_records = []
            break

    if ndjson_records:
        return ndjson_records

    raise BrightDataAPIError(
        f"{context} did not return valid JSON. "
        f"HTTP status: "
        f"{getattr(response, 'status_code', 'unknown')}. "
        f"Response preview: {text[:1000]}"
    )


def parse_ai_json(text):
    """
    Parse JSON-like AI output without leaking JSONDecodeError.

    Supports:
    - Plain JSON
    - Markdown-fenced JSON
    - Google AI Mode Markdown escaping
    - Introductory text surrounding JSON
    - Repairable malformed JSON
    """
    original_text = str(
        text or ""
    ).strip()

    if not original_text:
        raise ValueError(
            "AI returned empty answer text."
        )

    cleaned = clean_ai_json_text(
        original_text
    )

    candidate_strings = [
        cleaned
    ]

    first_brace = cleaned.find("{")
    last_brace = cleaned.rfind("}")

    if (
        first_brace >= 0
        and last_brace > first_brace
    ):
        candidate_strings.insert(
            0,
            cleaned[
                first_brace:last_brace + 1
            ],
        )

    errors = []

    for candidate in candidate_strings:
        if not candidate.strip():
            continue

        # Standard JSON.
        try:
            parsed = json.loads(
                candidate
            )

            if isinstance(parsed, dict):
                return parsed

            errors.append(
                "Standard JSON result was "
                f"{type(parsed).__name__}, not an object."
            )

        except Exception as exc:
            errors.append(
                f"Standard JSON: {exc}"
            )

        # JSON repair.
        try:
            repaired = repair_json(
                candidate,
                return_objects=True,
            )

            if isinstance(repaired, dict):
                return repaired

            if isinstance(repaired, list):
                for item in repaired:
                    if isinstance(item, dict):
                        return item

            if (
                isinstance(repaired, str)
                and repaired.strip()
            ):
                try:
                    reparsed = json.loads(
                        repaired
                    )

                    if isinstance(
                        reparsed,
                        dict,
                    ):
                        return reparsed

                except Exception as exc:
                    errors.append(
                        f"Repaired string JSON: {exc}"
                    )

        except Exception as exc:
            errors.append(
                f"JSON repair: {exc}"
            )

    raise ValueError(
        "AI response could not be parsed as a JSON object.\n\n"
        f"Response preview:\n{original_text[:2500]}\n\n"
        f"Parser errors:\n- "
        + "\n- ".join(errors[-6:])
    )


def reliable_snapshot_status(
    self,
    snapshot_id,
):
    """
    Snapshot status that treats temporary empty/non-JSON responses as
    unknown rather than crashing the audit.
    """
    try:
        response = requests.get(
            f"{BD_PROGRESS_URL}/"
            f"{snapshot_id}",
            headers=self.headers,
            timeout=30,
        )

        if not response.ok:
            return {
                "status": "unknown",
                "details": (
                    response.text[:1000]
                ),
            }

        data = decode_bright_data_response(
            response,
            context=(
                f"Progress endpoint for "
                f"{snapshot_id}"
            ),
        )

        if not isinstance(data, dict):
            return {
                "status": "unknown",
                "details": data,
            }

        return {
            "status": str(
                data.get(
                    "status",
                    "unknown",
                )
            ).lower(),
            "details": data,
        }

    except Exception as exc:
        self.log(
            f"Temporary progress error for "
            f"{snapshot_id}: {exc}",
            "yellow",
        )

        return {
            "status": "unknown",
            "details": str(exc),
        }


def reliable_download_snapshot(
    self,
    snapshot_id,
    attempts=4,
):
    """
    Retry temporarily empty or non-JSON snapshot downloads.
    """
    last_error = None

    for attempt in range(
        1,
        attempts + 1,
    ):
        try:
            response = requests.get(
                f"{BD_SNAPSHOT_URL}/"
                f"{snapshot_id}",
                headers=self.headers,
                params={"format": "json"},
                timeout=90,
            )

            if not response.ok:
                raise BrightDataAPIError(
                    f"Snapshot download failed. "
                    f"HTTP {response.status_code}: "
                    f"{response.text[:1500]}"
                )

            data = (
                decode_bright_data_response(
                    response,
                    context=(
                        f"Snapshot download "
                        f"{snapshot_id}"
                    ),
                )
            )

            records = (
                self.normalize_records(
                    data
                )
            )

            if not records:
                raise BrightDataAPIError(
                    "Snapshot download returned "
                    "no records."
                )

            return records

        except Exception as exc:
            last_error = exc

            self.log(
                f"Snapshot download attempt "
                f"{attempt}/{attempts} failed "
                f"for {snapshot_id}: {exc}",
                "yellow",
            )

            if attempt < attempts:
                time.sleep(
                    attempt * 3
                )

    raise BrightDataAPIError(
        f"Could not download snapshot "
        f"{snapshot_id} after "
        f"{attempts} attempts: "
        f"{last_error}"
    )


def reliable_trigger_dataset(
    self,
    dataset_id,
    payload,
):
    response = requests.post(
        BD_TRIGGER_URL,
        headers=self.headers,
        params={
            "dataset_id": dataset_id,
            "format": "json",
            "include_errors": "true",
        },
        json=payload,
        timeout=60,
    )

    if not response.ok:
        raise BrightDataAPIError(
            f"Snapshot trigger failed. "
            f"HTTP {response.status_code}: "
            f"{response.text[:2000]}"
        )

    data = decode_bright_data_response(
        response,
        context=(
            f"Snapshot trigger for "
            f"{dataset_id}"
        ),
    )

    snapshot_id = (
        data.get("snapshot_id")
        if isinstance(data, dict)
        else None
    )

    if not snapshot_id:
        raise BrightDataAPIError(
            "Snapshot trigger did not "
            f"return snapshot_id: {data}"
        )

    return snapshot_id


def reliable_scrape_dataset(
    self,
    dataset_id,
    payload,
    timeout_seconds=600,
    custom_output_fields=None,
):
    params = {
        "dataset_id": dataset_id,
        "format": "json",
        "notify": "false",
        "include_errors": "true",
    }

    if custom_output_fields:
        params[
            "custom_output_fields"
        ] = custom_output_fields

    response = requests.post(
        BD_SCRAPE_URL,
        headers=self.headers,
        params=params,
        json=payload,
        timeout=90,
    )

    if response.status_code not in {
        200,
        202,
    }:
        raise BrightDataAPIError(
            f"Dataset request failed. "
            f"HTTP {response.status_code}: "
            f"{response.text[:2000]}"
        )

    data = decode_bright_data_response(
        response,
        context=(
            f"Dataset request for "
            f"{dataset_id}"
        ),
    )

    if (
        isinstance(data, dict)
        and data.get("snapshot_id")
    ):
        snapshot_id = data[
            "snapshot_id"
        ]

        self.log(
            f"Continuing snapshot "
            f"{snapshot_id}"
        )

        return self.wait_for_snapshot(
            snapshot_id,
            timeout_seconds=(
                timeout_seconds
            ),
        )

    return self.normalize_records(data)


def remove_ai_boilerplate(text):
    """
    Remove AI-interface boilerplate and unwrap reports captured as one
    large Markdown list item.
    """
    if not isinstance(text, str):
        return ""

    text = text.strip()

    if not text:
        return ""

    lines = text.splitlines()

    # ChatGPT scraper output can wrap the complete response in a single
    # Markdown list item:
    #
    # *   Competitive Visibility Audit
    #     ============================
    #     Executive Summary
    #
    # Remove the outer bullet and indentation.
    first_nonempty_index = next(
        (
            index
            for index, line in enumerate(lines)
            if line.strip()
        ),
        None,
    )

    if first_nonempty_index is not None:
        first_line = lines[
            first_nonempty_index
        ]

        if re.match(
            r"^\s*[*+-]\s+"
            r"Competitive Visibility Audit\s*$",
            first_line,
            flags=re.IGNORECASE,
        ):
            lines[
                first_nonempty_index
            ] = "Competitive Visibility Audit"

            for index in range(
                first_nonempty_index + 1,
                len(lines),
            ):
                if lines[index].startswith(
                    "    "
                ):
                    lines[index] = (
                        lines[index][4:]
                    )

    exact_unwanted = {
        "log in",
        "login",
        "sign up",
        "sign up for free",
        "log insign up for free",
    }

    unwanted_prefixes = (
        "log in for more personalized",
        "if you want",
        "if useful",
        "would you like",
        "below is a professional audit",
        "below is the competitive visibility audit",
        "below is the requested audit",
        "here is the requested audit",
        "here's the requested audit",
    )

    cleaned_lines = []

    for line in lines:
        normalized = re.sub(
            r"\s+",
            " ",
            line,
        ).strip().lower()

        normalized_for_check = (
            normalized.lstrip(
                "•*-–—>#_ "
            )
        )

        if (
            normalized_for_check
            in exact_unwanted
        ):
            continue

        if any(
            normalized_for_check.startswith(
                prefix
            )
            for prefix
            in unwanted_prefixes
        ):
            continue

        cleaned_lines.append(line)

    cleaned = "\n".join(
        cleaned_lines
    ).strip()

    # Convert a Setext title to a standard Markdown heading.
    cleaned = re.sub(
        r"^Competitive Visibility Audit\s*\n"
        r"=+\s*$",
        "# Competitive Visibility Audit",
        cleaned,
        count=1,
        flags=(
            re.IGNORECASE
            | re.MULTILINE
        ),
    )

    # Remove duplicate titles.
    cleaned = re.sub(
        r"^# Competitive Visibility Audit\s*\n"
        r"\s*# Competitive Visibility Audit",
        "# Competitive Visibility Audit",
        cleaned,
        count=1,
        flags=re.IGNORECASE,
    )

    return cleaned



# Use the reliable implementations directly.
BrightDataClient.snapshot_status = (
    reliable_snapshot_status
)

BrightDataClient.download_snapshot = (
    reliable_download_snapshot
)

BrightDataClient.trigger_dataset = (
    reliable_trigger_dataset
)

BrightDataClient.scrape_dataset = (
    reliable_scrape_dataset
)


console.print(
    "[bold green]✓ Core audit engine loaded[/bold green]"
)
console.print(
    f"Country: {bd_client.country}"
)
console.print(
    f"Debug logging: "
    f"{'enabled' if bd_client.debug else 'disabled'}"
)


# ============================================================
# Flexible AI-answer and SERP normalization
# ============================================================

def reliable_answer_text(
    record,
):
    """
    Choose a sensible answer representation.

    Prefer Markdown for normal answers, but prefer plain text when the
    Markdown field is unexpectedly huge.
    """
    if not isinstance(record, dict):
        return str(record or "").strip()

    plain_text = (
        record.get("answer_text")
        or record.get("answer")
        or record.get("response")
        or record.get("text")
        or ""
    )

    markdown_text = (
        record.get("answer_text_markdown")
        or record.get("answer_markdown")
        or ""
    )

    if not isinstance(
        plain_text,
        str,
    ):
        plain_text = (
            json.dumps(
                plain_text,
                ensure_ascii=False,
            )
            if plain_text
            else ""
        )

    if not isinstance(
        markdown_text,
        str,
    ):
        markdown_text = (
            json.dumps(
                markdown_text,
                ensure_ascii=False,
            )
            if markdown_text
            else ""
        )

    plain_text = plain_text.strip()
    markdown_text = markdown_text.strip()

    # Some answer-engine records occasionally contain an extremely
    # large Markdown capture. Prefer the concise plain answer.
    if (
        len(markdown_text) > 100_000
        and plain_text
    ):
        return plain_text

    if markdown_text:
        return markdown_text

    return plain_text


def extract_serp_url_value(
    item,
):
    """
    Extract a usable result URL from different parsed SERP schemas.
    """
    from urllib.parse import (
        parse_qs,
        unquote,
        urlparse,
    )

    if not isinstance(item, dict):
        return ""

    preferred_fields = (
        "link",
        "url",
        "href",
        "result_url",
        "target_url",
        "redirect_url",
    )

    value = ""

    for field_name in preferred_fields:
        candidate = item.get(
            field_name
        )

        if isinstance(candidate, dict):
            candidate = (
                candidate.get("url")
                or candidate.get("link")
                or candidate.get("href")
                or ""
            )

        if isinstance(candidate, str):
            candidate = candidate.strip()

            if candidate:
                value = candidate
                break

    # As a last resort, search the record for a normal HTTP URL.
    if not value:
        for candidate in item.values():
            if (
                isinstance(candidate, str)
                and "http" in candidate
            ):
                match = re.search(
                    r"https?://[^\s\"'<>]+",
                    candidate,
                )

                if match:
                    value = match.group(0)
                    break

    if not value:
        return ""

    # Protocol-relative URL.
    if value.startswith("//"):
        value = f"https:{value}"

    # Google redirect URL.
    if value.startswith("/"):
        parsed_relative = urlparse(
            value
        )

        query = parse_qs(
            parsed_relative.query
        )

        redirected = (
            query.get("q", [""])[0]
            or query.get(
                "url",
                [""],
            )[0]
        )

        if redirected:
            value = unquote(
                redirected
            )
        else:
            return ""

    # A plain hostname can still be useful.
    if not value.lower().startswith(
        ("http://", "https://")
    ):
        if (
            "." in value
            and " " not in value
        ):
            value = (
                f"https://{value}"
            )
        else:
            return ""

    return value


def extract_serp_domain_value(
    item,
    result_url="",
):
    """
    Extract a root domain from a SERP result using multiple fallbacks.
    """
    domain = get_root_domain(
        result_url
    )

    if domain:
        return domain

    if not isinstance(item, dict):
        return ""

    domain_fields = (
        "domain",
        "displayed_link",
        "display_link",
        "visible_url",
        "source",
        "site_name",
        "hostname",
    )

    for field_name in domain_fields:
        candidate = item.get(
            field_name
        )

        if isinstance(candidate, dict):
            candidate = (
                candidate.get("domain")
                or candidate.get("url")
                or candidate.get("name")
                or ""
            )

        if not isinstance(
            candidate,
            str,
        ):
            continue

        candidate = (
            candidate.strip()
            .replace("›", "/")
        )

        if not candidate:
            continue

        # Keep only the first URL/domain-looking component.
        candidate = candidate.split()[0]
        candidate = candidate.split("/")[0]

        domain = get_root_domain(
            candidate
        )

        if domain:
            return domain

    return ""


def reliable_google_serp(
    self,
    query,
    language="en",
    num_results=10,
):
    """
    Query SERP API and normalize several possible parsed-result shapes.
    """
    search_url = (
        "https://www.google.com/search"
        f"?q={quote_plus(query)}"
        f"&gl={self.country.lower()}"
        f"&hl={language.lower()}"
        f"&num={num_results}"
    )

    response = requests.post(
        BD_REQUEST_URL,
        headers=self.headers,
        json={
            "zone": self.serp_zone,
            "url": search_url,
            "format": "raw",
            "data_format": (
                "parsed_light"
            ),
        },
        timeout=90,
    )

    if not response.ok:
        raise BrightDataAPIError(
            f"SERP request failed. "
            f"HTTP {response.status_code}: "
            f"{response.text[:1500]}"
        )

    data = decode_bright_data_response(
        response,
        context=(
            f"SERP request for {query!r}"
        ),
    )

    if not isinstance(data, dict):
        raise BrightDataAPIError(
            "SERP response was not a JSON object."
        )

    organic = (
        data.get("organic")
        or data.get("results")
        or data.get(
            "organic_results"
        )
        or []
    )

    if not isinstance(organic, list):
        organic = []

    results = []

    for position, item in enumerate(
        organic,
        start=1,
    ):
        if not isinstance(item, dict):
            continue

        result_url = (
            extract_serp_url_value(
                item
            )
        )

        domain = (
            extract_serp_domain_value(
                item,
                result_url,
            )
        )

        if (
            not result_url
            and domain
        ):
            result_url = (
                f"https://{domain}/"
            )

        if self.debug and not domain:
            self.log(
                f"SERP result had no domain. "
                f"Query={query!r}; "
                f"fields={list(item.keys())}; "
                f"record={str(item)[:500]}",
                "yellow",
            )

        results.append(
            {
                "rank": (
                    item.get("rank")
                    or item.get("position")
                    or position
                ),
                "title": (
                    item.get("title")
                    or item.get("name")
                    or ""
                ),
                "url": result_url,
                "domain": domain,
                "description": (
                    item.get("description")
                    or item.get("snippet")
                    or item.get("text")
                    or ""
                ),
            }
        )

    return {
        "query": query,
        "results": results,
        "raw_result_count": len(
            organic
        ),
    }


BrightDataClient.answer_text = staticmethod(
    reliable_answer_text
)

BrightDataClient.google_serp = (
    reliable_google_serp
)


# ============================================================
# Google citation redirect resolution
# ============================================================

GOOGLE_GOTO_CACHE = {}


def is_google_goto_url(url):
    """
    Return True for opaque Google /goto citation URLs.
    """
    try:
        parsed = urlparse(
            str(url or "")
        )

        hostname = (
            parsed.hostname
            or ""
        ).lower().removeprefix(
            "www."
        )

        return (
            hostname == "google.com"
            and parsed.path.startswith(
                "/goto"
            )
        )

    except Exception:
        return False


def resolve_google_goto_url(
    url,
    timeout_seconds=20,
):
    """
    Follow a Google /goto redirect and return its final destination.

    Results are cached for the lifetime of the notebook.
    """
    url = str(
        url or ""
    ).strip()

    if not url:
        return ""

    if not is_google_goto_url(
        url
    ):
        return url

    if url in GOOGLE_GOTO_CACHE:
        return GOOGLE_GOTO_CACHE[
            url
        ]

    session = requests.Session()
    session.max_redirects = 8

    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/140.0 Safari/537.36"
        ),
        "Accept": (
            "text/html,application/xhtml+xml,"
            "application/xml;q=0.9,*/*;q=0.8"
        ),
    }

    resolved_url = ""

    try:
        # HEAD avoids downloading the complete destination page when
        # the redirect chain supports it.
        response = session.head(
            url,
            headers=headers,
            allow_redirects=True,
            timeout=timeout_seconds,
        )

        candidate = str(
            response.url or ""
        ).strip()

        if (
            candidate
            and candidate.startswith(
                ("http://", "https://")
            )
            and not is_google_goto_url(
                candidate
            )
        ):
            resolved_url = (
                candidate
            )

    except Exception:
        pass

    if not resolved_url:
        try:
            # Some Google redirect endpoints do not support HEAD.
            # stream=True follows redirects without downloading the
            # complete destination body.
            response = session.get(
                url,
                headers=headers,
                allow_redirects=True,
                timeout=timeout_seconds,
                stream=True,
            )

            candidate = str(
                response.url or ""
            ).strip()

            response.close()

            if (
                candidate
                and candidate.startswith(
                    ("http://", "https://")
                )
                and not is_google_goto_url(
                    candidate
                )
            ):
                resolved_url = (
                    candidate
                )

        except Exception as exc:
            if (
                "bd_client" in globals()
                and bd_client.debug
            ):
                bd_client.log(
                    f"Could not resolve Google "
                    f"redirect: {exc}",
                    "yellow",
                )

    GOOGLE_GOTO_CACHE[
        url
    ] = resolved_url

    if (
        resolved_url
        and "bd_client" in globals()
        and bd_client.debug
    ):
        bd_client.log(
            f"Resolved Google citation: "
            f"{url[:70]}... -> "
            f"{resolved_url}"
        )

    return resolved_url


# ============================================================
# Search-engine selection: Bing, Google, or automatic fallback
# ============================================================

ACTIVE_SEARCH_ENGINE = None


def find_parsed_organic_results(
    data,
):
    """
    Locate organic-result records across several parsed SERP response
    shapes.
    """
    if isinstance(data, list):
        # Some responses wrap the parsed result inside a list.
        for item in data:
            found = (
                find_parsed_organic_results(
                    item
                )
            )

            if found:
                return found

        return []

    if not isinstance(data, dict):
        return []

    for field_name in (
        "organic",
        "organic_results",
        "results",
        "web_results",
        "webPages",
    ):
        value = data.get(
            field_name
        )

        if field_name == "webPages":
            if isinstance(value, dict):
                value = value.get(
                    "value",
                    []
                )

        if (
            isinstance(value, list)
            and value
        ):
            return value

    # Search one level deeper for a familiar result collection.
    for value in data.values():
        if isinstance(
            value,
            (dict, list),
        ):
            found = (
                find_parsed_organic_results(
                    value
                )
            )

            if found:
                return found

    return []


def normalize_serp_records(
    data,
    query,
    engine,
    debug=False,
):
    """
    Normalize Google and Bing parsed records into the notebook's common
    SERP shape.
    """
    organic = (
        find_parsed_organic_results(
            data
        )
    )

    results = []

    for position, item in enumerate(
        organic,
        start=1,
    ):
        if not isinstance(item, dict):
            continue

        result_url = (
            extract_serp_result_url(
                item
            )
            if (
                "extract_serp_result_url"
                in globals()
            )
            else str(
                item.get("link")
                or item.get("url")
                or item.get("href")
                or ""
            ).strip()
        )

        domain = (
            extract_serp_result_domain(
                item,
                result_url,
            )
            if (
                "extract_serp_result_domain"
                in globals()
            )
            else get_root_domain(
                result_url
                or item.get("domain")
                or ""
            )
        )

        if not result_url and domain:
            result_url = (
                f"https://{domain}/"
            )

        if debug and not domain:
            bd_client.log(
                f"{engine.title()} result had no domain. "
                f"Query={query!r}; "
                f"fields={list(item.keys())}; "
                f"record={str(item)[:600]}",
                "yellow",
            )

        raw_rank = (
            item.get("rank")
            or item.get("position")
            or item.get("ranking")
            or position
        )

        try:
            rank = int(raw_rank)

        except Exception:
            rank = position

        title = str(
            item.get("title")
            or item.get("name")
            or item.get("headline")
            or ""
        ).strip()

        description = str(
            item.get("description")
            or item.get("snippet")
            or item.get("text")
            or item.get("caption")
            or ""
        ).strip()

        results.append(
            {
                "rank": rank,
                "title": title,
                "url": result_url,
                "domain": domain,
                "description": (
                    description
                ),
            }
        )

    return {
        "query": query,
        "engine": engine,
        "results": results,
        "raw_result_count": len(
            organic
        ),
    }


def reliable_bing_serp(
    self,
    query,
    language="en",
    num_results=20,
):
    """
    Run a Bing search through the existing Bright Data SERP API zone.
    """
    country = (
        str(
            self.country
            or "US"
        )
        .strip()
        .upper()
    )

    language = (
        str(language or "en")
        .strip()
        .lower()
    )

    search_url = (
        "https://www.bing.com/search"
        f"?q={quote_plus(query)}"
        f"&count={num_results}"
        f"&cc={country}"
        f"&setlang={language}"
    )

    response = requests.post(
        BD_REQUEST_URL,
        headers=self.headers,
        json={
            "zone": self.serp_zone,
            "url": search_url,
            "format": "raw",
            "data_format": (
                "parsed_light"
            ),
        },
        timeout=90,
    )

    if not response.ok:
        raise BrightDataAPIError(
            f"Bing SERP request failed. "
            f"HTTP {response.status_code}: "
            f"{response.text[:1500]}"
        )

    data = decode_bright_data_response(
        response,
        context=(
            f"Bing SERP request for "
            f"{query!r}"
        ),
    )

    return normalize_serp_records(
        data=data,
        query=query,
        engine="bing",
        debug=self.debug,
    )


def search_engine_health(
    result,
    minimum_results=5,
):
    """
    Confirm that a SERP response contains enough usable organic
    results and domains.
    """
    if not isinstance(result, dict):
        return False

    results = result.get(
        "results",
        [],
    )

    usable_results = [
        item
        for item in results
        if (
            item.get("domain")
            and item.get("rank")
            is not None
        )
    ]

    return (
        len(usable_results)
        >= minimum_results
    )


def choose_search_engine(
    self,
    test_query,
    requested_engine="bing",
):
    """
    Resolve one search engine for the complete audit.

    No audit mixes Google and Bing rankings.
    """
    global ACTIVE_SEARCH_ENGINE

    requested_engine = str(
        requested_engine
        or "bing"
    ).strip().lower()

    if requested_engine not in {
        "auto",
        "google",
        "bing",
    }:
        raise ValueError(
            "SEARCH_ENGINE must be "
            "'auto', 'google', or 'bing'."
        )

    if requested_engine in {
        "google",
        "bing",
    }:
        ACTIVE_SEARCH_ENGINE = (
            requested_engine
        )

        self.active_search_engine = (
            requested_engine
        )

        self.log(
            f"Search engine selected: "
            f"{requested_engine.title()}"
        )

        return requested_engine

    self.log(
        "Testing Google Search availability"
    )

    try:
        google_result = (
            self.google_serp(
                query=test_query,
                num_results=10,
            )
        )

        if search_engine_health(
            google_result,
            minimum_results=5,
        ):
            ACTIVE_SEARCH_ENGINE = (
                "google"
            )

            self.active_search_engine = (
                "google"
            )

            self.log(
                "Google health check passed"
            )

            return "google"

        self.log(
            "Google health check returned "
            "insufficient usable results. "
            "Switching the complete run to Bing.",
            "yellow",
        )

    except Exception as exc:
        self.log(
            f"Google health check failed: "
            f"{exc}. Switching the complete "
            f"run to Bing.",
            "yellow",
        )

    # Verify Bing before launching all eight searches.
    bing_result = (
        self.bing_serp(
            query=test_query,
            num_results=10,
        )
    )

    if not search_engine_health(
        bing_result,
        minimum_results=5,
    ):
        raise BrightDataAPIError(
            "Neither Google nor Bing returned "
            "enough usable organic results."
        )

    ACTIVE_SEARCH_ENGINE = "bing"
    self.active_search_engine = "bing"

    self.log(
        "Bing health check passed"
    )

    return "bing"


def search_serp(
    self,
    query,
    engine=None,
    language="en",
    num_results=20,
):
    """
    Run one search using the engine selected for this audit.
    """
    selected_engine = str(
        engine
        or getattr(
            self,
            "active_search_engine",
            "",
        )
        or globals().get(
            "SEARCH_ENGINE",
            "bing",
        )
    ).strip().lower()

    if selected_engine == "google":
        result = self.google_serp(
            query=query,
            language=language,
            num_results=num_results,
        )

        result["engine"] = "google"

        return result

    if selected_engine == "bing":
        return self.bing_serp(
            query=query,
            language=language,
            num_results=num_results,
        )

    raise ValueError(
        f"Search engine has not been "
        f"resolved: {selected_engine}"
    )


BrightDataClient.bing_serp = (
    reliable_bing_serp
)

BrightDataClient.choose_search_engine = (
    choose_search_engine
)

BrightDataClient.search_serp = (
    search_serp
)


# ============================================================
# Markdown Google/Bing search
# ============================================================

MARKDOWN_SERP_CACHE = {}

ACTIVE_SEARCH_ENGINE = None
ACTIVE_SEARCH_STATUS = "unavailable"


def extract_markdown_payload(
    response,
):
    """
    Extract Markdown from a raw text response or JSON-wrapped response.
    """
    text = response.text or ""

    try:
        parsed = response.json()

    except Exception:
        parsed = None

    def find_text(value):
        if isinstance(value, str):
            return value

        if isinstance(value, list):
            for item in value:
                found = find_text(item)

                if (
                    found
                    and (
                        "# Search Results"
                        in found
                        or "## Web results"
                        in found
                        or "## ["
                        in found
                    )
                ):
                    return found

        if isinstance(value, dict):
            preferred_fields = (
                "markdown",
                "content",
                "body",
                "result",
                "response",
                "data",
                "text",
            )

            for field_name in (
                preferred_fields
            ):
                if field_name in value:
                    found = find_text(
                        value[
                            field_name
                        ]
                    )

                    if found:
                        return found

            for item in value.values():
                found = find_text(
                    item
                )

                if (
                    found
                    and (
                        "# Search Results"
                        in found
                        or "## Web results"
                        in found
                        or "## ["
                        in found
                    )
                ):
                    return found

        return ""

    if parsed is not None:
        extracted = find_text(
            parsed
        )

        if extracted:
            text = extracted

    return str(
        text or ""
    ).strip()


def clean_serp_markdown(
    markdown,
):
    """
    Remove embedded images and response-printing artifacts before
    parsing.
    """
    markdown = str(
        markdown or ""
    )

    # Embedded base64 and data-URI images can account for most of the
    # response size.
    markdown = re.sub(
        r"!\[[^\]]*\]"
        r"\(data:image[^)]*\)",
        "",
        markdown,
        flags=re.IGNORECASE
        | re.DOTALL,
    )

    markdown = re.sub(
        r"<Response\s*\[\d+\]>",
        "",
        markdown,
        flags=re.IGNORECASE,
    )

    markdown = markdown.replace(
        "\r\n",
        "\n",
    )

    markdown = re.sub(
        r"\n{4,}",
        "\n\n",
        markdown,
    )

    return markdown.strip()


def clean_displayed_url(
    value,
):
    """
    Convert a displayed SERP URL into a usable URL.
    """
    value = str(
        value or ""
    ).strip()

    if not value.startswith(
        ("http://", "https://")
    ):
        return ""

    value = value.split(
        " › ",
        1,
    )[0]

    value = value.split()[0]

    value = value.rstrip(
        ".,;:)]"
    )

    return value


def result_snippet_from_lines(
    lines,
):
    ignored_prefixes = (
        "http://",
        "https://",
        "](",
        "[read more]",
        "read more",
        "cached",
        "translate this result",
    )

    candidates = []

    for line in lines:
        line = re.sub(
            r"\s+",
            " ",
            str(line or ""),
        ).strip()

        if not line:
            continue

        lowered = line.lower()

        if lowered.startswith(
            ignored_prefixes
        ):
            continue

        if line.startswith(
            ("[", "]", "!", "#")
        ):
            continue

        if lowered in {
            "view all",
            "see more",
            "show all",
            "images",
            "videos",
        }:
            continue

        if len(line) < 35:
            continue

        candidates.append(line)

    return (
        max(
            candidates,
            key=len,
        )
        if candidates
        else ""
    )


def parse_google_markdown(
    markdown,
    query,
    num_results=20,
    requested_country="US",
):
    """
    Parse classic organic results from Google Markdown.

    Rich-result modules are skipped. Organic results are assigned
    sequential organic rank after filtering.
    """
    markdown = clean_serp_markdown(
        markdown
    )

    if "# Search Results" not in (
        markdown
    ):
        raise BrightDataAPIError(
            "Google Markdown did not contain "
            "the Search Results marker."
        )

    search_body = markdown.split(
        "# Search Results",
        1,
    )[1]

    if "# Page navigation" in (
        search_body
    ):
        search_body = (
            search_body.split(
                "# Page navigation",
                1,
            )[0]
        )

    lines = search_body.splitlines()

    modules = {
        "people also ask",
        "people also search for",
        "videos",
        "short videos",
        "images",
        "news",
        "forums",
        "shopping",
    }

    blocks = []
    current = None
    in_web_results = False

    def flush_current():
        nonlocal current

        if current:
            blocks.append(current)
            current = None

    for raw_line in lines:
        line = raw_line.strip()
        lowered = line.lower()

        if lowered == (
            "## web results"
        ):
            flush_current()
            in_web_results = True
            continue

        if (
            lowered in modules
            or any(
                lowered.startswith(
                    module + " "
                )
                for module in modules
            )
        ):
            flush_current()
            in_web_results = False
            continue

        if (
            line.startswith("## ")
            and lowered
            != "## web results"
        ):
            flush_current()
            in_web_results = False
            continue

        if (
            in_web_results
            and line.startswith(
                "### "
            )
        ):
            flush_current()

            current = {
                "title": line[4:].strip(),
                "lines": [],
            }

            continue

        if current is not None:
            current[
                "lines"
            ].append(raw_line)

    flush_current()

    results = []
    seen = set()

    for block in blocks:
        title = re.sub(
            r"\s+",
            " ",
            block["title"],
        ).strip()

        block_text = "\n".join(
            block["lines"]
        )

        displayed_url = ""

        for line in block["lines"]:
            candidate = (
                clean_displayed_url(
                    line
                )
            )

            if not candidate:
                continue

            hostname = (
                get_hostname(
                    candidate
                )
            )

            if (
                hostname
                and "google." not in (
                    hostname
                )
            ):
                displayed_url = (
                    candidate
                )
                break

        goto_match = re.search(
            r"\]\((/goto\?url=[^)]+)\)",
            block_text,
        )

        resolved_url = ""

        if goto_match:
            goto_url = (
                "https://www.google.com"
                + goto_match.group(1)
            )

            if (
                "resolve_google_goto_url"
                in globals()
            ):
                resolved_url = (
                    resolve_google_goto_url(
                        goto_url
                    )
                )

        final_url = (
            resolved_url
            or displayed_url
        )

        domain = get_root_domain(
            final_url
        )

        if not domain:
            continue

        dedupe_key = (
            canonical_source_url(
                final_url
            )
            or (
                domain
                + "|"
                + title.lower()
            )
        )

        if dedupe_key in seen:
            continue

        seen.add(dedupe_key)

        results.append(
            {
                "rank": (
                    len(results) + 1
                ),
                "raw_serp_position": (
                    len(results) + 1
                ),
                "title": title,
                "url": final_url,
                "domain": domain,
                "description": (
                    result_snippet_from_lines(
                        block[
                            "lines"
                        ]
                    )
                ),
            }
        )

        if len(results) >= (
            num_results
        ):
            break

    locale_match = re.search(
        r"[?&]hl="
        r"([a-z]{2})"
        r"(?:-([A-Z]{2}))?",
        markdown,
    )

    observed_language = (
        locale_match.group(1)
        if locale_match
        else None
    )

    observed_country = (
        locale_match.group(2)
        if (
            locale_match
            and locale_match.group(2)
        )
        else None
    )

    requested_country = str(
        requested_country
        or ""
    ).upper()

    localization_warning = (
        bool(
            observed_country
            and requested_country
            and observed_country
            != requested_country
        )
    )

    return {
        "query": query,
        "engine": "google",
        "results": results,
        "raw_result_count": len(
            results
        ),
        "requested_country": (
            requested_country
        ),
        "observed_language": (
            observed_language
        ),
        "observed_country": (
            observed_country
        ),
        "localization_warning": (
            localization_warning
        ),
    }


def decode_bing_redirect(
    url,
):
    """
    Decode the destination embedded in Bing's u=a1... tracking
    parameter.
    """
    from urllib.parse import (
        parse_qs,
        unquote,
        urlparse,
    )

    import base64

    url = str(
        url or ""
    ).strip()

    if not url:
        return ""

    try:
        parsed = urlparse(url)
        query = parse_qs(
            parsed.query
        )

        encoded = (
            query.get("u", [""])[0]
        )

        encoded = unquote(
            encoded
        )

        if encoded.startswith(
            "a1"
        ):
            encoded = encoded[2:]

        if not encoded:
            return ""

        padding = (
            "="
            * (
                (4 - len(encoded) % 4)
                % 4
            )
        )

        decoded = (
            base64.urlsafe_b64decode(
                encoded + padding
            )
            .decode(
                "utf-8",
                errors="ignore",
            )
            .strip()
        )

        if decoded.startswith(
            ("http://", "https://")
        ):
            return decoded

    except Exception:
        pass

    return ""


def parse_bing_markdown(
    markdown,
    query,
    num_results=20,
    requested_country="US",
):
    """
    Parse classic Bing organic results while excluding ads, Copilot
    summaries, videos, images and related searches.
    """
    markdown = clean_serp_markdown(
        markdown
    )

    lines = markdown.splitlines()

    start_indexes = []

    for index, line in enumerate(
        lines
    ):
        match = re.match(
            r"^\s*(\d+)\.\s+(.*)$",
            line,
        )

        if match:
            start_indexes.append(
                (
                    index,
                    int(
                        match.group(1)
                    ),
                )
            )

    blocks = []

    for position, (
        start_index,
        raw_position,
    ) in enumerate(start_indexes):
        end_index = (
            start_indexes[
                position + 1
            ][0]
            if position + 1
            < len(start_indexes)
            else len(lines)
        )

        blocks.append(
            {
                "raw_position": (
                    raw_position
                ),
                "lines": lines[
                    start_index:
                    end_index
                ],
            }
        )

    rejected_markers = (
        "sponsored",
        "about our ads",
        "this summary was generated by ai",
        "content was generated with ai",
        "videos of ",
        "images of ",
        "people also search for",
        "deep dive into",
        "pagination",
        "some results have been removed",
    )

    results = []
    seen = set()

    for block in blocks:
        block_text = "\n".join(
            block["lines"]
        )

        block_lower = (
            block_text.lower()
        )

        if any(
            marker in block_lower
            for marker in (
                rejected_markers
            )
        ):
            continue

        title_match = re.search(
            r"^## \[(.+?)\]"
            r"\((https?://"
            r"(?:www\.)?bing\.com/"
            r"(?:ck/a|aclk)\?[^)]+)\)",
            block_text,
            flags=re.MULTILINE,
        )

        if not title_match:
            continue

        title = re.sub(
            r"\s+",
            " ",
            title_match.group(1),
        ).strip()

        if title.lower().startswith(
            (
                "videos of ",
                "images of ",
                "more videos",
            )
        ):
            continue

        tracking_url = (
            title_match.group(2)
        )

        decoded_url = (
            decode_bing_redirect(
                tracking_url
            )
        )

        displayed_url = ""

        for line in block["lines"]:
            candidate = (
                clean_displayed_url(
                    line
                )
            )

            if not candidate:
                continue

            hostname = get_hostname(
                candidate
            )

            if (
                hostname
                and not hostname.endswith(
                    "bing.com"
                )
                and not hostname.endswith(
                    "microsoft.com"
                )
            ):
                displayed_url = (
                    candidate
                )
                break

        final_url = (
            decoded_url
            or displayed_url
        )

        domain = get_root_domain(
            final_url
        )

        if not domain:
            continue

        if domain in {
            "bing.com",
            "microsoft.com",
        }:
            continue

        dedupe_key = (
            canonical_source_url(
                final_url
            )
            or (
                domain
                + "|"
                + title.lower()
            )
        )

        if dedupe_key in seen:
            continue

        seen.add(dedupe_key)

        results.append(
            {
                "rank": (
                    len(results) + 1
                ),
                "raw_serp_position": (
                    block[
                        "raw_position"
                    ]
                ),
                "title": title,
                "url": final_url,
                "domain": domain,
                "description": (
                    result_snippet_from_lines(
                        block[
                            "lines"
                        ]
                    )
                ),
            }
        )

        if len(results) >= (
            num_results
        ):
            break

    return {
        "query": query,
        "engine": "bing",
        "results": results,
        "raw_result_count": len(
            results
        ),
        "requested_country": str(
            requested_country
            or ""
        ).upper(),
        "observed_language": None,
        "observed_country": None,
        "localization_warning": False,
    }


def markdown_serp_request(
    self,
    query,
    engine,
    language="en",
    num_results=20,
):
    """
    Run the exact tested Markdown-zone request.

    The zone controls the Markdown output format. Google and Bing use
    the same payload; only the search-engine hostname changes.
    """
    engine = str(
        engine or ""
    ).strip().lower()

    if engine not in {
        "google",
        "bing",
    }:
        raise ValueError(
            f"Unsupported search engine: "
            f"{engine}"
        )

    country = str(
        self.country
        or "US"
    ).strip().lower()

    language = str(
        language
        or "en"
    ).strip().lower()

    search_url = (
        f"https://www.{engine}.com/search"
        f"?q={quote_plus(query)}"
        f"&hl={language}"
        f"&gl={country}"
    )

    response = requests.post(
        BD_REQUEST_URL,
        json={
            "zone": self.serp_zone,
            "url": search_url,
            "format": "raw",
            "method": "GET",
        },
        headers=self.headers,
        timeout=900,
    )

    if not response.ok:
        raise BrightDataAPIError(
            f"{engine.title()} Markdown request "
            f"failed. HTTP {response.status_code}: "
            f"{response.text[:1500]}"
        )

    markdown = str(
        response.text
        or ""
    ).strip()

    if not markdown:
        raise BrightDataAPIError(
            f"{engine.title()} returned "
            f"an empty response."
        )

    if engine == "google":
        parsed = parse_google_markdown(
            markdown=markdown,
            query=query,
            num_results=num_results,
            requested_country=(
                country.upper()
            ),
        )

    else:
        parsed = parse_bing_markdown(
            markdown=markdown,
            query=query,
            num_results=num_results,
            requested_country=(
                country.upper()
            ),
        )

    parsed["search_url"] = (
        search_url
    )

    parsed["markdown_length"] = (
        len(markdown)
    )

    return parsed



def search_result_quality(
    result,
    engine,
):
    """
    Google must contain at least five usable organic results.

    Bing is accepted when its response parsed successfully and contains
    at least one classic organic result. It is not rejected simply
    because Copilot, ads, videos, or images dominate the page.
    """
    if not isinstance(
        result,
        dict,
    ):
        return "unavailable"

    usable_results = [
        item
        for item in result.get(
            "results",
            [],
        )
        if (
            item.get("domain")
            and item.get("rank")
            is not None
        )
    ]

    result_count = len(
        usable_results
    )

    engine = str(
        engine or ""
    ).lower()

    if engine == "google":
        return (
            "available"
            if result_count >= 5
            else "unavailable"
        )

    if engine == "bing":
        return (
            "available"
            if result_count >= 1
            else "unavailable"
        )

    return "unavailable"




def choose_markdown_search_engine(
    self,
    test_query,
    requested_engine="auto",
):
    """
    Select one engine for the complete audit.

    Google:
    - Initial request
    - Three retries
    - Fifteen seconds between retries

    Bing:
    - One request
    - Either timeout/failure or a valid response
    """
    global ACTIVE_SEARCH_ENGINE
    global ACTIVE_SEARCH_STATUS

    requested_engine = str(
        requested_engine
        or "auto"
    ).strip().lower()

    if requested_engine not in {
        "auto",
        "google",
        "bing",
        "none",
    }:
        raise ValueError(
            "SEARCH_ENGINE must be auto, "
            "google, bing, or none."
        )

    if requested_engine == "none":
        ACTIVE_SEARCH_ENGINE = None
        ACTIVE_SEARCH_STATUS = (
            "unavailable"
        )

        self.active_search_engine = (
            None
        )

        self.log(
            "Traditional search disabled"
        )

        return None

    # --------------------------------------------------------
    # Google: initial request plus three retries
    # --------------------------------------------------------

    if requested_engine in {
        "auto",
        "google",
    }:
        total_attempts = 4

        for attempt in range(
            1,
            total_attempts + 1,
        ):
            self.log(
                f"Testing Google Markdown "
                f"attempt {attempt}/"
                f"{total_attempts}"
            )

            try:
                result = self.markdown_serp(
                    query=test_query,
                    engine="google",
                    num_results=20,
                )

                quality = (
                    search_result_quality(
                        result,
                        "google",
                    )
                )

                result_count = len(
                    result.get(
                        "results",
                        [],
                    )
                )

                self.log(
                    f"Google parser found "
                    f"{result_count} organic "
                    f"results; status={quality}"
                )

                if quality == "available":
                    MARKDOWN_SERP_CACHE[
                        (
                            "google",
                            test_query,
                        )
                    ] = result

                    ACTIVE_SEARCH_ENGINE = (
                        "google"
                    )

                    ACTIVE_SEARCH_STATUS = (
                        "available"
                    )

                    self.active_search_engine = (
                        "google"
                    )

                    self.log(
                        "Search engine selected: "
                        "Google"
                    )

                    return "google"

                self.log(
                    "Google response was incomplete "
                    "or could not be parsed.",
                    "yellow",
                )

            except Exception as exc:
                self.log(
                    f"Google attempt {attempt}/"
                    f"{total_attempts} failed: "
                    f"{exc}",
                    "yellow",
                )

            if attempt < total_attempts:
                self.log(
                    "Waiting 15 seconds before "
                    "retrying Google"
                )

                time.sleep(15)

        if requested_engine == "google":
            ACTIVE_SEARCH_ENGINE = None
            ACTIVE_SEARCH_STATUS = (
                "unavailable"
            )

            self.active_search_engine = (
                None
            )

            self.log(
                "Google unavailable after "
                "one initial request and three "
                "retries. Continuing without "
                "traditional search.",
                "yellow",
            )

            return None

    # --------------------------------------------------------
    # Bing: one request only
    # --------------------------------------------------------

    self.log(
        "Testing Bing Markdown search"
    )

    try:
        result = self.markdown_serp(
            query=test_query,
            engine="bing",
            num_results=20,
        )

        quality = (
            search_result_quality(
                result,
                "bing",
            )
        )

        result_count = len(
            result.get(
                "results",
                [],
            )
        )

        self.log(
            f"Bing parser found "
            f"{result_count} organic "
            f"results; status={quality}"
        )

        if quality == "available":
            MARKDOWN_SERP_CACHE[
                (
                    "bing",
                    test_query,
                )
            ] = result

            ACTIVE_SEARCH_ENGINE = (
                "bing"
            )

            ACTIVE_SEARCH_STATUS = (
                "available"
            )

            self.active_search_engine = (
                "bing"
            )

            self.log(
                "Search engine selected: "
                "Bing"
            )

            return "bing"

        self.log(
            "Bing responded, but no classic "
            "organic results could be parsed.",
            "yellow",
        )

    except requests.Timeout:
        self.log(
            "Bing request timed out.",
            "yellow",
        )

    except Exception as exc:
        self.log(
            f"Bing request failed: {exc}",
            "yellow",
        )

    ACTIVE_SEARCH_ENGINE = None
    ACTIVE_SEARCH_STATUS = (
        "unavailable"
    )

    self.active_search_engine = None

    self.log(
        "Traditional search unavailable. "
        "Continuing with AI visibility "
        "and source analysis.",
        "yellow",
    )

    return None




def markdown_search_serp(
    self,
    query,
    engine=None,
    language="en",
    num_results=20,
):
    selected_engine = str(
        engine
        or getattr(
            self,
            "active_search_engine",
            "",
        )
    ).strip().lower()

    if not selected_engine:
        raise BrightDataAPIError(
            "No search engine is active."
        )

    cache_key = (
        selected_engine,
        query,
    )

    if cache_key in (
        MARKDOWN_SERP_CACHE
    ):
        return (
            MARKDOWN_SERP_CACHE.pop(
                cache_key
            )
        )

    return self.markdown_serp(
        query=query,
        engine=selected_engine,
        language=language,
        num_results=num_results,
    )


BrightDataClient.markdown_serp = (
    markdown_serp_request
)

BrightDataClient.choose_search_engine = (
    choose_markdown_search_engine
)

BrightDataClient.search_serp = (
    markdown_search_serp
)


In [ ]:
# ============================================================
# Generic direct-competitor validation
# ============================================================

DIRECT_COMPETITOR_TYPES = {
    "direct_competitor",
    "direct competitor",
    "competitor",
}

ADJACENT_COMPETITOR_TYPES = {
    "adjacent_competitor",
    "adjacent competitor",
    "indirect_competitor",
    "indirect competitor",
}

INTERMEDIARY_TYPES = {
    "retailer",
    "marketplace",
    "publisher",
    "review_site",
    "review site",
    "comparison_site",
    "comparison site",
    "directory",
    "distributor",
    "reseller",
    "community",
    "forum",
    "informational_resource",
    "informational resource",
    "media",
    "news",
    "unrelated",
}


def build_candidate_validation_prompt(
    target_brand,
    candidate,
):
    """
    Determine whether the candidate competes at the same value-chain
    level for the same primary customer and purchase decision.
    """
    return f"""
Inspect the current public website for this candidate:

Candidate domain: {candidate.domain}
Candidate URL: {candidate.homepage_url}

Compare it with this target:

Target name: {target_brand.brand_name}
Target website: {target_brand.official_url}
Target category: {target_brand.category}
Target positioning: {shorten(target_brand.positioning, 220)}
Target customers: {", ".join(target_brand.target_customers[:5])}
Target offerings: {", ".join(target_brand.products[:5])}

Determine the target's and candidate's positions in the market value
chain.

Possible roles include:

- manufacturer
- product_brand
- software_vendor
- service_provider
- professional_practice
- clinic_or_healthcare_provider
- retailer
- marketplace
- distributor
- reseller
- publisher
- review_site
- directory
- community
- educational_resource
- government_or_regulator
- supplier
- customer_or_downstream_provider
- other

A direct competitor must satisfy all of these conditions:

1. It sells, manufactures, provides, or operates its own offering.
2. It serves substantially the same primary customer or buyer.
3. It operates at substantially the same level of the value chain.
4. Its offering could substitute for the target in the same purchase
   or selection decision.
5. A customer would realistically compare the target and candidate
   before choosing one.

Examples of relationships that are not direct competition:

- Manufacturer versus clinic using the manufacturer's products
- Brand versus retailer selling that brand category
- Software vendor versus consulting company implementing software
- Hotel versus booking marketplace
- University versus course-review website
- Restaurant versus food-delivery marketplace
- Bank versus financial-news publisher
- Consumer brand versus product-review publication

Return only JSON:

{{
  "candidate_name": "canonical name",
  "candidate_domain": "{candidate.domain}",
  "official_url": "{candidate.homepage_url}",
  "target_market_role": "target role",
  "candidate_market_role": "candidate role",
  "target_primary_customer": "target's primary buyer or customer",
  "candidate_primary_customer": "candidate's primary buyer or customer",
  "candidate_type": "direct_competitor, adjacent_competitor, supplier, customer_or_downstream_provider, retailer, marketplace, distributor, reseller, publisher, review_site, directory, community, informational_resource, or unrelated",
  "sells_own_offering": true,
  "same_primary_customer": true,
  "same_value_chain_role": true,
  "offering_is_substitute": true,
  "addresses_same_customer_need": true,
  "is_direct_competitor": true,
  "reason": "concise explanation",
  "evidence": [
    "specific evidence from the candidate website"
  ],
  "confidence": 0.0
}}

Do not classify a company as a direct competitor merely because it
operates in the same industry or appears for the same search query.

Return JSON only without Markdown fences.
""".strip()



def build_candidate_structuring_prompt(
    target_brand,
    candidate,
    research_text,
):
    """
    Structure narrative validation research without performing new
    web research.
    """
    research_text = shorten(
        research_text,
        1500,
    )

    prompt = f"""
Convert the supplied candidate-validation research into JSON. Do not
perform new web research.

Target:
- Name: {target_brand.brand_name}
- Category: {target_brand.category}
- Customers: {", ".join(target_brand.target_customers[:4])}
- Offerings: {", ".join(target_brand.products[:4])}

Candidate:
- Domain: {candidate.domain}
- URL: {candidate.homepage_url}

Research:
{research_text}

A direct competitor must serve substantially the same primary buyer,
operate at the same value-chain level, and offer something that could
substitute for the target in the same selection decision.

Return only:

{{
  "candidate_name": "canonical name",
  "candidate_domain": "{candidate.domain}",
  "official_url": "{candidate.homepage_url}",
  "target_market_role": "target role",
  "candidate_market_role": "candidate role",
  "target_primary_customer": "target customer",
  "candidate_primary_customer": "candidate customer",
  "candidate_type": "direct_competitor, adjacent_competitor, supplier, customer_or_downstream_provider, retailer, marketplace, distributor, reseller, publisher, review_site, directory, community, informational_resource, or unrelated",
  "sells_own_offering": true,
  "same_primary_customer": true,
  "same_value_chain_role": true,
  "offering_is_substitute": true,
  "addresses_same_customer_need": true,
  "is_direct_competitor": true,
  "reason": "concise explanation",
  "evidence": ["evidence from the supplied research"],
  "confidence": 0.0
}}

Return JSON only.
""".strip()

    if len(prompt) > 3900:
        raise ValueError(
            "Candidate structuring prompt "
            "is too long."
        )

    return prompt



def normalize_candidate_validation(
    data,
    candidate,
):
    """
    Enforce buyer, substitution, and value-chain requirements locally.
    """
    if not isinstance(
        data,
        dict,
    ):
        data = {}

    candidate_type = str(
        data.get(
            "candidate_type"
        )
        or "unrelated"
    ).strip().lower()

    candidate_type = (
        candidate_type
        .replace("-", "_")
        .replace(" ", "_")
    )

    sells_own_offering = (
        normalize_boolean(
            data.get(
                "sells_own_offering",
                False,
            )
        )
    )

    same_primary_customer = (
        normalize_boolean(
            data.get(
                "same_primary_customer",
                False,
            )
        )
    )

    same_value_chain_role = (
        normalize_boolean(
            data.get(
                "same_value_chain_role",
                False,
            )
        )
    )

    offering_is_substitute = (
        normalize_boolean(
            data.get(
                "offering_is_substitute",
                False,
            )
        )
    )

    same_customer_need = (
        normalize_boolean(
            data.get(
                "addresses_same_customer_need",
                False,
            )
        )
    )

    ai_direct = (
        normalize_boolean(
            data.get(
                "is_direct_competitor",
                False,
            )
        )
    )

    non_competitor_types = {
        "retailer",
        "marketplace",
        "publisher",
        "review_site",
        "comparison_site",
        "directory",
        "distributor",
        "reseller",
        "community",
        "informational_resource",
        "government_or_regulator",
        "customer_or_downstream_provider",
        "supplier",
        "unrelated",
    }

    is_direct = all(
        [
            ai_direct,
            sells_own_offering,
            same_primary_customer,
            same_value_chain_role,
            offering_is_substitute,
            same_customer_need,
            candidate_type
            not in non_competitor_types,
        ]
    )

    name = str(
        data.get(
            "candidate_name"
        )
        or data.get(
            "brand_name"
        )
        or candidate.domain
    ).strip()

    if not name:
        name = candidate.domain

    official_url = (
        normalize_public_url(
            data.get(
                "official_url"
            )
            or candidate.homepage_url
        )
        or candidate.homepage_url
    )

    reason = str(
        data.get("reason")
        or ""
    ).strip()

    if not is_direct and not reason:
        reason = (
            "Candidate does not share the target's "
            "primary customer, value-chain role, "
            "or purchase decision."
        )

    return {
        "candidate_name": name,
        "candidate_domain": (
            candidate.domain
        ),
        "official_url": (
            official_url
        ),
        "target_market_role": str(
            data.get(
                "target_market_role"
            )
            or ""
        ).strip(),
        "candidate_market_role": str(
            data.get(
                "candidate_market_role"
            )
            or ""
        ).strip(),
        "target_primary_customer": str(
            data.get(
                "target_primary_customer"
            )
            or ""
        ).strip(),
        "candidate_primary_customer": str(
            data.get(
                "candidate_primary_customer"
            )
            or ""
        ).strip(),
        "candidate_type": (
            candidate_type
        ),
        "sells_own_offering": (
            sells_own_offering
        ),
        "same_primary_customer": (
            same_primary_customer
        ),
        "same_value_chain_role": (
            same_value_chain_role
        ),
        "offering_is_substitute": (
            offering_is_substitute
        ),
        "addresses_same_customer_need": (
            same_customer_need
        ),
        "is_direct_competitor": (
            is_direct
        ),
        "reason": reason,
        "evidence": (
            ensure_string_list(
                data.get("evidence")
            )[:5]
        ),
        "confidence": (
            normalize_confidence(
                data.get(
                    "confidence"
                )
            )
        ),
    }



def validate_candidate_sync(
    target_brand,
    candidate,
):
    """
    Validate one candidate.

    Google AI Mode performs the website research. ChatGPT is used only
    when Google AI Mode returns prose instead of JSON.
    """
    started_at = time.monotonic()

    prompt = (
        build_candidate_validation_prompt(
            target_brand,
            candidate,
        )
    )

    try:
        research_record = (
            bd_client.google_ai_mode(
                prompt,
                timeout_seconds=720,
            )
        )

        research_text = (
            bd_client.answer_text(
                research_record
            )
        )

        try:
            parsed = parse_ai_json(
                research_text
            )

            method = (
                "google_ai_mode_json"
            )

            structuring_record = None
            structuring_snapshot_id = None

        except Exception:
            structuring_prompt = (
                build_candidate_structuring_prompt(
                    target_brand=target_brand,
                    candidate=candidate,
                    research_text=research_text,
                )
            )

            structured = (
                run_chatgpt_without_web(
                    structuring_prompt,
                    timeout_seconds=900,
                )
            )

            parsed = parse_ai_json(
                structured["answer"]
            )

            method = (
                "google_ai_mode_research_"
                "chatgpt_structuring"
            )

            structuring_record = (
                structured["record"]
            )

            structuring_snapshot_id = (
                structured["snapshot_id"]
            )

        validation = (
            normalize_candidate_validation(
                parsed,
                candidate,
            )
        )

        return {
            "status": "success",
            "candidate": candidate,
            "validation": validation,
            "method": method,
            "duration_seconds": round(
                time.monotonic()
                - started_at,
                2,
            ),
            "research_record": (
                research_record
            ),
            "structuring_record": (
                structuring_record
            ),
            "structuring_snapshot_id": (
                structuring_snapshot_id
            ),
            "error": None,
        }

    except Exception as exc:
        return {
            "status": "failed",
            "candidate": candidate,
            "validation": {
                "candidate_name": (
                    candidate.domain
                ),
                "candidate_domain": (
                    candidate.domain
                ),
                "official_url": (
                    candidate.homepage_url
                ),
                "candidate_type": (
                    "validation_error"
                ),
                "sells_own_offering": False,
                "addresses_same_customer_need": (
                    False
                ),
                "is_direct_competitor": (
                    False
                ),
                "reason": str(exc),
                "evidence": [],
                "confidence": 0.0,
            },
            "method": "failed",
            "duration_seconds": round(
                time.monotonic()
                - started_at,
                2,
            ),
            "research_record": None,
            "structuring_record": None,
            "structuring_snapshot_id": (
                None
            ),
            "error": str(exc),
        }


def validate_candidate_batch(
    target_brand,
    candidates,
    max_workers=4,
):
    """
    Validate a batch of candidates concurrently.
    """
    indexed_results = {}

    with ThreadPoolExecutor(
        max_workers=min(
            max_workers,
            len(candidates),
        )
    ) as executor:
        future_map = {
            executor.submit(
                validate_candidate_sync,
                target_brand,
                candidate,
            ): index
            for index, candidate
            in enumerate(candidates)
        }

        for future in as_completed(
            future_map
        ):
            index = future_map[
                future
            ]

            try:
                indexed_results[
                    index
                ] = future.result()
            except Exception as exc:
                candidate = candidates[
                    index
                ]

                indexed_results[
                    index
                ] = {
                    "status": "failed",
                    "candidate": candidate,
                    "validation": {
                        "candidate_name": (
                            candidate.domain
                        ),
                        "candidate_domain": (
                            candidate.domain
                        ),
                        "official_url": (
                            candidate.homepage_url
                        ),
                        "candidate_type": (
                            "validation_error"
                        ),
                        "sells_own_offering": (
                            False
                        ),
                        "addresses_same_customer_need": (
                            False
                        ),
                        "is_direct_competitor": (
                            False
                        ),
                        "reason": str(exc),
                        "evidence": [],
                        "confidence": 0.0,
                    },
                    "method": "failed",
                    "error": str(exc),
                }

    return [
        indexed_results[index]
        for index in range(
            len(candidates)
        )
    ]


# ============================================================
# Direct-competitor guardrails
# ============================================================







# ============================================================
# Google AI Mode customer-question discovery
# ============================================================

LAST_AI_MODE_DISCOVERY = None


def normalize_citation(
    citation,
    position,
):
    """
    Normalize an AI Mode citation and resolve opaque Google redirects.
    """
    if not isinstance(
        citation,
        dict,
    ):
        return None

    original_url = str(
        citation.get("url")
        or citation.get("link")
        or citation.get("href")
        or ""
    ).strip()

    if not original_url:
        return None

    resolved_url = (
        resolve_google_goto_url(
            original_url
        )
    )

    final_url = (
        resolved_url
        or original_url
    )

    # If the Google redirect could not be resolved, its google.com
    # domain is not useful for competitor/source classification.
    if (
        is_google_goto_url(
            final_url
        )
    ):
        return None

    domain = get_root_domain(
        final_url
    )

    if not domain:
        return None

    return {
        "position": position,
        "title": str(
            citation.get("title")
            or citation.get("name")
            or domain
        ).strip(),
        "url": final_url,
        "original_url": (
            original_url
        ),
        "resolved": bool(
            resolved_url
            and resolved_url
            != original_url
        ),
        "domain": domain,
    }



async def run_ai_mode_question(
    question,
    question_index,
):
    """
    Ask one neutral customer question through Google AI Mode.
    """
    started_at = time.monotonic()

    prompt = f"""
A customer is researching this need:

{question}

Recommend and compare the leading brands, products, services, or
providers that genuinely address this need.

Use evaluation criteria relevant to the category. Use current public
web information. Do not ask follow-up questions.
""".strip()

    try:
        record = await asyncio.to_thread(
            bd_client.google_ai_mode,
            prompt,
            720,
        )

        answer = (
            bd_client.answer_text(
                record
            )
        )

        raw_citations = (
            record.get(
                "citations",
                [],
            )
            if isinstance(
                record,
                dict,
            )
            else []
        )

        citations = []

        for position, citation in enumerate(
            raw_citations,
            start=1,
        ):
            normalized = normalize_citation(
                citation,
                position,
            )

            if normalized:
                citations.append(
                    normalized
                )

        return {
            "question_index": (
                question_index
            ),
            "question": question,
            "prompt": prompt,
            "success": bool(answer),
            "answer": answer,
            "citations": citations,
            "duration_seconds": round(
                time.monotonic()
                - started_at,
                2,
            ),
            "error": None,
        }

    except Exception as exc:
        return {
            "question_index": (
                question_index
            ),
            "question": question,
            "prompt": prompt,
            "success": False,
            "answer": "",
            "citations": [],
            "duration_seconds": round(
                time.monotonic()
                - started_at,
                2,
            ),
            "error": str(exc),
        }


def build_ai_mode_source_candidates(
    question_results,
    target_domain,
):
    """
    Convert recurring AI Mode citation domains into candidate records.
    """
    target_root = get_root_domain(
        target_domain
    )

    excluded = {
        target_root,
        "google.com",
        "youtube.com",
        "facebook.com",
        "instagram.com",
        "linkedin.com",
        "twitter.com",
        "x.com",
        "tiktok.com",
        "pinterest.com",
        "wikipedia.org",
    }

    source_data = defaultdict(
        lambda: {
            "questions": set(),
            "positions": [],
            "urls": [],
            "titles": [],
        }
    )

    for result in question_results:
        if not result.get(
            "success"
        ):
            continue

        question = result[
            "question"
        ]

        seen_for_question = set()

        for citation in result.get(
            "citations",
            [],
        ):
            domain = citation[
                "domain"
            ]

            if (
                not domain
                or domain in excluded
                or domain
                in seen_for_question
            ):
                continue

            seen_for_question.add(
                domain
            )

            source_data[domain][
                "questions"
            ].add(question)

            source_data[domain][
                "positions"
            ].append(
                citation["position"]
            )

            source_data[domain][
                "urls"
            ].append(
                citation["url"]
            )

            source_data[domain][
                "titles"
            ].append(
                citation["title"]
            )

    candidates = []

    total_questions = max(
        1,
        len(question_results),
    )

    for domain, data in (
        source_data.items()
    ):
        positions = data[
            "positions"
        ]

        frequency = len(
            data["questions"]
        )

        preferred_url = (
            data["urls"][0]
            if data["urls"]
            else f"https://{domain}/"
        )

        preferred_hostname = (
            get_hostname(
                preferred_url
            )
            or domain
        )

        rank_score = sum(
            1 / max(position, 1)
            for position in positions
        )

        candidates.append(
            CompetitorCandidate(
                domain=domain,
                preferred_hostname=(
                    preferred_hostname
                ),
                homepage_url=(
                    f"https://"
                    f"{preferred_hostname}/"
                ),
                frequency=frequency,
                keyword_coverage=round(
                    frequency
                    / total_questions,
                    4,
                ),
                best_rank=min(
                    positions
                ),
                average_rank=round(
                    sum(positions)
                    / len(positions),
                    2,
                ),
                rank_score=round(
                    rank_score,
                    4,
                ),
                total_score=round(
                    frequency * 80
                    + rank_score * 20,
                    2,
                ),
                matched_keywords=sorted(
                    data["questions"]
                ),
                serp_urls=list(
                    dict.fromkeys(
                        data["urls"]
                    )
                ),
                serp_titles=list(
                    dict.fromkeys(
                        data["titles"]
                    )
                ),
            )
        )

    candidates.sort(
        key=lambda item: (
            -item.frequency,
            -item.total_score,
            item.average_rank,
            item.domain,
        )
    )

    return candidates


def merge_discovery_candidates(
    serp_candidates,
    ai_mode_candidates,
):
    """
    Merge candidates without treating SERP and AI citation positions
    as the same metric.
    """
    merged = {
        candidate.domain: candidate
        for candidate in serp_candidates
    }

    for ai_candidate in (
        ai_mode_candidates
    ):
        existing = merged.get(
            ai_candidate.domain
        )

        if existing is None:
            merged[
                ai_candidate.domain
            ] = ai_candidate

            continue

        existing.total_score = round(
            existing.total_score
            + ai_candidate.frequency * 40
            + ai_candidate.rank_score * 10,
            2,
        )

        existing.matched_keywords = list(
            dict.fromkeys(
                [
                    *existing.matched_keywords,
                    *[
                        "AI Mode: " + item
                        for item in (
                            ai_candidate
                            .matched_keywords
                        )
                    ],
                ]
            )
        )

        existing.serp_urls = list(
            dict.fromkeys(
                [
                    *existing.serp_urls,
                    *ai_candidate.serp_urls,
                ]
            )
        )

        existing.serp_titles = list(
            dict.fromkeys(
                [
                    *existing.serp_titles,
                    *ai_candidate.serp_titles,
                ]
            )
        )

    merged_list = list(
        merged.values()
    )

    merged_list.sort(
        key=lambda item: (
            -item.total_score,
            -item.frequency,
            item.average_rank,
            item.domain,
        )
    )

    return merged_list


#@title 3B. Load analysis pipeline
#@markdown Google AI Mode research, ChatGPT structuring, parallel SERPs, competitor selection, and profiles.
#@markdown This cell normally does not need to be edited.

# ============================================================
# Input normalization
# ============================================================

def normalize_keyword_records(
    raw_keywords,
):
    if isinstance(raw_keywords, str):
        raw_keywords = [
            item.strip()
            for item in raw_keywords.split(",")
            if item.strip()
        ]

    if not isinstance(raw_keywords, list):
        return []

    normalized = []
    seen = set()

    for item in raw_keywords:
        if isinstance(item, str):
            keyword = item.strip()

            record = {
                "keyword": keyword,
                "intent": "commercial",
                "rationale": "",
            }

        elif isinstance(item, dict):
            keyword = str(
                item.get("keyword")
                or item.get("query")
                or item.get("term")
                or ""
            ).strip()

            record = {
                "keyword": keyword,
                "intent": str(
                    item.get("intent")
                    or "commercial"
                ).strip(),
                "rationale": str(
                    item.get("rationale")
                    or item.get("reason")
                    or ""
                ).strip(),
            }

        else:
            continue

        key = keyword.lower()

        if not key or key in seen:
            continue

        seen.add(key)
        normalized.append(record)

    return normalized


def normalize_company_intake(
    data,
    company_name,
    company_url,
):
    if not isinstance(data, dict):
        raise ValueError(
            "Company analysis must be a JSON object."
        )

    if "brand" not in data:
        brand_keys = {
            "brand_name",
            "official_url",
            "domain",
            "category",
            "description",
            "positioning",
            "target_customers",
            "products",
            "key_features",
            "differentiators",
            "confidence",
            "evidence",
        }

        data["brand"] = {
            key: data[key]
            for key in data
            if key in brand_keys
        }

    brand = data.get("brand") or {}

    if not isinstance(brand, dict):
        brand = {}

    official_url = normalize_public_url(
        brand.get("official_url")
        or company_url
    )

    if not official_url:
        official_url = normalize_public_url(
            company_url
        )

    domain = get_root_domain(
        brand.get("domain")
        or official_url
    )

    brand["brand_name"] = str(
        brand.get("brand_name")
        or company_name
    ).strip()

    if not brand["brand_name"]:
        brand["brand_name"] = company_name

    brand["official_url"] = official_url
    brand["domain"] = domain

    for field_name in (
        "category",
        "description",
        "positioning",
    ):
        brand[field_name] = str(
            brand.get(field_name)
            or ""
        ).strip()

    for field_name in (
        "target_customers",
        "products",
        "key_features",
        "differentiators",
        "evidence",
    ):
        brand[field_name] = (
            ensure_string_list(
                brand.get(field_name)
            )[:8]
        )

    brand["confidence"] = (
        normalize_confidence(
            brand.get("confidence")
        )
    )

    keywords = normalize_keyword_records(
        data.get("buyer_intent_keywords")
        or data.get("keywords")
        or []
    )

    normalized = {
        "brand": brand,
        "buyer_intent_keywords": keywords,
    }

    return validate_model(
        CompanyIntake,
        normalized,
    )


# ============================================================
# Reliable company-analysis workflow
# ============================================================

def run_chatgpt_without_web(
    prompt,
    timeout_seconds=900,
):
    """
    Run one ChatGPT snapshot with web search disabled.

    The longer timeout avoids abandoning a valid structuring snapshot
    shortly before it completes.
    """
    if len(prompt) > 4096:
        raise ValueError(
            f"ChatGPT transformation prompt is too long: "
            f"{len(prompt)} characters."
        )

    item = {
        "url": "https://chatgpt.com/",
        "prompt": prompt,
        "country": bd_client.country,
        "web_search": False,
    }

    snapshot_id = (
        bd_client.trigger_dataset(
            CHATGPT_DATASET_ID,
            [item],
        )
    )

    bd_client.log(
        f"ChatGPT transformation snapshot: "
        f"{snapshot_id}"
    )

    records = (
        bd_client.wait_for_snapshot(
            snapshot_id,
            timeout_seconds=timeout_seconds,
        )
    )

    for record in records:
        answer = (
            bd_client.answer_text(
                record
            )
        )

        if answer:
            return {
                "snapshot_id": (
                    snapshot_id
                ),
                "record": record,
                "answer": answer,
            }

    raise BrightDataAPIError(
        "ChatGPT transformation snapshot "
        "returned no answer text."
    )



def build_company_research_prompt(
    settings,
):
    """
    Ask Google AI Mode to research any company, brand, product,
    service, institution, or local business.
    """
    audit_focus = str(
        settings.get(
            "audit_focus",
            "",
        )
        or ""
    ).strip()

    focus_instruction = (
        f"Specific audit focus: {audit_focus}"
        if audit_focus
        else (
            "Audit focus: infer the primary product, service, "
            "offering, or customer need represented by the website."
        )
    )

    return f"""
Analyze the current public website for this organization or brand.

Name: {settings["company_name"]}
Website: {settings["company_url"]}
Country: {settings["country"]}
{focus_instruction}

First determine what kind of market this is, such as consumer product,
B2B product, software, professional service, local business,
health/beauty product, retailer, financial product, education,
hospitality, or another category.

Provide a concise research brief covering:

- Canonical brand, organization, or product name
- The specific product, service, or offering being audited
- Market and product category
- Intended customer, user, or audience
- Primary customer need or problem addressed
- Main products, services, or alternatives offered
- Important features, benefits, claims, or capabilities
- Relevant proof, trust signals, ingredients, specifications, or
  evidence when applicable
- Meaningful differentiators
- Price, price tier, pricing approach, or availability when public
- The criteria a real customer would use to compare alternatives
- Exactly eight non-branded buyer-intent searches that could be used
  to find this offering and competing alternatives

The buyer searches must match the actual market. They may be consumer,
commercial, transactional, local, or solution-evaluation searches.

Do not include the brand name or competitor names in the searches.

Use current public information. Keep the response concise. Do not ask
follow-up questions.
""".strip()



def build_company_structuring_prompt(
    settings,
    research_text,
    strict_retry=False,
):
    """
    Select relevant evidence from the complete research answer instead
    of blindly taking its first 1,700 characters.
    """
    audit_focus = str(
        settings.get(
            "audit_focus",
            "",
        )
        or ""
    ).strip()

    relevant_research = (
        select_relevant_company_research(
            research_text=research_text,
            company_name=(
                settings[
                    "company_name"
                ]
            ),
            company_domain=(
                settings[
                    "company_domain"
                ]
            ),
            audit_focus=(
                audit_focus
            ),
            max_characters=2400,
        )
    )

    retry_instruction = (
        "A previous formatting attempt failed. "
        "Return the JSON object directly."
        if strict_retry
        else ""
    )

    prompt = f"""
Structure the supplied company research. Do not perform new research.

Known company: {settings["company_name"]}
Known URL: {settings["company_url"]}
Known domain: {settings["company_domain"]}
Audit focus: {audit_focus or "infer the primary offering"}

RESEARCH
--------
{relevant_research}
--------
END RESEARCH

Return only JSON:

{{
  "brand": {{
    "brand_name": "canonical name",
    "official_url": "{settings["company_url"]}",
    "domain": "{settings["company_domain"]}",
    "category": "specific market category",
    "description": "one or two sentences",
    "positioning": "customer-facing positioning",
    "target_customers": ["specific buyer or audience"],
    "products": ["specific product, service, or offering"],
    "key_features": ["specific feature, benefit, claim, or attribute"],
    "differentiators": ["meaningful differentiator"],
    "confidence": 0.0,
    "evidence": ["evidence from the supplied research"]
  }},
  "buyer_intent_keywords": [
    {{
      "keyword": "specific non-branded buyer search",
      "intent": "commercial",
      "rationale": "short reason"
    }}
  ]
}}

Requirements:
- Return exactly eight unique buyer searches.
- Every search must clearly relate to the company's actual market.
- Do not use generic placeholders such as products and services,
  primary offering, product options, service options, or buy products.
- Do not include the audited company name.
- Do not invent unsupported information.
- Return JSON only.

{retry_instruction}
""".strip()

    if len(prompt) > 3900:
        raise ValueError(
            f"Company structuring prompt is too long: "
            f"{len(prompt)} characters."
        )

    if bd_client.debug:
        bd_client.log(
            f"Company structuring prompt: "
            f"{len(prompt)} characters; "
            f"selected research: "
            f"{len(relevant_research)} characters"
        )

    return prompt





def complete_company_keywords(
    settings,
    brand,
    current_keywords,
):
    """
    Complete a buyer-intent keyword set for any product, service,
    organization, or consumer category.
    """
    existing = [
        item.keyword
        for item in current_keywords
    ]

    audit_focus = str(
        settings.get(
            "audit_focus",
            "",
        )
        or ""
    ).strip()

    prompt = f"""
Using only the supplied information, return exactly eight unique
non-branded buyer-intent searches appropriate for this market.

Category: {brand.category}
Audit focus: {audit_focus or "primary offering"}
Positioning: {brand.positioning}
Offerings: {", ".join(brand.products[:5])}
Attributes or benefits: {", ".join(brand.key_features[:6])}

Existing keywords:
{json.dumps(existing, ensure_ascii=False)}

The searches should sound like something a real customer would enter
when looking for, evaluating, comparing, or buying alternatives.

They may be consumer, B2B, local, commercial, transactional, or
solution-evaluation searches depending on the category.

Return only JSON:

{{
  "buyer_intent_keywords": [
    {{
      "keyword": "buyer query",
      "intent": "commercial",
      "rationale": "short reason"
    }}
  ]
}}

Do not include the audited brand or competitor names.
""".strip()

    result = run_chatgpt_without_web(
        prompt
    )

    parsed = parse_ai_json(
        result["answer"]
    )

    completed = normalize_keyword_records(
        parsed.get(
            "buyer_intent_keywords"
        )
        or parsed.get("keywords")
        or []
    )

    combined = []
    seen = set()

    audited_name = (
        settings["company_name"]
        .lower()
        .strip()
    )

    for item in [
        *[
            model_to_dict(keyword)
            for keyword in current_keywords
        ],
        *completed,
    ]:
        keyword = str(
            item.get("keyword")
            or ""
        ).strip()

        key = keyword.lower()

        if not key or key in seen:
            continue

        if audited_name in key:
            continue

        seen.add(key)

        combined.append(
            validate_model(
                BuyerIntentKeyword,
                item,
            )
        )

        if len(combined) == 8:
            break

    return {
        "keywords": combined,
        "record": result["record"],
        "snapshot_id": (
            result["snapshot_id"]
        ),
    }



def analyze_company_stage(
    settings,
):
    """
    Reliable company analysis:

    1. Google AI Mode performs natural-language research.
    2. ChatGPT transforms the research into JSON.
    3. A second ChatGPT formatting attempt runs only if necessary.
    4. Keyword completion runs only if fewer than eight were returned.
    """
    bd_client.log(
        "Starting Google AI Mode company research"
    )

    research_prompt = (
        build_company_research_prompt(
            settings
        )
    )

    research_record = (
        bd_client.google_ai_mode(
            research_prompt,
            timeout_seconds=720,
        )
    )

    research_text = (
        bd_client.answer_text(
            research_record
        )
    )

    if not research_text:
        raise BrightDataAPIError(
            "Google AI Mode returned no "
            "company research text."
        )

    bd_client.log(
        f"Google AI Mode research returned "
        f"{len(research_text):,} characters"
    )

    structuring_errors = []
    structured_result = None
    intake = None

    for attempt in (
        1,
        2,
    ):
        bd_client.log(
            f"Starting ChatGPT structuring "
            f"attempt {attempt}/2"
        )

        structuring_prompt = (
            build_company_structuring_prompt(
                settings=settings,
                research_text=research_text,
                strict_retry=(
                    attempt == 2
                ),
            )
        )

        try:
            candidate_result = (
                run_chatgpt_without_web(
                    structuring_prompt
                )
            )

            parsed = parse_ai_json(
                candidate_result[
                    "answer"
                ]
            )

            candidate_intake = (
                normalize_company_intake(
                    data=parsed,
                    company_name=settings[
                        "company_name"
                    ],
                    company_url=settings[
                        "company_url"
                    ],
                )
            )

            structured_result = (
                candidate_result
            )

            intake = candidate_intake
            break

        except Exception as exc:
            structuring_errors.append(
                f"Attempt {attempt}: "
                f"{type(exc).__name__}: "
                f"{exc}"
            )

            bd_client.log(
                f"ChatGPT structuring attempt "
                f"{attempt} failed: {exc}",
                "yellow",
            )

    if intake is None:
        raise BrightDataAPIError(
            "ChatGPT could not structure the "
            "Google AI Mode research.\n- "
            + "\n- ".join(
                structuring_errors
            )
        )

    keyword_completion = None

    if len(
        intake.buyer_intent_keywords
    ) != 8:
        bd_client.log(
            f"Structured result contained "
            f"{len(intake.buyer_intent_keywords)} "
            f"keywords; completing the set"
        )

        keyword_completion = (
            complete_company_keywords(
                settings=settings,
                brand=intake.brand,
                current_keywords=(
                    intake.buyer_intent_keywords
                ),
            )
        )

        intake.buyer_intent_keywords = (
            keyword_completion[
                "keywords"
            ]
        )

    if len(
        intake.buyer_intent_keywords
    ) != 8:
        raise BrightDataAPIError(
            "The company-analysis workflow "
            f"produced "
            f"{len(intake.buyer_intent_keywords)} "
            f"keywords instead of eight."
        )

    return {
        "intake": intake,

        # Preserve Google AI Mode as the main
        # research record expected by the orchestrator.
        "record": research_record,

        "prompt": research_prompt,

        "research_text": research_text,

        "structuring_record": (
            structured_result[
                "record"
            ]
        ),

        "structuring_snapshot_id": (
            structured_result[
                "snapshot_id"
            ]
        ),

        "keyword_completion": (
            keyword_completion
        ),

        "workflow": (
            "google_ai_research_"
            "chatgpt_structuring"
        ),
    }


# ============================================================
# SERP execution with retries
# ============================================================

async def run_keyword_serp_task(
    keyword,
    semaphore,
    num_results=20,
    search_engine=None,
):
    """
    Google receives one initial request plus three retries.

    Bing receives one request only.
    """
    async with semaphore:
        engine = str(
            search_engine
            or ""
        ).lower()

        max_attempts = (
            4
            if engine == "google"
            else 1
        )

        last_error = None

        for attempt in range(
            1,
            max_attempts + 1,
        ):
            try:
                response = (
                    await asyncio.to_thread(
                        bd_client.search_serp,
                        keyword,
                        engine,
                        "en",
                        num_results,
                    )
                )

                quality = (
                    search_result_quality(
                        response,
                        engine,
                    )
                )

                if quality != "available":
                    raise BrightDataAPIError(
                        f"{engine.title()} returned "
                        f"no usable organic results."
                    )

                return {
                    "keyword": keyword,
                    "engine": engine,
                    "success": True,
                    "results": (
                        response.get(
                            "results",
                            [],
                        )
                    ),
                    "raw_result_count": (
                        response.get(
                            "raw_result_count",
                            0,
                        )
                    ),
                    "requested_country": (
                        response.get(
                            "requested_country"
                        )
                    ),
                    "observed_country": (
                        response.get(
                            "observed_country"
                        )
                    ),
                    "localization_warning": (
                        response.get(
                            "localization_warning",
                            False,
                        )
                    ),
                    "attempt": attempt,
                    "error": None,
                }

            except Exception as exc:
                last_error = exc

                bd_client.log(
                    f"{engine.title()} attempt "
                    f"{attempt}/{max_attempts} "
                    f"failed for {keyword!r}: "
                    f"{exc}",
                    "yellow",
                )

                if (
                    engine == "google"
                    and attempt
                    < max_attempts
                ):
                    bd_client.log(
                        "Waiting 15 seconds before "
                        f"retrying {keyword!r}"
                    )

                    await asyncio.sleep(15)

        return {
            "keyword": keyword,
            "engine": engine,
            "success": False,
            "results": [],
            "raw_result_count": 0,
            "attempt": max_attempts,
            "error": str(last_error),
        }






# ============================================================
# Competitor and profile pipeline
# ============================================================

async def run_serp_stage(
    keywords,
    target_domain,
):
    """
    Run traditional Google SERPs and three Google AI Mode customer
    questions concurrently.

    The existing function name is retained to minimize changes to the
    orchestration code.
    """
    global LAST_AI_MODE_DISCOVERY

    semaphore = asyncio.Semaphore(
        min(8, len(keywords))
    )

    serp_tasks = [
        run_keyword_serp_task(
            keyword=keyword,
            semaphore=semaphore,
            num_results=20,
        )
        for keyword in keywords
    ]

    # Use three distinct buyer questions for AI Mode.
    question_inputs = [
        keyword
        for keyword in keywords[:3]
    ]

    ai_mode_tasks = [
        run_ai_mode_question(
            question=question,
            question_index=index,
        )
        for index, question in enumerate(
            question_inputs,
            start=1,
        )
    ]

    serp_results, ai_mode_results = (
        await asyncio.gather(
            asyncio.gather(
                *serp_tasks
            ),
            asyncio.gather(
                *ai_mode_tasks
            ),
        )
    )

    successful_serps = [
        result
        for result in serp_results
        if result["success"]
    ]

    failed_serps = [
        result
        for result in serp_results
        if not result["success"]
    ]

    successful_ai_mode = [
        result
        for result in ai_mode_results
        if result["success"]
    ]

    if not successful_serps:
        raise BrightDataAPIError(
            "All Google SERP requests failed."
        )

    if bd_client.debug:
        for result in serp_results:
            if not result["success"]:
                bd_client.log(
                    f"SERP failed for "
                    f"{result['keyword']!r}: "
                    f"{result.get('error')}",
                    "yellow",
                )
                continue

            domains = []

            for item in result.get(
                "results",
                [],
            ):
                domain = get_root_domain(
                    item.get("domain")
                    or item.get("url")
                    or ""
                )

                if (
                    domain
                    and domain
                    not in domains
                ):
                    domains.append(domain)

            bd_client.log(
                f"SERP {result['keyword']!r}: "
                f"{len(result['results'])} results; "
                f"domains="
                f"{', '.join(domains[:8])}"
            )

        for result in ai_mode_results:
            if result["success"]:
                bd_client.log(
                    f"AI Mode question "
                    f"{result['question_index']}: "
                    f"{len(result['citations'])} citations; "
                    f"{len(result['answer'])} answer characters"
                )
            else:
                bd_client.log(
                    f"AI Mode question failed: "
                    f"{result.get('error')}",
                    "yellow",
                )

    serp_candidates = (
        aggregate_competitor_domains(
            keyword_serp_results=(
                serp_results
            ),
            target_domain=target_domain,
            total_keyword_count=len(
                keywords
            ),
        )
    )

    ai_mode_candidates = (
        build_ai_mode_source_candidates(
            question_results=(
                ai_mode_results
            ),
            target_domain=target_domain,
        )
    )

    merged_candidates = (
        merge_discovery_candidates(
            serp_candidates=(
                serp_candidates
            ),
            ai_mode_candidates=(
                ai_mode_candidates
            ),
        )
    )

    if not merged_candidates:
        raise ValueError(
            "Neither Google SERP nor Google AI Mode "
            "produced usable competitor candidates."
        )

    LAST_AI_MODE_DISCOVERY = {
        "questions": question_inputs,
        "results": ai_mode_results,
        "successful": len(
            successful_ai_mode
        ),
        "failed": (
            len(ai_mode_results)
            - len(successful_ai_mode)
        ),
        "source_candidates": [
            model_to_dict(item)
            for item in (
                ai_mode_candidates
            )
        ],
    }

    return {
        "keyword_results": (
            serp_results
        ),
        "candidates": (
            merged_candidates
        ),
        "successful": len(
            successful_serps
        ),
        "failed": len(
            failed_serps
        ),
        "ai_mode_successful": len(
            successful_ai_mode
        ),
        "ai_mode_failed": (
            len(ai_mode_results)
            - len(successful_ai_mode)
        ),
        "ai_mode_discovery": (
            LAST_AI_MODE_DISCOVERY
        ),
    }




def compact_candidate_data(
    candidates,
    limit,
):
    return [
        {
            "domain": item.domain,
            "url": item.homepage_url,
            "appearances": item.frequency,
            "best_rank": item.best_rank,
            "queries": (
                item.matched_keywords[:3]
            ),
        }
        for item in candidates[:limit]
    ]










def select_competitors_stage(
    target_brand,
    candidates,
    keywords,
):
    """
    Ask Google AI Mode once for ten genuine direct competitors and use
    the first two valid results.

    A second request is made only if the first response cannot be
    parsed or contains fewer than two valid competitors.
    """
    observed_domains = [
        candidate.domain
        for candidate in candidates[:15]
    ]

    base_prompt = f"""
Research the current market and identify the ten strongest genuine
direct competitors for this target.

TARGET

Name: {target_brand.brand_name}
Website: {target_brand.official_url}
Category: {target_brand.category}
Positioning: {shorten(target_brand.positioning, 240)}
Primary customers: {", ".join(target_brand.target_customers[:5])}
Products or services: {", ".join(target_brand.products[:6])}

BUYER SEARCHES

{json.dumps(keywords, ensure_ascii=False)}

DOMAINS OBSERVED IN GOOGLE OR AI MODE

{json.dumps(observed_domains, ensure_ascii=False)}

You may select competitors outside the observed-domain list when
current public research shows that they are stronger direct
competitors.

A direct competitor must:

1. Sell, manufacture, provide, or operate its own offering.
2. Serve substantially the same primary buyer.
3. Operate at substantially the same value-chain level.
4. Offer a substitute in the same purchase or selection decision.
5. Be something a buyer would realistically compare with the target.

Do not include:

- Customers or downstream providers
- Clinics using a manufacturer's products
- Retailers
- Marketplaces
- Directories
- Distributors or resellers
- Publishers
- Review and comparison sites
- Forums or communities
- Industry lists
- Government organizations
- Educational resources
- Suppliers that do not offer a substitute

Before returning the result, review every candidate and remove any
candidate that fails one of the five direct-competitor requirements.

Return JSON only:

{{
  "target_market_role": "target's role in the value chain",
  "target_primary_customer": "target's primary buyer",
  "competitors": [
    {{
      "rank_by_directness": 1,
      "brand_name": "canonical competitor name",
      "domain": "official root domain",
      "official_url": "official public URL",
      "candidate_market_role": "candidate role",
      "candidate_primary_customer": "candidate buyer",
      "sells_own_offering": true,
      "same_primary_customer": true,
      "same_value_chain_role": true,
      "offering_is_substitute": true,
      "is_direct_competitor": true,
      "reason": "why customers directly compare it with the target",
      "evidence": ["specific public evidence"],
      "confidence": 0.0
    }}
  ],
  "excluded_examples": [
    {{
      "name": "excluded organization",
      "candidate_type": "marketplace, clinic, publisher, or other",
      "reason": "why it is not a direct competitor"
    }}
  ]
}}

Return ten competitors ordered by directness. Return JSON without a
Markdown code fence.
""".strip()

    if len(base_prompt) > 3900:
        raise ValueError(
            f"Fast competitor prompt is too long: "
            f"{len(base_prompt)} characters."
        )

    last_error = None
    last_record = None
    last_parsed = None

    for attempt in range(
        1,
        3,
    ):
        prompt = base_prompt

        if attempt == 2:
            prompt += """

STRICT RETRY:
The previous response was missing valid JSON or valid direct
competitors. Return the JSON object directly. Do not include prose.
Verify same buyer, same value-chain role, and substitute offering for
every competitor.
""".rstrip()

        if len(prompt) > 4096:
            prompt = prompt[
                :4090
            ]

        bd_client.log(
            f"Asking Google AI Mode for "
            f"top 10 direct competitors "
            f"(attempt {attempt}/2)"
        )

        try:
            record = (
                bd_client.google_ai_mode(
                    prompt,
                    timeout_seconds=720,
                )
            )

            last_record = record

            answer = (
                bd_client.answer_text(
                    record
                )
            )

            parsed = parse_ai_json(
                answer
            )

            last_parsed = parsed

            raw_competitors = (
                parsed.get(
                    "competitors"
                )
                or parsed.get(
                    "selected_competitors"
                )
                or []
            )

            if not isinstance(
                raw_competitors,
                list,
            ):
                raw_competitors = []

            selected = []
            selected_domains = set()
            rejected = []

            ordered_competitors = sorted(
                [
                    item
                    for item
                    in raw_competitors
                    if isinstance(
                        item,
                        dict,
                    )
                ],
                key=lambda item: (
                    int(
                        item.get(
                            "rank_by_directness",
                            999,
                        )
                        or 999
                    )
                ),
            )

            for item in (
                ordered_competitors
            ):
                domain = get_root_domain(
                    item.get("domain")
                    or item.get(
                        "official_url"
                    )
                    or ""
                )

                if not domain:
                    rejected.append(
                        {
                            "domain": "",
                            "reason": (
                                "Missing official domain"
                            ),
                        }
                    )
                    continue

                if domain == get_root_domain(
                    target_brand.domain
                ):
                    continue

                sells_own_offering = (
                    normalize_boolean(
                        item.get(
                            "sells_own_offering",
                            False,
                        )
                    )
                )

                same_primary_customer = (
                    normalize_boolean(
                        item.get(
                            "same_primary_customer",
                            False,
                        )
                    )
                )

                same_value_chain_role = (
                    normalize_boolean(
                        item.get(
                            "same_value_chain_role",
                            False,
                        )
                    )
                )

                offering_is_substitute = (
                    normalize_boolean(
                        item.get(
                            "offering_is_substitute",
                            False,
                        )
                    )
                )

                is_direct = (
                    normalize_boolean(
                        item.get(
                            "is_direct_competitor",
                            False,
                        )
                    )
                )

                confidence = (
                    normalize_confidence(
                        item.get(
                            "confidence"
                        )
                    )
                )

                valid = all(
                    [
                        sells_own_offering,
                        same_primary_customer,
                        same_value_chain_role,
                        offering_is_substitute,
                        is_direct,
                        confidence >= 0.5,
                    ]
                )

                if not valid:
                    rejected.append(
                        {
                            "domain": domain,
                            "reason": (
                                item.get("reason")
                                or (
                                    "Did not pass all "
                                    "direct-competitor "
                                    "requirements"
                                )
                            ),
                        }
                    )
                    continue

                if domain in (
                    selected_domains
                ):
                    continue

                brand_name = str(
                    item.get(
                        "brand_name"
                    )
                    or domain
                ).strip()

                official_url = (
                    normalize_public_url(
                        item.get(
                            "official_url"
                        )
                        or f"https://{domain}/"
                    )
                )

                selected.append(
                    SelectedCompetitor(
                        brand_name=(
                            brand_name
                        ),
                        domain=domain,
                        official_url=(
                            official_url
                            or f"https://{domain}/"
                        ),
                        reason=str(
                            item.get("reason")
                            or ""
                        ).strip(),
                        confidence=confidence,
                    )
                )

                selected_domains.add(
                    domain
                )

                if len(selected) == 2:
                    break

            if len(selected) == 2:
                validation_results = [
                    {
                        "candidate": {
                            "domain": get_root_domain(
                                item.get("domain")
                                or item.get(
                                    "official_url"
                                )
                                or ""
                            ),
                            "brand_name": (
                                item.get(
                                    "brand_name"
                                )
                            ),
                        },
                        "validation": item,
                        "method": (
                            "single_ai_mode_"
                            "market_research"
                        ),
                    }
                    for item in (
                        ordered_competitors
                    )
                ]

                return {
                    "selected": selected,
                    "rejected": (
                        rejected
                        + parsed.get(
                            "excluded_examples",
                            [],
                        )
                    ),
                    "record": record,
                    "prompt": prompt,
                    "used_fallback": (
                        attempt > 1
                    ),
                    "shortlist": (
                        candidates[:15]
                    ),
                    "validation_results": (
                        validation_results
                    ),
                    "top_ten_competitors": (
                        ordered_competitors
                    ),
                }

            last_error = (
                f"AI Mode returned only "
                f"{len(selected)} valid direct "
                f"competitors."
            )

        except Exception as exc:
            last_error = str(exc)

            bd_client.log(
                f"Fast competitor discovery "
                f"attempt {attempt} failed: "
                f"{exc}",
                "yellow",
            )

    returned_summary = ""

    if isinstance(
        last_parsed,
        dict,
    ):
        returned_summary = shorten(
            json.dumps(
                last_parsed,
                ensure_ascii=False,
            ),
            800,
        )

    raise BrightDataAPIError(
        "Google AI Mode could not return two valid "
        "direct competitors after two attempts. "
        f"Last error: {last_error}. "
        f"Returned data: {returned_summary}"
    )






def build_profile_prompt(
    job,
    target_brand,
):
    if job["role"] == "target":
        role_instruction = (
            "This is the audited target. Describe the current "
            "offering and how it is positioned for customers."
        )

        direct_competitor = "false"

    else:
        role_instruction = (
            f"This is being evaluated as an alternative to "
            f"{target_brand.brand_name}. Focus on why a customer "
            f"might compare the two. Selection reason: "
            f"{job['reason']}"
        )

        direct_competitor = "true"

    return f"""
Analyze this current public website.

Name: {job["brand_name"]}
Website: {job["official_url"]}
Domain: {job["domain"]}

{role_instruction}

Adapt the analysis to the actual category. Relevant details may include
products, services, benefits, features, claims, ingredients,
specifications, use cases, audience, price, availability, proof,
reviews, trust signals, location, or delivery model.

Return only JSON:

{{
  "brand_name": "canonical brand, product, service, or organization",
  "official_url": "{job["official_url"]}",
  "domain": "{job["domain"]}",
  "category": "specific category",
  "positioning": "one customer-facing sentence",
  "target_customers": ["customer, user, or audience"],
  "relevant_products": ["relevant product, service, or offering"],
  "key_features": ["feature, benefit, claim, ingredient, or attribute"],
  "differentiators": ["meaningful differentiator"],
  "pricing_model": "price, price tier, pricing approach, or unknown",
  "competitor_reason": "why customers would compare it",
  "direct_competitor": {direct_competitor},
  "confidence": 0.0,
  "evidence": ["specific public evidence"]
}}

Keep lists to five items or fewer. Do not invent claims or pricing.
Return JSON only without Markdown fences.
""".strip()



def normalize_brand_profile(
    data,
    job,
):
    if not isinstance(data, dict):
        data = {}

    name = str(
        data.get("brand_name")
        or data.get("name")
        or job["brand_name"]
    ).strip()

    if not name:
        name = job["brand_name"]

    official_url = normalize_public_url(
        data.get("official_url")
        or data.get("website")
        or job["official_url"]
    )

    domain = get_root_domain(
        data.get("domain")
        or official_url
        or job["domain"]
    )

    normalized = {
        "brand_name": name,
        "official_url": (
            official_url
            or job["official_url"]
        ),
        "domain": (
            domain
            or job["domain"]
        ),
        "category": str(
            data.get("category")
            or ""
        ).strip(),
        "positioning": str(
            data.get("positioning")
            or ""
        ).strip(),
        "target_customers": (
            ensure_string_list(
                data.get(
                    "target_customers"
                )
            )[:5]
        ),
        "relevant_products": (
            ensure_string_list(
                data.get(
                    "relevant_products"
                )
                or data.get("products")
            )[:5]
        ),
        "key_features": (
            ensure_string_list(
                data.get(
                    "key_features"
                )
            )[:5]
        ),
        "differentiators": (
            ensure_string_list(
                data.get(
                    "differentiators"
                )
            )[:5]
        ),
        "pricing_model": str(
            data.get("pricing_model")
            or "unknown"
        ).strip(),
        "competitor_reason": (
            "Target company"
            if job["role"] == "target"
            else str(
                data.get(
                    "competitor_reason"
                )
                or job["reason"]
                or ""
            ).strip()
        ),
        "direct_competitor": (
            job["role"]
            == "competitor"
        ),
        "confidence": (
            normalize_confidence(
                data.get("confidence")
            )
        ),
        "evidence": (
            ensure_string_list(
                data.get("evidence")
            )[:5]
        ),
    }

    return validate_model(
        BrandProfile,
        normalized,
    )


def generate_profile_sync(
    job,
    target_brand,
):
    prompt = build_profile_prompt(
        job,
        target_brand,
    )

    try:
        record = (
            bd_client.google_ai_mode(
                prompt,
                timeout_seconds=600,
            )
        )

        parsed = parse_ai_json(
            bd_client.answer_text(record)
        )

        return {
            "status": "success",
            "job": job,
            "profile": (
                normalize_brand_profile(
                    parsed,
                    job,
                )
            ),
            "record": record,
            "prompt": prompt,
            "error": None,
            "snapshot_id": None,
        }

    except SnapshotTimeoutError as exc:
        return {
            "status": "pending",
            "job": job,
            "profile": None,
            "record": None,
            "prompt": prompt,
            "error": str(exc),
            "snapshot_id": (
                exc.snapshot_id
            ),
        }

    except Exception as exc:
        return {
            "status": "failed",
            "job": job,
            "profile": None,
            "record": None,
            "prompt": prompt,
            "error": str(exc),
            "snapshot_id": None,
        }


def recover_profile_sync(
    task_result,
):
    snapshot_id = task_result[
        "snapshot_id"
    ]

    try:
        records = (
            bd_client.wait_for_snapshot(
                snapshot_id,
                timeout_seconds=600,
            )
        )

        for record in records:
            answer = (
                bd_client.answer_text(
                    record
                )
            )

            if not answer:
                continue

            parsed = parse_ai_json(
                answer
            )

            return {
                **task_result,
                "status": "success",
                "profile": (
                    normalize_brand_profile(
                        parsed,
                        task_result["job"],
                    )
                ),
                "record": record,
                "error": None,
            }

        raise BrightDataAPIError(
            "Recovered snapshot returned "
            "no answer text."
        )

    except Exception as exc:
        return {
            **task_result,
            "status": "failed",
            "error": str(exc),
        }


def fallback_profile(
    job,
    target_brand,
):
    if job["role"] == "target":
        return BrandProfile(
            brand_name=(
                target_brand.brand_name
            ),
            official_url=(
                target_brand.official_url
            ),
            domain=target_brand.domain,
            category=target_brand.category,
            positioning=(
                target_brand.positioning
            ),
            target_customers=(
                target_brand.target_customers
            ),
            relevant_products=(
                target_brand.products
            ),
            key_features=(
                target_brand.key_features
            ),
            differentiators=(
                target_brand.differentiators
            ),
            pricing_model="unknown",
            competitor_reason=(
                "Target company"
            ),
            direct_competitor=False,
            confidence=(
                target_brand.confidence
            ),
            evidence=(
                target_brand.evidence
            ),
        )

    return BrandProfile(
        brand_name=job["brand_name"],
        official_url=job[
            "official_url"
        ],
        domain=job["domain"],
        category="",
        positioning="",
        target_customers=[],
        relevant_products=[],
        key_features=[],
        differentiators=[],
        pricing_model="unknown",
        competitor_reason=(
            job["reason"]
        ),
        direct_competitor=True,
        confidence=0.0,
        evidence=[],
    )


async def run_profile_stage(
    target_brand,
    selected_competitors,
):
    jobs = [
        {
            "role": "target",
            "brand_name": (
                target_brand.brand_name
            ),
            "official_url": (
                target_brand.official_url
            ),
            "domain": (
                target_brand.domain
            ),
            "reason": "",
        }
    ]

    jobs.extend(
        {
            "role": "competitor",
            "brand_name": (
                competitor.brand_name
            ),
            "official_url": (
                competitor.official_url
            ),
            "domain": (
                competitor.domain
            ),
            "reason": (
                competitor.reason
            ),
        }
        for competitor in (
            selected_competitors
        )
    )

    tasks = [
        asyncio.to_thread(
            generate_profile_sync,
            job,
            target_brand,
        )
        for job in jobs
    ]

    results = await asyncio.gather(
        *tasks
    )

    pending = [
        result
        for result in results
        if result["status"]
        == "pending"
    ]

    if pending:
        console.print(
            f"      Waiting for "
            f"{len(pending)} late profile "
            f"snapshot(s)..."
        )

        recovered = await asyncio.gather(
            *[
                asyncio.to_thread(
                    recover_profile_sync,
                    result,
                )
                for result in pending
            ]
        )

        recovered_by_domain = {
            item["job"]["domain"]: item
            for item in recovered
        }

        results = [
            recovered_by_domain.get(
                item["job"]["domain"],
                item,
            )
            if item["status"]
            == "pending"
            else item
            for item in results
        ]

    profiles = []

    for result in results:
        profile = result.get(
            "profile"
        )

        if profile is None:
            profile = fallback_profile(
                result["job"],
                target_brand,
            )

            result["profile"] = profile
            result["used_fallback"] = True

        else:
            result["used_fallback"] = False

        profiles.append(profile)

    # Deduplicate by role/domain.
    deduplicated = []
    seen = set()

    for profile in profiles:
        key = (
            profile.direct_competitor,
            get_root_domain(
                profile.domain
            ),
        )

        if key in seen:
            continue

        seen.add(key)
        deduplicated.append(
            profile
        )

    target_profile = next(
        (
            profile
            for profile in deduplicated
            if not profile.direct_competitor
        ),
        fallback_profile(
            jobs[0],
            target_brand,
        ),
    )

    competitor_profiles = [
        profile
        for profile in deduplicated
        if profile.direct_competitor
    ]

    return {
        "target_profile": (
            target_profile
        ),
        "competitor_profiles": (
            competitor_profiles
        ),
        "all_profiles": [
            target_profile,
            *competitor_profiles,
        ],
        "task_results": results,
        "successful": sum(
            result["status"]
            == "success"
            for result in results
        ),
        "fallbacks": sum(
            result.get(
                "used_fallback",
                False,
            )
            for result in results
        ),
    }


console.print(
    "[bold green]✓ Analysis pipeline loaded[/bold green]"
)


# ============================================================
# Company research and keyword semantic validation
# ============================================================

GENERIC_KEYWORD_PLACEHOLDERS = {
    "primary offering",
    "main offering",
    "core offering",
    "products and services",
    "product and service",
    "product options",
    "service options",
    "offering options",
    "buy products",
    "purchase products",
    "purchase services",
    "buy services",
    "best product options",
    "best service options",
    "best products",
    "best services",
    "company products",
    "company services",
    "business solutions",
    "available products",
    "available services",
}


def normalize_keyword_for_quality(
    keyword,
):
    return re.sub(
        r"\s+",
        " ",
        str(keyword or "")
        .strip()
        .lower(),
    )


def is_generic_placeholder_keyword(
    keyword,
):
    normalized = (
        normalize_keyword_for_quality(
            keyword
        )
    )

    if not normalized:
        return True

    if normalized in (
        GENERIC_KEYWORD_PLACEHOLDERS
    ):
        return True

    words = normalized.split()

    # Single generic nouns are not useful buyer searches.
    if (
        len(words) == 1
        and words[0] in {
            "products",
            "services",
            "solutions",
            "offerings",
            "options",
            "companies",
            "providers",
            "suppliers",
        }
    ):
        return True

    return False


def validate_keyword_quality(
    keywords,
):
    normalized = [
        normalize_keyword_for_quality(
            keyword
        )
        for keyword in keywords
    ]

    generic = [
        keyword
        for keyword in normalized
        if is_generic_placeholder_keyword(
            keyword
        )
    ]

    unique = list(
        dict.fromkeys(
            keyword
            for keyword in normalized
            if keyword
        )
    )

    return {
        "valid": (
            len(unique) == 8
            and not generic
        ),
        "unique_count": len(
            unique
        ),
        "generic_keywords": (
            generic
        ),
    }


def select_relevant_company_research(
    research_text,
    company_name,
    company_domain,
    audit_focus="",
    max_characters=2400,
):
    """
    Select relevant blocks from across the complete research response.
    """
    text = str(
        research_text or ""
    ).strip()

    if len(text) <= max_characters:
        return text

    company_name_lower = (
        str(company_name or "")
        .strip()
        .lower()
    )

    domain_stem = (
        get_root_domain(
            company_domain
        )
        .split(".")[0]
        .lower()
    )

    focus_terms = {
        word
        for word in re.findall(
            r"[a-z0-9]{4,}",
            str(
                audit_focus or ""
            ).lower(),
        )
    }

    useful_terms = {
        "company",
        "category",
        "market",
        "product",
        "products",
        "service",
        "services",
        "customers",
        "customer",
        "buyers",
        "audience",
        "positioning",
        "offering",
        "offerings",
        "features",
        "benefits",
        "differentiators",
        "manufacturer",
        "provider",
        "platform",
        "specializes",
        "specialization",
    }

    blocks = [
        block.strip()
        for block in re.split(
            r"\n\s*\n",
            text,
        )
        if block.strip()
    ]

    # If the response has few paragraphs, divide it into chunks.
    if len(blocks) < 4:
        blocks = [
            text[index:index + 600]
            for index in range(
                0,
                len(text),
                600,
            )
        ]

    scored_blocks = []

    for index, block in enumerate(
        blocks
    ):
        block_lower = (
            block.lower()
        )

        score = 0

        if (
            company_name_lower
            and company_name_lower
            in block_lower
        ):
            score += 20

        if (
            domain_stem
            and domain_stem
            in block_lower
        ):
            score += 12

        score += sum(
            5
            for term in focus_terms
            if term in block_lower
        )

        score += sum(
            1
            for term in useful_terms
            if term in block_lower
        )

        # Prefer evidence text over blocks consisting mainly of links.
        url_count = (
            block_lower.count(
                "http://"
            )
            + block_lower.count(
                "https://"
            )
        )

        if url_count >= 3:
            score -= 5

        scored_blocks.append(
            (
                score,
                index,
                block,
            )
        )

    scored_blocks.sort(
        key=lambda item: (
            -item[0],
            item[1],
        )
    )

    selected = []
    used_characters = 0

    for score, index, block in (
        scored_blocks
    ):
        if score <= 0 and selected:
            continue

        remaining = (
            max_characters
            - used_characters
        )

        if remaining <= 100:
            break

        selected_block = block[
            :remaining
        ]

        selected.append(
            (
                index,
                selected_block,
            )
        )

        used_characters += (
            len(selected_block)
            + 2
        )

    selected.sort(
        key=lambda item: item[0]
    )

    result = "\n\n".join(
        block
        for _, block in selected
    ).strip()

    if not result:
        result = text[
            :max_characters
        ]

    return result[
        :max_characters
    ]


# Preserve the original normalizer and remove generic placeholders
# from its output. This forces the existing keyword-completion stage
# to generate real market-specific replacements.
_original_normalize_company_intake = (
    normalize_company_intake
)


def normalize_company_intake(
    data,
    *args,
    **kwargs,
):
    intake = (
        _original_normalize_company_intake(
            data,
            *args,
            **kwargs,
        )
    )

    cleaned_keywords = []

    for item in intake.buyer_intent_keywords:
        keyword = (
            item.keyword
            if hasattr(item, "keyword")
            else str(item)
        )

        if is_generic_placeholder_keyword(
            keyword
        ):
            if bd_client.debug:
                bd_client.log(
                    f"Rejected generic keyword: "
                    f"{keyword!r}",
                    "yellow",
                )

            continue

        cleaned_keywords.append(item)

    intake.buyer_intent_keywords = (
        cleaned_keywords
    )

    return intake



# Stop before expensive SERP and candidate-validation requests when
# Stage 1 still produces unusable generic searches.
_original_run_serp_stage = (
    run_serp_stage
)


async def run_serp_stage(
    keywords,
    target_domain,
):
    """
    Run one coherent Markdown search engine plus Google AI Mode.

    If search is unavailable, AI discovery continues.
    """
    global LAST_AI_MODE_DISCOVERY
    global ACTIVE_SEARCH_ENGINE
    global ACTIVE_SEARCH_STATUS

    requested_engine = str(
        globals().get(
            "SEARCH_ENGINE",
            "auto",
        )
    ).strip().lower()

    active_engine = (
        await asyncio.to_thread(
            bd_client.choose_search_engine,
            keywords[0],
            requested_engine,
        )
    )

    question_inputs = list(
        keywords[:3]
    )

    ai_mode_tasks = [
        run_ai_mode_question(
            question=question,
            question_index=index,
        )
        for index, question in enumerate(
            question_inputs,
            start=1,
        )
    ]

    if active_engine:
        semaphore = asyncio.Semaphore(
            min(8, len(keywords))
        )

        search_tasks = [
            run_keyword_serp_task(
                keyword=keyword,
                semaphore=semaphore,
                num_results=20,
                search_engine=(
                    active_engine
                ),
            )
            for keyword in keywords
        ]

        (
            search_results,
            ai_mode_results,
        ) = await asyncio.gather(
            asyncio.gather(
                *search_tasks
            ),
            asyncio.gather(
                *ai_mode_tasks
            ),
        )

    else:
        search_results = []

        ai_mode_results = (
            await asyncio.gather(
                *ai_mode_tasks
            )
        )

    successful_searches = [
        result
        for result in search_results
        if result.get("success")
    ]

    failed_searches = [
        result
        for result in search_results
        if not result.get("success")
    ]

    successful_ai_mode = [
        result
        for result in ai_mode_results
        if result.get("success")
    ]

    if bd_client.debug:
        for result in (
            successful_searches
        ):
            domains = []

            for item in result.get(
                "results",
                [],
            ):
                domain = item.get(
                    "domain"
                )

                if (
                    domain
                    and domain
                    not in domains
                ):
                    domains.append(domain)

            bd_client.log(
                f"{active_engine.title()} "
                f"SERP {result['keyword']!r}: "
                f"{len(result['results'])} "
                f"organic results; domains="
                f"{', '.join(domains[:8])}"
            )

        for result in ai_mode_results:
            if result.get("success"):
                bd_client.log(
                    f"AI Mode question "
                    f"{result['question_index']}: "
                    f"{len(result['citations'])} "
                    f"citations; "
                    f"{len(result['answer'])} "
                    f"answer characters"
                )

    search_candidates = []

    if search_results:
        search_candidates = (
            aggregate_competitor_domains(
                keyword_serp_results=(
                    search_results
                ),
                target_domain=(
                    target_domain
                ),
                total_keyword_count=len(
                    keywords
                ),
            )
        )

    ai_mode_candidates = (
        build_ai_mode_source_candidates(
            question_results=(
                ai_mode_results
            ),
            target_domain=target_domain,
        )
    )

    merged_candidates = (
        merge_discovery_candidates(
            serp_candidates=(
                search_candidates
            ),
            ai_mode_candidates=(
                ai_mode_candidates
            ),
        )
    )

    LAST_AI_MODE_DISCOVERY = {
        "questions": question_inputs,
        "results": ai_mode_results,
        "successful": len(
            successful_ai_mode
        ),
        "failed": (
            len(ai_mode_results)
            - len(successful_ai_mode)
        ),
        "source_candidates": [
            model_to_dict(item)
            for item in (
                ai_mode_candidates
            )
        ],
    }

    return {
        "search_engine": (
            active_engine
        ),
        "search_status": (
            ACTIVE_SEARCH_STATUS
        ),
        "keyword_results": (
            search_results
        ),
        "candidates": (
            merged_candidates
        ),
        "successful": len(
            successful_searches
        ),
        "failed": len(
            failed_searches
        ),
        "ai_mode_successful": len(
            successful_ai_mode
        ),
        "ai_mode_failed": (
            len(ai_mode_results)
            - len(successful_ai_mode)
        ),
        "ai_mode_discovery": (
            LAST_AI_MODE_DISCOVERY
        ),
    }




In [ ]:
#@title 3C. Load visibility, reporting, and orchestration engine
#@markdown Internal AI visibility, evidence synthesis, checkpoints, report display, and export.
#@markdown This cell normally does not need to be edited.

# ============================================================
# Visibility and reporting
# ============================================================

# ============================================================
# AI visibility prompt
# ============================================================

def build_visibility_prompt(
    target_profile,
    keywords,
):
    category = (
        target_profile.category
        or "product or service category"
    )

    audit_focus = str(
        AUDIT_SETTINGS.get(
            "audit_focus",
            "",
        )
        or ""
    ).strip()

    keyword_text = "; ".join(
        keywords
    )

    prompt = f"""
I am evaluating alternatives in this category:

{category}

Specific need or focus:
{audit_focus or "the needs represented by the searches below"}

Customer searches and requirements:

{keyword_text}

Recommend the leading brands, products, services, providers, or
organizations a real customer should consider.

Compare them using criteria that actually matter in this category.
These may include suitability, benefits, quality, evidence, claims,
ingredients, specifications, features, price, availability,
reputation, service model, performance, ease of use, or other
category-relevant factors.

Ignore criteria that are irrelevant to this market.

Use current public web information. Provide an independent shortlist.
Do not ask follow-up questions.
""".strip()

    if len(prompt) > 4096:
        raise ValueError(
            f"AI visibility prompt is too long: "
            f"{len(prompt)} characters."
        )

    return prompt



# ============================================================
# Brand mention detection
# ============================================================

GENERIC_ALIAS_WORDS = {
    "brand",
    "company",
    "group",
    "global",
    "international",
    "official",
    "online",
    "product",
    "products",
    "service",
    "services",
    "platform",
    "store",
    "shop",
    "market",
    "marketplace",
}



def build_brand_aliases(
    profile,
):
    """
    Build conservative aliases for brands, products, services, and
    organizations across different industries.
    """
    aliases = set()

    brand_name = str(
        profile.brand_name
        or ""
    ).strip()

    brand_lower = (
        brand_name.lower()
    )

    if brand_lower:
        aliases.add(
            brand_lower
        )

        # Normalize punctuation differences such as:
        # La Roche-Posay -> la roche posay
        normalized_brand = re.sub(
            r"[^a-z0-9]+",
            " ",
            brand_lower,
        ).strip()

        if (
            normalized_brand
            and normalized_brand
            != brand_lower
        ):
            aliases.add(
                normalized_brand
            )

    root_domain = get_root_domain(
        profile.domain
    )

    domain_stem = (
        root_domain.split(".")[0]
        if root_domain
        else ""
    )

    normalized_domain_stem = re.sub(
        r"[^a-z0-9]+",
        " ",
        domain_stem.lower(),
    ).strip()

    generic_terms = {
        "amazon",
        "azure",
        "google",
        "microsoft",
        "oracle",
        "company",
        "group",
        "global",
        "international",
        "product",
        "products",
        "service",
        "services",
        "store",
        "shop",
        "official",
        "online",
    }

    if (
        len(normalized_domain_stem) >= 4
        and normalized_domain_stem
        not in generic_terms
    ):
        aliases.add(
            normalized_domain_stem
        )

    words = re.findall(
        r"[A-Za-z0-9]+",
        brand_name,
    )

    normalized_words = [
        word.lower()
        for word in words
    ]

    for index, original_word in enumerate(
        words
    ):
        word = original_word.lower()

        has_internal_capital = any(
            character.isupper()
            for character
            in original_word[1:]
        )

        looks_like_named_product = (
            has_internal_capital
            or (
                len(word) >= 6
                and word.endswith("db")
            )
            or word
            == normalized_domain_stem
        )

        if (
            word not in generic_terms
            and looks_like_named_product
        ):
            aliases.add(word)

        # Preserve distinctive multi-word product names.
        if (
            index + 1
            < len(normalized_words)
            and normalized_words[
                index + 1
            ] == "db"
            and word
            not in generic_terms
        ):
            aliases.add(
                f"{word} db"
            )

    return sorted(
        aliases,
        key=len,
        reverse=True,
    )



def find_brand_mentions(
    answer,
    profiles,
):
    answer = str(
        answer or ""
    )

    results = []

    for profile in profiles:
        aliases = build_brand_aliases(
            profile
        )

        all_positions = []
        matched_aliases = []

        for alias in aliases:
            matches = list(
                re.finditer(
                    rf"\b{re.escape(alias)}\b",
                    answer,
                    flags=re.IGNORECASE,
                )
            )

            if not matches:
                continue

            matched_aliases.append(
                alias
            )

            all_positions.extend(
                match.start()
                for match in matches
            )

        # Multiple aliases can match the same occurrence,
        # so deduplicate character positions.
        unique_positions = sorted(
            set(all_positions)
        )

        results.append(
            {
                "brand_name": (
                    profile.brand_name
                ),
                "domain": profile.domain,
                "role": (
                    "competitor"
                    if profile.direct_competitor
                    else "target"
                ),
                "mentioned": bool(
                    unique_positions
                ),
                "mention_count": len(
                    unique_positions
                ),
                "first_position": (
                    unique_positions[0]
                    if unique_positions
                    else None
                ),
                "matched_aliases": sorted(
                    set(matched_aliases)
                ),
            }
        )

    results.sort(
        key=lambda item: (
            not item["mentioned"],
            (
                item["first_position"]
                if item["first_position"]
                is not None
                else float("inf")
            ),
        )
    )

    return results


def mention_order(
    mentions,
):
    return [
        item["brand_name"]
        for item in mentions
        if item["mentioned"]
    ]


# ============================================================
# Run visibility stage
# ============================================================

async def run_visibility_stage(
    target_profile,
    all_profiles,
    keywords,
):
    """
    Measure Google AI Mode, ChatGPT, and Gemini visibility.

    Every mention record includes answer coverage:
    - Google AI Mode: appearances across successful customer questions
    - ChatGPT: zero or one measured answer
    - Gemini: zero or one measured answer
    """
    prompt = build_visibility_prompt(
        target_profile=target_profile,
        keywords=keywords,
    )

    async def run_engine(
        engine,
    ):
        started_at = time.monotonic()

        try:
            result = await asyncio.to_thread(
                bd_client.race_ai_engine,
                engine,
                prompt,
                3,
                600,
            )

            result[
                "duration_seconds"
            ] = round(
                time.monotonic()
                - started_at,
                2,
            )

            return result

        except Exception as exc:
            return {
                "engine": engine,
                "engine_name": (
                    "ChatGPT"
                    if engine == "chatgpt"
                    else "Gemini"
                ),
                "status": "failed",
                "answer": "",
                "citations": [],
                "web_search_triggered": None,
                "duration_seconds": round(
                    time.monotonic()
                    - started_at,
                    2,
                ),
                "error": str(exc),
            }

    chatgpt_result, gemini_result = (
        await asyncio.gather(
            run_engine("chatgpt"),
            run_engine("gemini"),
        )
    )

    ai_mode_discovery = (
        LAST_AI_MODE_DISCOVERY
        or {
            "results": [],
            "successful": 0,
            "failed": 0,
        }
    )

    successful_ai_answers = [
        result
        for result in (
            ai_mode_discovery.get(
                "results",
                [],
            )
        )
        if (
            result.get("success")
            and result.get("answer")
        )
    ]

    ai_mode_answers = [
        result["answer"]
        for result in (
            successful_ai_answers
        )
    ]

    ai_mode_citations = []

    for result in successful_ai_answers:
        ai_mode_citations.extend(
            result.get(
                "citations",
                [],
            )
        )

    google_ai_result = {
        "engine": "google_ai_mode",
        "engine_name": "Google AI Mode",
        "status": (
            "success"
            if ai_mode_answers
            else "failed"
        ),
        "answer": "\n\n".join(
            ai_mode_answers
        ),
        "citations": (
            ai_mode_citations
        ),
        "web_search_triggered": None,
        "question_count": len(
            ai_mode_discovery.get(
                "results",
                [],
            )
        ),
        "successful_questions": len(
            successful_ai_answers
        ),
        "failed_questions": (
            len(
                ai_mode_discovery.get(
                    "results",
                    [],
                )
            )
            - len(
                successful_ai_answers
            )
        ),
    }

    engine_results = {
        "google_ai_mode": (
            google_ai_result
        ),
        "chatgpt": (
            chatgpt_result
        ),
        "gemini": (
            gemini_result
        ),
    }

    mentions = {}

    # --------------------------------------------------------
    # Google AI Mode: coverage across individual answers
    # --------------------------------------------------------

    combined_google_mentions = (
        find_brand_mentions(
            google_ai_result.get(
                "answer",
                "",
            ),
            all_profiles,
        )
        if google_ai_result[
            "status"
        ] == "success"
        else []
    )

    google_mentions_by_domain = {
        item["domain"]: item
        for item in (
            combined_google_mentions
        )
    }

    google_coverage_total = len(
        successful_ai_answers
    )

    for profile in all_profiles:
        domain = profile.domain

        mention = (
            google_mentions_by_domain.get(
                domain,
                {
                    "brand_name": (
                        profile.brand_name
                    ),
                    "domain": domain,
                    "role": (
                        "competitor"
                        if profile.direct_competitor
                        else "target"
                    ),
                    "mentioned": False,
                    "mention_count": 0,
                    "first_position": None,
                    "matched_aliases": [],
                },
            )
        )

        answer_appearances = 0

        for question_result in (
            successful_ai_answers
        ):
            question_mentions = (
                find_brand_mentions(
                    question_result[
                        "answer"
                    ],
                    [profile],
                )
            )

            if (
                question_mentions
                and question_mentions[0][
                    "mentioned"
                ]
            ):
                answer_appearances += 1

        mention[
            "answer_appearances"
        ] = answer_appearances

        mention[
            "answer_total"
        ] = google_coverage_total

        mention[
            "answer_coverage"
        ] = (
            answer_appearances
            / google_coverage_total
            if google_coverage_total
            else 0
        )

        google_mentions_by_domain[
            domain
        ] = mention

    mentions[
        "google_ai_mode"
    ] = sorted(
        google_mentions_by_domain.values(),
        key=lambda item: (
            not item["mentioned"],
            (
                item["first_position"]
                if item[
                    "first_position"
                ] is not None
                else float("inf")
            ),
        ),
    )

    # --------------------------------------------------------
    # ChatGPT and Gemini: one winning answer each
    # --------------------------------------------------------

    for engine in (
        "chatgpt",
        "gemini",
    ):
        result = engine_results[
            engine
        ]

        if result.get(
            "status"
        ) != "success":
            mentions[engine] = []
            continue

        engine_mentions = (
            find_brand_mentions(
                result.get(
                    "answer",
                    "",
                ),
                all_profiles,
            )
        )

        for mention in engine_mentions:
            mention[
                "answer_appearances"
            ] = (
                1
                if mention[
                    "mentioned"
                ]
                else 0
            )

            mention[
                "answer_total"
            ] = 1

            mention[
                "answer_coverage"
            ] = (
                1.0
                if mention[
                    "mentioned"
                ]
                else 0.0
            )

        mentions[engine] = (
            engine_mentions
        )

    audited_domains = {
        profile.domain: (
            profile.brand_name
        )
        for profile in all_profiles
    }

    return {
        "prompt": prompt,
        "engines": engine_results,
        "mentions": mentions,
        "ai_mode_discovery": (
            ai_mode_discovery
        ),
        "audited_domains": (
            audited_domains
        ),
    }




# ============================================================
# SERP visibility metrics
# ============================================================

def calculate_serp_metrics(
    domain,
    keyword_serp_results,
):
    target_root = get_root_domain(
        domain
    )

    appearances = []

    for keyword_result in (
        keyword_serp_results
    ):
        if not keyword_result.get(
            "success"
        ):
            continue

        keyword = keyword_result[
            "keyword"
        ]

        for position, result in enumerate(
            keyword_result.get(
                "results",
                [],
            ),
            start=1,
        ):
            result_root = (
                get_root_domain(
                    result.get("domain")
                    or result.get("url")
                    or ""
                )
            )

            if result_root != target_root:
                continue

            try:
                rank = int(
                    result.get(
                        "rank",
                        position,
                    )
                )
            except Exception:
                rank = position

            appearances.append(
                {
                    "keyword": keyword,
                    "rank": rank,
                    "url": result.get(
                        "url",
                        "",
                    ),
                }
            )

            break

    ranks = [
        item["rank"]
        for item in appearances
    ]

    total_keywords = len(
        keyword_serp_results
    )

    return {
        "domain": target_root,
        "appearances": len(
            appearances
        ),
        "total_keywords": (
            total_keywords
        ),
        "coverage": (
            len(appearances)
            / total_keywords
            if total_keywords
            else 0
        ),
        "best_rank": (
            min(ranks)
            if ranks
            else None
        ),
        "average_rank": (
            round(
                sum(ranks) / len(ranks),
                2,
            )
            if ranks
            else None
        ),
        "details": appearances,
    }


def calculate_all_serp_metrics(
    profiles,
    keyword_serp_results,
):
    return {
        profile.domain: (
            calculate_serp_metrics(
                profile.domain,
                keyword_serp_results,
            )
        )
        for profile in profiles
    }


# ============================================================
# Compact, priority-preserving evidence
# ============================================================

def format_serp_metric(
    metric,
):
    appearances = metric[
        "appearances"
    ]

    total = metric[
        "total_keywords"
    ]

    if appearances == 0:
        return f"0/{total} SERPs"

    return (
        f"{appearances}/{total} SERPs, "
        f"best #{metric['best_rank']}, "
        f"avg {metric['average_rank']}"
    )


def format_engine_visibility(
    engine_name,
    result,
    mentions,
):
    if result.get(
        "status"
    ) != "success":
        return (
            f"{engine_name}: failed; "
            f"error="
            f"{shorten(result.get('error'), 80)}"
        )

    ordered = mention_order(
        mentions
    )

    absent = [
        item["brand_name"]
        for item in mentions
        if not item["mentioned"]
    ]

    target_mention = next(
        (
            item
            for item in mentions
            if item["role"] == "target"
        ),
        None,
    )

    if target_mention:
        target_coverage = (
            f"{target_mention.get('answer_appearances', 0)}/"
            f"{target_mention.get('answer_total', 0)}"
        )

        target_mentions = (
            target_mention.get(
                "mention_count",
                0,
            )
        )

    else:
        target_coverage = "0/0"
        target_mentions = 0

    return (
        f"{engine_name}: "
        f"first_appearance_order="
        f"{' > '.join(ordered) or 'none'}; "
        f"target_answer_coverage="
        f"{target_coverage}; "
        f"target_mentions="
        f"{target_mentions}; "
        f"absent="
        f"{', '.join(absent) or 'none'}; "
        f"citations="
        f"{len(result.get('citations', []))}; "
        f"web_search="
        f"{result.get('web_search_triggered')}."
    )



def build_report_evidence(
    target_profile,
    competitor_profiles,
    keywords,
    serp_metrics,
    visibility,
):
    """
    Build compact evidence while preserving all measured brands and
    answer engines.
    """
    lines = []

    lines.append(
        "TARGET:"
        f"{target_profile.brand_name}|"
        f"{target_profile.domain}|"
        f"{shorten(target_profile.category, 60)}|"
        f"{shorten(target_profile.positioning, 130)}"
    )

    target_metric = serp_metrics.get(
        target_profile.domain,
        {
            "appearances": 0,
            "total_keywords": len(
                keywords
            ),
            "best_rank": None,
            "average_rank": None,
        },
    )

    lines.append(
        "TARGET_SERP:"
        + format_serp_metric(
            target_metric
        )
    )

    lines.append(
        "TARGET_STRENGTHS:"
        + shorten(
            "; ".join(
                target_profile.differentiators[
                    :4
                ]
            ),
            180,
        )
    )

    lines.append(
        "KEYWORDS:"
        + ";".join(
            shorten(keyword, 44)
            for keyword in keywords
        )
    )

    lines.append(
        "COMPETITORS:"
    )

    for profile in (
        competitor_profiles
    ):
        metric = serp_metrics.get(
            profile.domain,
            {
                "appearances": 0,
                "total_keywords": len(
                    keywords
                ),
                "best_rank": None,
                "average_rank": None,
            },
        )

        lines.append(
            f"-{profile.brand_name}|"
            f"{profile.domain}|"
            f"{format_serp_metric(metric)}|"
            f"{shorten(profile.category, 40)}|"
            f"{shorten(profile.positioning, 65)}"
        )

    lines.append(
        "AI_VISIBILITY:"
    )

    for engine, display_name in (
        (
            "google_ai_mode",
            "Google AI Mode",
        ),
        (
            "chatgpt",
            "ChatGPT",
        ),
        (
            "gemini",
            "Gemini",
        ),
    ):
        lines.append(
            format_engine_visibility(
                engine_name=display_name,
                result=visibility[
                    "engines"
                ].get(engine, {}),
                mentions=visibility[
                    "mentions"
                ].get(engine, []),
            )
        )

    sources = (
        collect_visibility_sources(
            visibility,
            max_per_engine=10,
        )
    )

    source_type_counts = Counter(
        source["source_type"]
        for source in sources
    )

    if source_type_counts:
        source_mix = "; ".join(
            f"{source_type}={count}"
            for source_type, count
            in source_type_counts.most_common()
        )

        lines.append(
            "SOURCE_MIX:"
            + shorten(
                source_mix,
                240,
            )
        )

    evidence = "\n".join(
        lines
    )

    if len(evidence) > 2700:
        raise ValueError(
            f"Structured evidence is too long: "
            f"{len(evidence)} characters."
        )

    return evidence



# ============================================================
# Final report prompt
# ============================================================

def build_final_report_prompt(
    evidence,
):
    prompt = f"""
Using only the evidence below, write a professional Markdown
competitive visibility audit adapted to the actual market.

Required sections:

# Competitive Visibility Audit
## Executive Summary
## Competitive Landscape
## Search Visibility
## AI Answer-Engine Visibility
## Source Influence
## Positioning and Information Gaps
## Prioritized Recommendations
## Methodology and Limitations

Requirements:
- Compare the target with all supplied direct competitors.
- Never describe a retailer, marketplace, directory, publisher,
  review site, distributor, or product catalog as a competitor.
- For the measured search engine, SERP coverage is the primary visibility metric.
- Name the exact search engine from the SEARCH_ENGINE evidence field. Never describe Bing results as Google results.
- Best and average rank describe placement only when a brand appears.
- Never call a lower-coverage brand the strongest overall merely
  because it achieved one #1 result.
- Compare Google AI Mode, ChatGPT, and Gemini.
- Read SEARCH_ENGINE, SEARCH_STATUS, and SEARCH_COUNTRY from the evidence.
- If SEARCH_STATUS is unavailable, state that traditional search was not measured. Do not report zero SERP coverage as if it were measured.
- Never describe Bing results as Google results.
- Treat answer coverage as the primary AI visibility metric.
- Mention counts are supporting detail, not market share.
- Describe first appearance only as first appearance among audited
  brands, not as a formal ranking.
- Explain which source types shape the observed answers.
- Separate measurements from inference.
- Give six recommendations with Priority, Action, Evidence, and
  Expected Impact.
- Do not invent competitors, metrics, claims, specifications, prices,
  medical information, or causal relationships.
- Do not add a source appendix.

EVIDENCE
--------
{evidence}
--------
END EVIDENCE
""".strip()

    if len(prompt) > 3900:
        raise ValueError(
            f"Final report prompt is too long: "
            f"{len(prompt)} characters; maximum is 3,900."
        )

    return prompt






def generate_report_stage(
    target_profile,
    competitor_profiles,
    keywords,
    keyword_serp_results,
    visibility,
):
    all_profiles = [
        target_profile,
        *competitor_profiles,
    ]

    serp_metrics = (
        calculate_all_serp_metrics(
            profiles=all_profiles,
            keyword_serp_results=(
                keyword_serp_results
            ),
        )
    )

    evidence = build_report_evidence(
        target_profile=target_profile,
        competitor_profiles=(
            competitor_profiles
        ),
        keywords=keywords,
        serp_metrics=serp_metrics,
        visibility=visibility,
    )

    prompt = build_final_report_prompt(
        evidence
    )

    report_result = (
        bd_client.generate_chatgpt_report(
            prompt=prompt,
            timeout_seconds=600,
        )
    )

    return {
        "report": report_result[
            "answer"
        ],
        "record": report_result[
            "record"
        ],
        "snapshot_id": (
            report_result[
                "snapshot_id"
            ]
        ),
        "prompt": prompt,
        "evidence": evidence,
        "serp_metrics": (
            serp_metrics
        ),
    }


# ============================================================
# Source appendix
# ============================================================

def collect_visibility_sources(
    visibility,
    max_per_engine=10,
):
    """
    Resolve, classify, and deduplicate AI sources.
    """
    collected = []
    seen = set()

    audited_domains = (
        visibility.get(
            "audited_domains",
            {},
        )
    )

    social_domains = {
        "linkedin.com",
        "reddit.com",
        "youtube.com",
        "facebook.com",
        "instagram.com",
        "x.com",
        "twitter.com",
        "medium.com",
        "quora.com",
    }

    retailer_domains = {
        "amazon.com",
        "walmart.com",
        "target.com",
        "sephora.com",
        "ulta.com",
        "ebay.com",
        "etsy.com",
        "medicalexpo.com",
        "directindustry.com",
    }

    scientific_domains = {
        "nature.com",
        "sciencedirect.com",
        "springer.com",
        "wiley.com",
        "nih.gov",
        "bmj.com",
        "thelancet.com",
        "nejm.org",
    }

    known_market_research_domains = {
        "tracxn.com",
        "marketsandmarkets.com",
        "researchandmarkets.com",
        "mordorintelligence.com",
        "verifiedmarketresearch.com",
        "sphericalinsights.com",
        "delveinsight.com",
        "fortunebusinessinsights.com",
        "grandviewresearch.com",
    }

    known_publishers = {
        "reviewofophthalmology.com",
        "forbes.com",
        "techcrunch.com",
        "businessinsider.com",
        "allure.com",
        "vogue.com",
        "byrdie.com",
    }

    official_path_terms = (
        "/product",
        "/products",
        "/professional",
        "/professionals",
        "/media-release",
        "/press-release",
        "/press-releases",
        "/investor",
        "/news-release",
        "/our-products",
    )

    market_research_title_patterns = (
        r"\bmarket size\b",
        r"\bmarket report\b",
        r"\bmarket forecast\b",
        r"\bindustry report\b",
        r"\btop \d+ companies\b",
        r"\bleading companies\b",
        r"\bcompanies in\b",
    )

    def classify_source(
        final_url,
        title,
    ):
        parsed = urlparse(
            final_url
        )

        hostname = (
            parsed.hostname
            or ""
        ).lower().removeprefix(
            "www."
        )

        root_domain = get_root_domain(
            hostname
        )

        path = (
            parsed.path
            or ""
        ).lower()

        title_lower = (
            title
            or ""
        ).lower()

        # Most specific checks first.
        if root_domain in (
            audited_domains
        ):
            return (
                "Official audited brand"
            )

        if (
            root_domain.endswith(
                ".gov"
            )
            or hostname.endswith(
                ".gov"
            )
        ):
            return (
                "Government or regulator"
            )

        if (
            root_domain.endswith(
                ".edu"
            )
            or hostname.endswith(
                ".edu"
            )
        ):
            return (
                "Academic or educational"
            )

        if (
            root_domain
            in scientific_domains
            or any(
                hostname.endswith(
                    "." + domain
                )
                for domain
                in scientific_domains
            )
        ):
            return (
                "Scientific or professional"
            )

        if root_domain in (
            social_domains
        ):
            return (
                "Social or community"
            )

        if root_domain in (
            retailer_domains
        ):
            return (
                "Retailer or marketplace"
            )

        if root_domain in (
            known_market_research_domains
        ):
            return (
                "Market research or directory"
            )

        if any(
            re.search(
                pattern,
                title_lower,
            )
            for pattern
            in market_research_title_patterns
        ):
            return (
                "Market research or directory"
            )

        # Product, professional, investor, and press pages hosted on
        # company domains are normally official company sources.
        if any(
            term in path
            for term in (
                official_path_terms
            )
        ):
            return (
                "Official company or product"
            )

        if root_domain in (
            known_publishers
        ):
            return (
                "Publisher or editorial"
            )

        return (
            "Publisher or other source"
        )

    for engine, display_name in (
        (
            "google_ai_mode",
            "Google AI Mode",
        ),
        (
            "chatgpt",
            "ChatGPT",
        ),
        (
            "gemini",
            "Gemini",
        ),
    ):
        result = visibility[
            "engines"
        ].get(engine, {})

        engine_count = 0

        for citation in result.get(
            "citations",
            [],
        ):
            if not isinstance(
                citation,
                dict,
            ):
                continue

            raw_url = str(
                citation.get("url")
                or citation.get("link")
                or ""
            ).strip()

            if not raw_url:
                continue

            resolved_url = ""

            if (
                "resolve_google_goto_url"
                in globals()
            ):
                resolved_url = (
                    resolve_google_goto_url(
                        raw_url
                    )
                )

            final_url = (
                resolved_url
                or raw_url
            )

            if (
                "is_google_goto_url"
                in globals()
                and is_google_goto_url(
                    final_url
                )
            ):
                continue

            canonical_url = (
                canonical_source_url(
                    final_url
                )
            )

            if (
                not canonical_url
                or canonical_url in seen
            ):
                continue

            seen.add(
                canonical_url
            )

            title = str(
                citation.get("title")
                or citation.get("name")
                or citation.get("domain")
                or "Untitled source"
            ).strip()

            source_type = classify_source(
                final_url=final_url,
                title=title,
            )

            collected.append(
                {
                    "engine": (
                        display_name
                    ),
                    "title": title,
                    "url": final_url,
                    "canonical_url": (
                        canonical_url
                    ),
                    "domain": (
                        get_root_domain(
                            final_url
                        )
                    ),
                    "source_type": (
                        source_type
                    ),
                }
            )

            engine_count += 1

            if engine_count >= (
                max_per_engine
            ):
                break

    return collected






def build_source_appendix(
    sources,
):
    lines = [
        "",
        "",
        "## Observed AI Sources",
        "",
        (
            "Sources returned by the measured Google AI Mode, "
            "ChatGPT, and Gemini visibility queries."
        ),
        "",
    ]

    if not sources:
        lines.append(
            "No citation records were returned."
        )

        return "\n".join(lines)

    grouped = defaultdict(list)

    for source in sources:
        grouped[
            source["engine"]
        ].append(source)

    for engine in (
        "Google AI Mode",
        "ChatGPT",
        "Gemini",
    ):
        engine_sources = grouped.get(
            engine,
            [],
        )

        if not engine_sources:
            continue

        lines.append(
            f"### {engine}"
        )
        lines.append("")

        for index, source in enumerate(
            engine_sources,
            start=1,
        ):
            title = (
                source["title"]
                .replace("[", "")
                .replace("]", "")
            )

            lines.append(
                f"{index}. [{title}]"
                f"({source['canonical_url']})"
                f" — *{source['source_type']}*"
            )

        lines.append("")

    return "\n".join(lines)




def finalize_report(
    report,
    visibility,
):
    sources = (
        collect_visibility_sources(
            visibility
        )
    )

    appendix = (
        build_source_appendix(
            sources
        )
    )

    clean_report = (
        remove_ai_boilerplate(
            report
        )
    )

    if "## Observed AI Sources" in (
        clean_report
    ):
        complete_report = clean_report
    else:
        complete_report = (
            clean_report.rstrip()
            + appendix
        )

    return {
        "report": complete_report,
        "sources": sources,
    }


# ============================================================
# Orchestration and export
# ============================================================

# ============================================================
# File helpers
# ============================================================

def clean_record_for_storage(
    record,
):
    """
    Remove unnecessarily large UI fields before writing records.
    """
    if not isinstance(record, dict):
        return record

    excluded_fields = {
        "answer_html",
        "additional_answer_html",
        "screenshot",
        "html",
    }

    return {
        key: value
        for key, value in record.items()
        if key not in excluded_fields
    }


def write_json(
    path,
    data,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            data,
            file,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    return path


def write_text(
    path,
    text,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            str(text or "")
        )

    return path


def create_audit_zip(
    output_directory,
):
    output_directory = Path(
        output_directory
    )

    zip_path = (
        output_directory.parent
        / f"{output_directory.name}.zip"
    )

    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=(
            zipfile.ZIP_DEFLATED
        ),
    ) as archive:
        for path in (
            output_directory.rglob("*")
        ):
            if not path.is_file():
                continue

            archive.write(
                path,
                arcname=path.relative_to(
                    output_directory
                ),
            )

    return zip_path


# ============================================================
# Progress display
# ============================================================

def print_stage(
    stage_number,
    title,
):
    console.print(
        f"\n[bold cyan]"
        f"[{stage_number}/6] "
        f"{title}"
        f"[/bold cyan]"
    )


def print_stage_success(
    message,
):
    console.print(
        f"      [bold green]✓ "
        f"{message}[/bold green]"
    )


def print_stage_warning(
    message,
):
    console.print(
        f"      [bold yellow]⚠ "
        f"{message}[/bold yellow]"
    )


def format_duration(
    seconds,
):
    if seconds < 60:
        return f"{seconds:.1f}s"

    minutes = int(
        seconds // 60
    )

    remaining = int(
        seconds % 60
    )

    return (
        f"{minutes}m {remaining}s"
    )


# ============================================================
# Visibility serialization
# ============================================================

def serialize_engine_result(
    result,
):
    if not isinstance(result, dict):
        return result

    serialized = {
        key: value
        for key, value in result.items()
        if key != "record"
    }

    if isinstance(
        result.get("record"),
        dict,
    ):
        serialized["record"] = (
            clean_record_for_storage(
                result["record"]
            )
        )

    return serialized


def serialize_profile_task(
    result,
):
    return {
        "status": result.get(
            "status"
        ),
        "job": result.get(
            "job"
        ),
        "profile": (
            model_to_dict(
                result["profile"]
            )
            if result.get("profile")
            is not None
            else None
        ),
        "error": result.get(
            "error"
        ),
        "snapshot_id": result.get(
            "snapshot_id"
        ),
        "used_fallback": result.get(
            "used_fallback",
            False,
        ),
        "record": (
            clean_record_for_storage(
                result["record"]
            )
            if isinstance(
                result.get("record"),
                dict,
            )
            else None
        ),
    }


# ============================================================
# Main pipeline
# ============================================================

async def run_competitive_visibility_audit(
    settings,
):
    """
    Run the complete Competitive Visibility Audit.

    Stages:
    1. Company analysis and keyword generation
    2. Eight parallel Google SERPs
    3. Competitor validation and selection
    4. Six parallel company profiles
    5. ChatGPT and Gemini visibility
    6. Final report and export
    """
    settings = dict(settings)

    audit_started_at = (
        time.monotonic()
    )

    run_timestamp = datetime.now(
        timezone.utc
    )

    run_id = (
        f"{slugify(settings['company_name'])}"
        f"-{run_timestamp.strftime('%Y%m%d-%H%M%S')}"
    )

    output_directory = (
        Path("/content")
        / f"competitive-visibility-{run_id}"
    )

    raw_directory = (
        output_directory / "raw"
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    raw_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    stage_durations = {}
    warnings = []

    # --------------------------------------------------------
    # Stage 1: company analysis and keywords
    # --------------------------------------------------------

    print_stage(
        1,
        "Company analysis and buyer keywords",
    )

    stage_started_at = (
        time.monotonic()
    )

    company_result = await asyncio.to_thread(
        analyze_company_stage,
        settings,
    )

    company_intake = company_result[
        "intake"
    ]

    target_brand = company_intake.brand

    keyword_records = (
        company_intake.buyer_intent_keywords
    )

    keywords = [
        item.keyword
        for item in keyword_records
    ]

    stage_durations[
        "company_analysis"
    ] = (
        time.monotonic()
        - stage_started_at
    )

    print_stage_success(
        f"{target_brand.brand_name} analyzed; "
        f"{len(keywords)} buyer keywords generated"
    )

    write_json(
        output_directory
        / "01_company_analysis.json",
        {
            "created_at": (
                run_timestamp.isoformat()
            ),
            "brand": model_to_dict(
                target_brand
            ),
            "buyer_intent_keywords": [
                model_to_dict(item)
                for item in keyword_records
            ],
            "duration_seconds": round(
                stage_durations[
                    "company_analysis"
                ],
                2,
            ),
        },
    )

    write_json(
        raw_directory
        / "01_company_ai_record.json",
        clean_record_for_storage(
            company_result["record"]
        ),
    )

    if isinstance(
        company_result.get(
            "structuring_record"
        ),
        dict,
    ):
        write_json(
            raw_directory
            / "01_company_structuring_record.json",
            clean_record_for_storage(
                company_result[
                    "structuring_record"
                ]
            ),
        )

    keyword_completion = (
        company_result.get(
            "keyword_completion"
        )
    )

    if (
        isinstance(
            keyword_completion,
            dict,
        )
        and isinstance(
            keyword_completion.get(
                "record"
            ),
            dict,
        )
    ):
        write_json(
            raw_directory
            / "01_keyword_completion_record.json",
            clean_record_for_storage(
                keyword_completion[
                    "record"
                ]
            ),
        )


    # --------------------------------------------------------
    # Stage 2: Google SERPs
    # --------------------------------------------------------

    print_stage(
        2,
        "Web Search and AI Mode competitor discovery",
    )

    stage_started_at = (
        time.monotonic()
    )

    serp_result = await run_serp_stage(
        keywords=keywords,
        target_domain=(
            target_brand.domain
        ),
    )

    keyword_serp_results = (
        serp_result[
            "keyword_results"
        ]
    )

    competitor_candidates = (
        serp_result["candidates"]
    )

    stage_durations[
        "serp_discovery"
    ] = (
        time.monotonic()
        - stage_started_at
    )

    print_stage_success(
        f"{serp_result['successful']}/"
        f"{len(keywords)} searches completed "
        f"in "
        f"{format_duration(stage_durations['serp_discovery'])}"
    )

    if serp_result["failed"]:
        warning = (
            f"{serp_result['failed']} "
            f"SERP request(s) failed"
        )

        warnings.append(warning)
        print_stage_warning(warning)

    write_json(
        output_directory
        / "02_serp_results.json",
        {
            "created_at": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
            "keywords": keywords,
            "successful": (
                serp_result["successful"]
            ),
            "failed": (
                serp_result["failed"]
            ),
            "keyword_results": (
                keyword_serp_results
            ),
            "competitor_candidates": [
                model_to_dict(item)
                for item in (
                    competitor_candidates
                )
            ],
            "duration_seconds": round(
                stage_durations[
                    "serp_discovery"
                ],
                2,
            ),
        },
    )

    # --------------------------------------------------------
    # Stage 3: competitor selection
    # --------------------------------------------------------

    print_stage(
        3,
        "Direct competitor selection",
    )

    stage_started_at = (
        time.monotonic()
    )

    selection_result = (
        await asyncio.to_thread(
            select_competitors_stage,
            target_brand,
            competitor_candidates,
            keywords,
        )
    )

    selected_competitors = (
        selection_result["selected"]
    )

    stage_durations[
        "competitor_selection"
    ] = (
        time.monotonic()
        - stage_started_at
    )

    if selection_result.get(
        "used_fallback"
    ):
        warning = (
            "AI competitor validation failed; "
            "SERP-ranked fallback was used"
        )

        warnings.append(warning)
        print_stage_warning(warning)

    for competitor in (
        selected_competitors
    ):
        console.print(
            f"      ✓ "
            f"{competitor.brand_name}"
        )

    write_json(
        output_directory
        / "03_competitor_selection.json",
        {
            "created_at": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
            "selected_competitors": [
                model_to_dict(item)
                for item in (
                    selected_competitors
                )
            ],
            "rejected_candidates": (
                selection_result.get(
                    "rejected",
                    [],
                )
            ),
            "used_fallback": (
                selection_result.get(
                    "used_fallback",
                    False,
                )
            ),
            "validation_results": (
                selection_result.get(
                    "validation_results",
                    [],
                )
            ),
            "duration_seconds": round(
                stage_durations[
                    "competitor_selection"
                ],
                2,
            ),
        },
    )

    if isinstance(
        selection_result.get("record"),
        dict,
    ):
        write_json(
            raw_directory
            / "03_selection_ai_record.json",
            clean_record_for_storage(
                selection_result["record"]
            ),
        )

    # --------------------------------------------------------
    # Stage 4: profiles
    # --------------------------------------------------------

    print_stage(
        4,
        "Target and competitor profiles",
    )

    stage_started_at = (
        time.monotonic()
    )

    profile_result = (
        await run_profile_stage(
            target_brand=target_brand,
            selected_competitors=(
                selected_competitors
            ),
        )
    )

    target_profile = (
        profile_result[
            "target_profile"
        ]
    )

    competitor_profiles = (
        profile_result[
            "competitor_profiles"
        ]
    )

    all_profiles = (
        profile_result[
            "all_profiles"
        ]
    )

    stage_durations[
        "brand_profiles"
    ] = (
        time.monotonic()
        - stage_started_at
    )

    print_stage_success(
        f"{len(all_profiles)}/3 profiles available"
    )

    if profile_result[
        "fallbacks"
    ]:
        warning = (
            f"{profile_result['fallbacks']} "
            f"profile fallback(s) used"
        )

        warnings.append(warning)
        print_stage_warning(warning)

    write_json(
        output_directory
        / "04_brand_profiles.json",
        {
            "created_at": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
            "target_profile": (
                model_to_dict(
                    target_profile
                )
            ),
            "competitor_profiles": [
                model_to_dict(item)
                for item in (
                    competitor_profiles
                )
            ],
            "successful_profiles": (
                profile_result[
                    "successful"
                ]
            ),
            "fallback_profiles": (
                profile_result[
                    "fallbacks"
                ]
            ),
            "tasks": [
                serialize_profile_task(
                    result
                )
                for result in (
                    profile_result[
                        "task_results"
                    ]
                )
            ],
            "duration_seconds": round(
                stage_durations[
                    "brand_profiles"
                ],
                2,
            ),
        },
    )

    # --------------------------------------------------------
    # Stage 5: AI visibility
    # --------------------------------------------------------

    print_stage(
        5,
        "Cross-engine AI visibility",
    )

    stage_started_at = (
        time.monotonic()
    )

    visibility_result = (
        await run_visibility_stage(
            target_profile=(
                target_profile
            ),
            all_profiles=all_profiles,
            keywords=keywords,
        )
    )

    stage_durations[
        "ai_visibility"
    ] = (
        time.monotonic()
        - stage_started_at
    )

    for engine in (
        "chatgpt",
        "gemini",
    ):
        engine_result = (
            visibility_result[
                "engines"
            ][engine]
        )

        engine_name = (
            engine_result.get(
                "engine_name",
                engine.title(),
            )
        )

        if engine_result.get(
            "status"
        ) == "success":
            print_stage_success(
                f"{engine_name} completed in "
                f"{format_duration(engine_result['duration_seconds'])}"
            )
        else:
            warning = (
                f"{engine_name} visibility failed: "
                f"{engine_result.get('error', 'unknown error')}"
            )

            warnings.append(warning)
            print_stage_warning(warning)

    write_json(
        output_directory
        / "05_ai_visibility.json",
        {
            "created_at": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
            "prompt": (
                visibility_result["prompt"]
            ),
            "engines": {
                engine: (
                    serialize_engine_result(
                        result
                    )
                )
                for engine, result in (
                    visibility_result[
                        "engines"
                    ].items()
                )
            },
            "mentions": (
                visibility_result[
                    "mentions"
                ]
            ),
            "duration_seconds": round(
                stage_durations[
                    "ai_visibility"
                ],
                2,
            ),
        },
    )

    # --------------------------------------------------------
    # Stage 6: final report
    # --------------------------------------------------------

    print_stage(
        6,
        "Final report and export",
    )

    stage_started_at = (
        time.monotonic()
    )

    report_result = (
        await asyncio.to_thread(
            generate_report_stage,
            target_profile,
            competitor_profiles,
            keywords,
            keyword_serp_results,
            visibility_result,
        )
    )

    finalized_report = (
        finalize_report(
            report=report_result[
                "report"
            ],
            visibility=(
                visibility_result
            ),
        )
    )

    final_report = (
        finalized_report["report"]
    )

    final_sources = (
        finalized_report["sources"]
    )

    stage_durations[
        "final_report"
    ] = (
        time.monotonic()
        - stage_started_at
    )

    report_markdown_path = (
        output_directory
        / "06_competitive_visibility_audit.md"
    )

    write_text(
        report_markdown_path,
        final_report,
    )

    write_json(
        raw_directory
        / "06_final_report_record.json",
        clean_record_for_storage(
            report_result["record"]
        ),
    )

    print_stage_success(
        f"Report generated in "
        f"{format_duration(stage_durations['final_report'])}"
    )

    # --------------------------------------------------------
    # Final structured audit object
    # --------------------------------------------------------

    completed_at = datetime.now(
        timezone.utc
    )

    total_duration = (
        time.monotonic()
        - audit_started_at
    )

    audit_data = {
        "run_id": run_id,
        "created_at": (
            run_timestamp.isoformat()
        ),
        "completed_at": (
            completed_at.isoformat()
        ),
        "configuration": {
            "company_name": settings[
                "company_name"
            ],
            "company_url": settings[
                "company_url"
            ],
            "company_domain": settings[
                "company_domain"
            ],
            "country": settings[
                "country"
            ],
            "serp_zone": settings[
                "serp_zone"
            ],
            "debug": settings.get(
                "debug",
                False,
            ),
        },
        "target": model_to_dict(
            target_profile
        ),
        "buyer_intent_keywords": [
            model_to_dict(item)
            for item in keyword_records
        ],
        "serp": {
            "keyword_results": (
                keyword_serp_results
            ),
            "metrics": (
                report_result[
                    "serp_metrics"
                ]
            ),
            "competitor_candidates": [
                model_to_dict(item)
                for item in (
                    competitor_candidates
                )
            ],
        },
        "competitor_selection": {
            "selected": [
                model_to_dict(item)
                for item in (
                    selected_competitors
                )
            ],
            "rejected": (
                selection_result.get(
                    "rejected",
                    [],
                )
            ),
            "used_fallback": (
                selection_result.get(
                    "used_fallback",
                    False,
                )
            ),
        },
        "profiles": {
            "target": model_to_dict(
                target_profile
            ),
            "competitors": [
                model_to_dict(item)
                for item in (
                    competitor_profiles
                )
            ],
        },
        "ai_visibility": {
            "prompt": (
                visibility_result["prompt"]
            ),
            "engines": {
                engine: (
                    serialize_engine_result(
                        result
                    )
                )
                for engine, result in (
                    visibility_result[
                        "engines"
                    ].items()
                )
            },
            "mentions": (
                visibility_result[
                    "mentions"
                ]
            ),
        },
        "final_report": {
            "generator": globals().get("LAST_UTILITY_REPORT_RESULT", {}).get("engine_name", "Unknown"),
            "web_search": False,
            "snapshot_id": (
                report_result[
                    "snapshot_id"
                ]
            ),
            "evidence": (
                report_result[
                    "evidence"
                ]
            ),
            "prompt": (
                report_result[
                    "prompt"
                ]
            ),
            "markdown": final_report,
            "sources": final_sources,
        },
        "warnings": warnings,
        "durations": {
            **{
                stage: round(
                    duration,
                    2,
                )
                for stage, duration in (
                    stage_durations.items()
                )
            },
            "total_seconds": round(
                total_duration,
                2,
            ),
        },
        "files": {},
    }

    audit_json_path = (
        output_directory
        / "06_competitive_visibility_audit.json"
    )

    audit_data["files"] = {
        "output_directory": str(
            output_directory
        ),
        "markdown_report": str(
            report_markdown_path
        ),
        "json_report": str(
            audit_json_path
        ),
    }

    write_json(
        audit_json_path,
        audit_data,
    )

    zip_path = create_audit_zip(
        output_directory
    )

    audit_data["files"][
        "zip_archive"
    ] = str(zip_path)

    # Rewrite JSON so it includes the ZIP path.
    write_json(
        audit_json_path,
        audit_data,
    )

    print_stage_success(
        "Markdown, JSON and ZIP saved"
    )

    console.print(
        f"\n[bold green]"
        f"✓ Audit complete in "
        f"{format_duration(total_duration)}"
        f"[/bold green]"
    )

    return audit_data


# ============================================================
# Report display
# ============================================================

# NAN-DISPLAY-PATCH: display_clean_dataframe
def display_clean_dataframe(rows):
    """
    Display report tables without Pandas NaN values.

    - Missing values are shown as an em dash.
    - Integral floats such as 6.0 and 8156.0 are shown as 6 and 8156.
    - The underlying audit data and exported JSON are not modified.
    """
    dataframe = pd.DataFrame(rows).copy()

    def clean_display_value(value):
        if value is None:
            return "—"

        try:
            if pd.isna(value):
                return "—"
        except (TypeError, ValueError):
            pass

        if (
            isinstance(value, float)
            and value.is_integer()
        ):
            return int(value)

        return value

    for column in dataframe.columns:
        dataframe[column] = (
            dataframe[column].map(
                clean_display_value
            )
        )

    display(dataframe)


def display_audit(
    audit,
):
    target = audit["target"]

    console.print(
        "\n[bold cyan]"
        "Competitive Visibility Audit"
        "[/bold cyan]"
    )

    console.print(
        f"Company: "
        f"[bold]{target['brand_name']}[/bold]"
    )
    console.print(
        f"Website: "
        f"{target['official_url']}"
    )
    console.print(
        f"Category: "
        f"{target['category']}"
    )
    console.print(
        f"Country: "
        f"{audit['configuration']['country']}"
    )

    # --------------------------------------------------------
    # Keywords
    # --------------------------------------------------------

    console.print(
        "\n[bold cyan]Buyer-intent keywords[/bold cyan]"
    )

    keyword_rows = []

    for index, item in enumerate(
        audit[
            "buyer_intent_keywords"
        ],
        start=1,
    ):
        keyword_rows.append(
            {
                "#": index,
                "keyword": item[
                    "keyword"
                ],
                "intent": item[
                    "intent"
                ],
            }
        )

    display_clean_dataframe(keyword_rows)

    # --------------------------------------------------------
    # Competitors and SERP metrics
    # --------------------------------------------------------

    console.print(
        "\n[bold cyan]Competitive search visibility[/bold cyan]"
    )

    metrics = audit[
        "serp"
    ]["metrics"]

    profile_rows = []

    all_profile_data = [
        audit["profiles"]["target"],
        *audit["profiles"][
            "competitors"
        ],
    ]

    for profile in all_profile_data:
        metric = metrics.get(
            profile["domain"],
            {},
        )

        profile_rows.append(
            {
                "role": (
                    "competitor"
                    if profile[
                        "direct_competitor"
                    ]
                    else "target"
                ),
                "brand": profile[
                    "brand_name"
                ],
                "domain": profile[
                    "domain"
                ],
                "SERP coverage": (
                    f"{metric.get('appearances', 0)}/"
                    f"{metric.get('total_keywords', 0)}"
                ),
                "best rank": metric.get(
                    "best_rank"
                ),
                "average rank": (
                    metric.get(
                        "average_rank"
                    )
                ),
            }
        )

    display_clean_dataframe(profile_rows)

    # --------------------------------------------------------
    # AI visibility
    # --------------------------------------------------------

    console.print(
        "\n[bold cyan]AI visibility[/bold cyan]"
    )

    visibility_rows = []

    for engine, mentions in (
        audit[
            "ai_visibility"
        ]["mentions"].items()
    ):
        for mention in mentions:
            visibility_rows.append(
                {
                    "engine": {
                        "google_ai_mode": "Google AI Mode",
                        "chatgpt": "ChatGPT",
                        "gemini": "Gemini",
                    }.get(
                        engine,
                        engine.replace(
                            "_",
                            " ",
                        ).title(),
                    ),
                    "role": mention[
                        "role"
                    ],
                    "brand": mention[
                        "brand_name"
                    ],
                    "mentioned": mention[
                        "mentioned"
                    ],
                    "answer coverage": (
                        f"{mention.get('answer_appearances', 0)}/"
                        f"{mention.get('answer_total', 0)}"
                    ),
                    "mentions": mention[
                        "mention_count"
                    ],
                    "first mention offset": (
                        mention[
                            "first_position"
                        ]
                    ),
                }
            )

    display_clean_dataframe(visibility_rows)

    # --------------------------------------------------------
    # Final report
    # --------------------------------------------------------

    console.print(
        "\n[bold cyan]Final report[/bold cyan]\n"
    )

    console.print(
        Markdown(
            audit[
                "final_report"
            ]["markdown"]
        )
    )

    # --------------------------------------------------------
    # Files
    # --------------------------------------------------------

    console.print(
        "\n[bold green]Saved files[/bold green]"
    )

    console.print(
        f"Markdown: "
        f"{audit['files']['markdown_report']}"
    )
    console.print(
        f"JSON: "
        f"{audit['files']['json_report']}"
    )
    console.print(
        f"ZIP: "
        f"{audit['files']['zip_archive']}"
    )

    if audit.get("warnings"):
        console.print(
            "\n[bold yellow]Warnings[/bold yellow]"
        )

        for warning in audit[
            "warnings"
        ]:
            console.print(
                f"• {warning}"
            )


def download_audit(
    audit,
):
    from google.colab import files

    zip_path = audit.get(
        "files",
        {},
    ).get(
        "zip_archive"
    )

    if not zip_path:
        raise ValueError(
            "The audit ZIP path is missing."
        )

    if not Path(zip_path).exists():
        raise FileNotFoundError(
            f"Audit ZIP does not exist: "
            f"{zip_path}"
        )

    files.download(zip_path)


console.print(
    "[bold green]✓ Visibility, reporting, and orchestration engine loaded[/bold green]"
)


# ============================================================
# Search-engine evidence label
# ============================================================

_original_build_report_evidence_search_engine = (
    build_report_evidence
)


def build_report_evidence(
    target_profile,
    competitor_profiles,
    keywords,
    serp_metrics,
    visibility,
):
    evidence = (
        _original_build_report_evidence_search_engine(
            target_profile=target_profile,
            competitor_profiles=(
                competitor_profiles
            ),
            keywords=keywords,
            serp_metrics=serp_metrics,
            visibility=visibility,
        )
    )

    measured_engine = str(
        globals().get(
            "ACTIVE_SEARCH_ENGINE",
            "",
        )
        or globals().get(
            "SEARCH_ENGINE",
            "bing",
        )
    ).strip().lower()

    return (
        f"SEARCH_ENGINE:"
        f"{measured_engine.title()}\n"
        + evidence
    )


# ============================================================
# Markdown search evidence
# ============================================================

_previous_build_report_evidence_markdown = (
    build_report_evidence
)


def build_report_evidence(
    target_profile,
    competitor_profiles,
    keywords,
    serp_metrics,
    visibility,
):
    evidence = (
        _previous_build_report_evidence_markdown(
            target_profile=target_profile,
            competitor_profiles=(
                competitor_profiles
            ),
            keywords=keywords,
            serp_metrics=serp_metrics,
            visibility=visibility,
        )
    )

    # Remove earlier search metadata wrappers if present.
    evidence_lines = [
        line
        for line in evidence.splitlines()
        if not line.startswith(
            (
                "SEARCH_ENGINE:",
                "SEARCH_STATUS:",
                "SEARCH_COUNTRY:",
            )
        )
    ]

    engine = (
        ACTIVE_SEARCH_ENGINE
        or "unavailable"
    )

    return "\n".join(
        [
            (
                "SEARCH_ENGINE:"
                + str(engine).title()
            ),
            (
                "SEARCH_STATUS:"
                + str(
                    ACTIVE_SEARCH_STATUS
                )
            ),
            (
                "SEARCH_COUNTRY:"
                + str(
                    bd_client.country
                ).upper()
            ),
            *evidence_lines,
        ]
    )


In [ ]:
#@title 3D. Enable Gemini + ChatGPT utility race
#@markdown Gemini and ChatGPT start together for internal utility tasks.
#@markdown The first response that passes validation wins.
#@markdown This does not change cross-engine visibility measurement.

import threading

from concurrent.futures import (
    ThreadPoolExecutor as UtilityThreadPoolExecutor,
    as_completed as utility_as_completed,
    TimeoutError as UtilityFuturesTimeoutError,
)


# ============================================================
# Utility-race configuration
# ============================================================

UTILITY_AI_MODE = "race"

UTILITY_AI_ENGINES = (
    "gemini",
    "chatgpt",
)

UTILITY_RACE_POLL_SECONDS = 3

UTILITY_RACE_TIMEOUT_SECONDS = 900

UTILITY_REPORT_MIN_CHARACTERS = 800

LAST_UTILITY_AI_RESULT = None

LAST_UTILITY_REPORT_RESULT = None


# ============================================================
# Utility-race validation
# ============================================================

def validate_utility_json_answer(answer):
    """
    Accept only an answer that can be parsed as a JSON object.

    More specific normalization and Pydantic validation still happens
    in the existing downstream pipeline.
    """
    try:
        parsed = parse_ai_json(answer)

        if not isinstance(parsed, dict):
            return {
                "valid": False,
                "reason": (
                    "Parsed result was not a JSON object."
                ),
            }

        return {
            "valid": True,
            "reason": "Parseable JSON object",
            "parsed": parsed,
        }

    except Exception as exc:
        return {
            "valid": False,
            "reason": (
                f"{type(exc).__name__}: {exc}"
            ),
        }


# FLEXIBLE-REPORT-VALIDATOR-PATCH
def validate_utility_report_answer(answer):
    """
    Validate and normalize the final Markdown report.

    AI interfaces do not always preserve requested Markdown heading
    levels. This validator recognizes semantically equivalent headings
    and converts them to the canonical report structure.
    """
    cleaned = remove_ai_boilerplate(
        str(answer or "")
    ).strip()

    if len(cleaned) < 600:
        return {
            "valid": False,
            "reason": (
                "Report was too short: "
                f"{len(cleaned)} characters. "
                f"Preview: {cleaned[:300]!r}"
            ),
        }

    canonical_sections = (
        (
            "Competitive Visibility Audit",
            "# Competitive Visibility Audit",
        ),
        (
            "Executive Summary",
            "## Executive Summary",
        ),
        (
            "Competitive Landscape",
            "## Competitive Landscape",
        ),
        (
            "Search Visibility",
            "## Search Visibility",
        ),
        (
            "AI Answer-Engine Visibility",
            "## AI Answer-Engine Visibility",
        ),
        (
            "Source Influence",
            "## Source Influence",
        ),
        (
            "Positioning and Information Gaps",
            "## Positioning and Information Gaps",
        ),
        (
            "Prioritized Recommendations",
            "## Prioritized Recommendations",
        ),
        (
            "Methodology and Limitations",
            "## Methodology and Limitations",
        ),
    )

    canonical_by_name = {
        name.lower(): heading
        for name, heading
        in canonical_sections
    }

    section_names = {
        name.lower(): name
        for name, _
        in canonical_sections
    }

    def normalize_heading_candidate(line):
        """
        Convert a possible Markdown/bullet/numbered heading into plain
        semantic text for comparison.
        """
        value = str(
            line or ""
        ).strip()

        if not value:
            return ""

        # Remove blockquote and Markdown heading prefixes.
        value = re.sub(
            r"^\s*(?:>\s*)+",
            "",
            value,
        )

        value = re.sub(
            r"^\s*#{1,6}\s*",
            "",
            value,
        )

        # Remove bullet prefixes.
        value = re.sub(
            r"^\s*[-*+•–—]\s+",
            "",
            value,
        )

        # Remove numeric and alphabetic section numbering.
        value = re.sub(
            r"^\s*(?:"
            r"\d+(?:\.\d+)*"
            r"|[A-Za-z]"
            r"|[IVXLCDMivxlcdm]+"
            r")"
            r"[\.\):\-]\s+",
            "",
            value,
        )

        # Remove surrounding Markdown emphasis.
        value = value.strip()

        previous = None

        while value != previous:
            previous = value

            value = re.sub(
                r"^(?:\*\*|__|\*|_)+",
                "",
                value,
            )

            value = re.sub(
                r"(?:\*\*|__|\*|_)+$",
                "",
                value,
            )

            value = value.strip()

        # Remove a trailing colon often used for plain-text headings.
        value = value.rstrip(
            ":"
        ).strip()

        # Normalize different dash characters.
        value = (
            value
            .replace("–", "-")
            .replace("—", "-")
            .replace("‑", "-")
        )

        value = re.sub(
            r"\s+",
            " ",
            value,
        ).strip()

        return value

    def semantic_section_name(line):
        candidate = normalize_heading_candidate(
            line
        )

        candidate_lower = (
            candidate.lower()
        )

        if candidate_lower in section_names:
            return section_names[
                candidate_lower
            ]

        # Tolerate punctuation directly following the heading name.
        candidate_without_punctuation = (
            candidate_lower.rstrip(
                ".:;,-"
            ).strip()
        )

        if (
            candidate_without_punctuation
            in section_names
        ):
            return section_names[
                candidate_without_punctuation
            ]

        return None

    lines = cleaned.splitlines()

    # Some AI scraper output wraps the entire response in one Markdown
    # bullet and indents every following line by four spaces.
    first_nonempty_index = next(
        (
            index
            for index, line
            in enumerate(lines)
            if line.strip()
        ),
        None,
    )

    if first_nonempty_index is not None:
        first_line = lines[
            first_nonempty_index
        ]

        first_semantic = (
            semantic_section_name(
                first_line
            )
        )

        if (
            first_semantic
            == "Competitive Visibility Audit"
        ):
            lines[
                first_nonempty_index
            ] = (
                "# Competitive Visibility Audit"
            )

            following_nonempty = [
                line
                for line in lines[
                    first_nonempty_index + 1:
                ]
                if line.strip()
            ]

            indented_count = sum(
                line.startswith("    ")
                for line in following_nonempty
            )

            # Dedent when most of the report appears wrapped inside the
            # first Markdown list item.
            should_dedent = (
                bool(following_nonempty)
                and indented_count
                >= max(
                    2,
                    int(
                        len(following_nonempty)
                        * 0.6
                    ),
                )
            )

            if should_dedent:
                for index in range(
                    first_nonempty_index + 1,
                    len(lines),
                ):
                    if lines[index].startswith(
                        "    "
                    ):
                        lines[index] = (
                            lines[index][4:]
                        )

    normalized_lines = []

    found_sections = set()

    index = 0

    while index < len(lines):
        line = lines[index]

        semantic_name = (
            semantic_section_name(
                line
            )
        )

        if semantic_name:
            canonical_heading = (
                canonical_by_name[
                    semantic_name.lower()
                ]
            )

            # Replace bold, numbered, bullet, Setext, and incorrect
            # Markdown-level headings with the canonical heading.
            normalized_lines.append(
                canonical_heading
            )

            found_sections.add(
                semantic_name
            )

            # Remove a Setext underline directly after the heading.
            if index + 1 < len(lines):
                next_line = lines[
                    index + 1
                ].strip()

                if re.fullmatch(
                    r"(?:=+|-+)",
                    next_line,
                ):
                    index += 1

            index += 1
            continue

        normalized_lines.append(line)
        index += 1

    normalized_report = "\n".join(
        normalized_lines
    ).strip()

    # Catch canonical headings already embedded in unusual surrounding
    # formatting that line-by-line normalization may not have changed.
    for section_name, heading in (
        canonical_sections
    ):
        if re.search(
            rf"(?im)^\s*"
            rf"{re.escape(heading)}"
            rf"\s*$",
            normalized_report,
        ):
            found_sections.add(
                section_name
            )

    required_section_names = {
        name
        for name, _
        in canonical_sections
    }

    missing_sections = [
        name
        for name, _
        in canonical_sections
        if name not in found_sections
    ]

    if missing_sections:
        preview = re.sub(
            r"\s+",
            " ",
            cleaned[:500],
        ).strip()

        return {
            "valid": False,
            "reason": (
                "Missing required report sections: "
                + ", ".join(
                    missing_sections
                )
                + ". Report preview: "
                + repr(preview)
            ),
            "missing_sections": (
                missing_sections
            ),
            "found_sections": sorted(
                found_sections
            ),
        }

    # Remove immediately duplicated canonical headings.
    for _, heading in canonical_sections:
        normalized_report = re.sub(
            rf"(?im)^"
            rf"{re.escape(heading)}"
            rf"\s*\n\s*"
            rf"{re.escape(heading)}"
            rf"\s*$",
            heading,
            normalized_report,
        )

    if len(normalized_report) < 600:
        return {
            "valid": False,
            "reason": (
                "Normalized report was too short: "
                f"{len(normalized_report)} characters."
            ),
        }

    return {
        "valid": True,
        "reason": (
            "All required report sections were "
            "recognized and normalized"
        ),
        "found_sections": sorted(
            required_section_names
        ),
        "cleaned_answer": (
            normalized_report
        ),
    }


def normalize_utility_validation_result(
    validation_result,
):
    """
    Allow validators to return either a bool or a result dictionary.
    """
    if isinstance(validation_result, dict):
        return {
            **validation_result,
            "valid": bool(
                validation_result.get(
                    "valid",
                    False,
                )
            ),
        }

    return {
        "valid": bool(validation_result),
        "reason": (
            "Validator returned "
            f"{bool(validation_result)}"
        ),
    }


# ============================================================
# Provider-specific request construction
# ============================================================

def build_utility_engine_request(
    engine,
    prompt,
):
    """
    Build the correct Bright Data dataset request shape for each
    provider.

    ChatGPT supports an explicit web_search=False setting.

    The current Gemini dataset request shape does not use the ChatGPT
    web_search field. Utility prompts already instruct the model to use
    only supplied information where appropriate.
    """
    engine = str(
        engine or ""
    ).strip().lower()

    if engine == "chatgpt":
        item = {
            "url": "https://chatgpt.com/",
            "prompt": prompt,
            "country": bd_client.country,
            "index": 1,
            "web_search": False,
        }

        return {
            "engine": "chatgpt",
            "engine_name": "ChatGPT",
            "dataset_id": CHATGPT_DATASET_ID,
            "payload": [item],
        }

    if engine == "gemini":
        item = {
            "url": "https://gemini.google.com/",
            "prompt": prompt,
            "country": bd_client.country,
            "index": 1,
        }

        return {
            "engine": "gemini",
            "engine_name": "Gemini",
            "dataset_id": GEMINI_DATASET_ID,
            "payload": {
                "input": [item],
            },
        }

    raise ValueError(
        f"Unsupported utility AI engine: {engine}"
    )


# ============================================================
# Snapshot helpers
# ============================================================

def utility_snapshot_is_materializing(
    records,
):
    """
    A ready snapshot can briefly return a materialization-status record.
    """
    if (
        len(records) != 1
        or not isinstance(records[0], dict)
    ):
        return False

    status = str(
        records[0].get(
            "status",
            "",
        )
    ).strip().lower()

    return status in {
        "building",
        "collecting",
        "digesting",
        "running",
        "processing",
        "pending",
    }


def poll_utility_snapshot(
    request,
    snapshot_id,
    prompt,
    validator,
    timeout_seconds,
    stop_event,
    started_at,
):
    """
    Poll one utility snapshot until:

    - It returns a valid result.
    - It fails.
    - It times out.
    - Another engine wins and sets stop_event.
    """
    engine = request["engine"]

    engine_name = request[
        "engine_name"
    ]

    invalid_answer_reasons = []

    while not stop_event.is_set():
        elapsed = (
            time.monotonic()
            - started_at
        )

        if elapsed >= timeout_seconds:
            return {
                "engine": engine,
                "engine_name": engine_name,
                "status": "timeout",
                "snapshot_id": snapshot_id,
                "answer": "",
                "record": None,
                "validation": {
                    "valid": False,
                    "reason": (
                        f"Timed out after "
                        f"{timeout_seconds} seconds."
                    ),
                },
                "duration_seconds": round(
                    elapsed,
                    2,
                ),
                "error": (
                    f"{engine_name} utility snapshot "
                    f"timed out after "
                    f"{timeout_seconds} seconds."
                ),
            }

        status_result = (
            bd_client.snapshot_status(
                snapshot_id
            )
        )

        status = str(
            status_result.get(
                "status",
                "unknown",
            )
        ).lower()

        if status in FAILED_STATUSES:
            return {
                "engine": engine,
                "engine_name": engine_name,
                "status": "failed",
                "snapshot_id": snapshot_id,
                "answer": "",
                "record": None,
                "validation": {
                    "valid": False,
                    "reason": (
                        "Snapshot ended with status "
                        f"{status}."
                    ),
                },
                "duration_seconds": round(
                    time.monotonic()
                    - started_at,
                    2,
                ),
                "error": (
                    f"{engine_name} snapshot "
                    f"{snapshot_id} ended with "
                    f"status {status}."
                ),
            }

        if status != "ready":
            stop_event.wait(
                UTILITY_RACE_POLL_SECONDS
            )
            continue

        try:
            records = (
                bd_client.download_snapshot(
                    snapshot_id
                )
            )

        except Exception as exc:
            bd_client.log(
                f"{engine_name} utility snapshot "
                f"download failed temporarily: {exc}",
                "yellow",
            )

            stop_event.wait(
                UTILITY_RACE_POLL_SECONDS
            )
            continue

        if utility_snapshot_is_materializing(
            records
        ):
            stop_event.wait(
                UTILITY_RACE_POLL_SECONDS
            )
            continue

        answer_found = False

        for record in records:
            answer = (
                bd_client.answer_text(
                    record
                )
            )

            if not answer:
                continue

            answer_found = True

            try:
                raw_validation = validator(
                    answer
                )

                validation = (
                    normalize_utility_validation_result(
                        raw_validation
                    )
                )

            except Exception as exc:
                validation = {
                    "valid": False,
                    "reason": (
                        "Validator raised "
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    ),
                }

            if validation["valid"]:
                cleaned_answer = (
                    validation.get(
                        "cleaned_answer"
                    )
                    or answer
                )

                return {
                    "engine": engine,
                    "engine_name": engine_name,
                    "status": "success",
                    "snapshot_id": (
                        snapshot_id
                    ),
                    "answer": (
                        cleaned_answer
                    ),
                    "record": record,
                    "validation": (
                        validation
                    ),
                    "duration_seconds": round(
                        time.monotonic()
                        - started_at,
                        2,
                    ),
                    "error": None,
                }

            invalid_answer_reasons.append(
                validation.get(
                    "reason",
                    "Validation failed",
                )
            )

        if answer_found:
            return {
                "engine": engine,
                "engine_name": engine_name,
                "status": "invalid",
                "snapshot_id": snapshot_id,
                "answer": "",
                "record": None,
                "validation": {
                    "valid": False,
                    "reason": (
                        "; ".join(
                            invalid_answer_reasons[
                                -3:
                            ]
                        )
                        or "Answer failed validation."
                    ),
                },
                "duration_seconds": round(
                    time.monotonic()
                    - started_at,
                    2,
                ),
                "error": (
                    f"{engine_name} returned an "
                    "answer that failed validation."
                ),
            }

        return {
            "engine": engine,
            "engine_name": engine_name,
            "status": "invalid",
            "snapshot_id": snapshot_id,
            "answer": "",
            "record": None,
            "validation": {
                "valid": False,
                "reason": (
                    "Snapshot returned no answer text."
                ),
            },
            "duration_seconds": round(
                time.monotonic()
                - started_at,
                2,
            ),
            "error": (
                f"{engine_name} snapshot returned "
                "no answer text."
            ),
        }

    return {
        "engine": engine,
        "engine_name": engine_name,
        "status": "stopped",
        "snapshot_id": snapshot_id,
        "answer": "",
        "record": None,
        "validation": {
            "valid": False,
            "reason": (
                "Another utility engine won."
            ),
        },
        "duration_seconds": round(
            time.monotonic()
            - started_at,
            2,
        ),
        "error": (
            "Polling stopped because another "
            "utility engine won."
        ),
    }


# ============================================================
# First-valid-response utility race
# ============================================================

def race_utility_ai(
    prompt,
    validator=None,
    timeout_seconds=UTILITY_RACE_TIMEOUT_SECONDS,
    task_name="utility task",
):
    """
    Trigger Gemini and ChatGPT concurrently and return the first valid
    response.

    The first completed response does not automatically win. A response
    wins only after passing validator(answer).
    """
    global LAST_UTILITY_AI_RESULT

    prompt = str(
        prompt or ""
    ).strip()

    if not prompt:
        raise ValueError(
            "Utility AI prompt cannot be empty."
        )

    if len(prompt) > 4096:
        raise ValueError(
            f"Utility AI prompt is too long: "
            f"{len(prompt)} characters."
        )

    if validator is None:
        validator = (
            validate_utility_json_answer
        )

    started_at = time.monotonic()

    requests_by_engine = {
        engine: (
            build_utility_engine_request(
                engine=engine,
                prompt=prompt,
            )
        )
        for engine in UTILITY_AI_ENGINES
    }

    snapshot_ids = {}

    trigger_errors = {}

    bd_client.log(
        f"Starting Gemini + ChatGPT race "
        f"for {task_name}"
    )

    # Trigger both requests concurrently before polling. This makes the
    # race fair and ensures both snapshot IDs are captured.
    with UtilityThreadPoolExecutor(
        max_workers=len(
            requests_by_engine
        )
    ) as trigger_executor:
        trigger_futures = {
            trigger_executor.submit(
                bd_client.trigger_dataset,
                request["dataset_id"],
                request["payload"],
            ): engine
            for engine, request
            in requests_by_engine.items()
        }

        for future in utility_as_completed(
            trigger_futures
        ):
            engine = trigger_futures[
                future
            ]

            try:
                snapshot_id = (
                    future.result()
                )

                snapshot_ids[
                    engine
                ] = snapshot_id

                bd_client.log(
                    f"{requests_by_engine[engine]['engine_name']} "
                    f"utility snapshot: {snapshot_id}"
                )

            except Exception as exc:
                trigger_errors[
                    engine
                ] = (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )

                bd_client.log(
                    f"{requests_by_engine[engine]['engine_name']} "
                    f"utility trigger failed: {exc}",
                    "yellow",
                )

    if not snapshot_ids:
        raise BrightDataAPIError(
            "Both Gemini and ChatGPT utility "
            "triggers failed. "
            + json.dumps(
                trigger_errors,
                ensure_ascii=False,
            )
        )

    stop_event = threading.Event()

    poll_executor = (
        UtilityThreadPoolExecutor(
            max_workers=len(
                snapshot_ids
            )
        )
    )

    poll_futures = {
        poll_executor.submit(
            poll_utility_snapshot,
            requests_by_engine[engine],
            snapshot_id,
            prompt,
            validator,
            timeout_seconds,
            stop_event,
            started_at,
        ): engine
        for engine, snapshot_id
        in snapshot_ids.items()
    }

    completed_results = []

    winner = None

    try:
        for future in utility_as_completed(
            poll_futures,
            timeout=timeout_seconds + 30,
        ):
            engine = poll_futures[
                future
            ]

            try:
                result = future.result()

            except Exception as exc:
                result = {
                    "engine": engine,
                    "engine_name": (
                        requests_by_engine[
                            engine
                        ][
                            "engine_name"
                        ]
                    ),
                    "status": "failed",
                    "snapshot_id": (
                        snapshot_ids.get(
                            engine
                        )
                    ),
                    "answer": "",
                    "record": None,
                    "validation": {
                        "valid": False,
                        "reason": (
                            f"{type(exc).__name__}: "
                            f"{exc}"
                        ),
                    },
                    "duration_seconds": round(
                        time.monotonic()
                        - started_at,
                        2,
                    ),
                    "error": str(exc),
                }

            completed_results.append(
                result
            )

            if (
                result.get("status")
                == "success"
                and result.get(
                    "validation",
                    {},
                ).get("valid")
            ):
                winner = result

                stop_event.set()

                break

    except UtilityFuturesTimeoutError:
        stop_event.set()

    finally:
        # Do not wait for the losing polling worker. It will observe the
        # stop event and exit, normally within the polling interval.
        poll_executor.shutdown(
            wait=False,
            cancel_futures=True,
        )

    if winner is None:
        failure_summary = [
            {
                "engine": result.get(
                    "engine_name"
                ),
                "status": result.get(
                    "status"
                ),
                "reason": result.get(
                    "validation",
                    {},
                ).get(
                    "reason"
                ),
                "error": result.get(
                    "error"
                ),
            }
            for result in completed_results
        ]

        for engine, error in (
            trigger_errors.items()
        ):
            failure_summary.append(
                {
                    "engine": (
                        requests_by_engine[
                            engine
                        ][
                            "engine_name"
                        ]
                    ),
                    "status": (
                        "trigger_failed"
                    ),
                    "reason": error,
                    "error": error,
                }
            )

        raise BrightDataAPIError(
            "Neither Gemini nor ChatGPT returned "
            f"a valid result for {task_name}. "
            + json.dumps(
                failure_summary,
                ensure_ascii=False,
            )
        )

    winner["task_name"] = task_name

    winner["all_snapshot_ids"] = {
        engine: snapshot_ids.get(
            engine
        )
        for engine in (
            UTILITY_AI_ENGINES
        )
    }

    winner["trigger_errors"] = (
        trigger_errors
    )

    winner["race_duration_seconds"] = round(
        time.monotonic()
        - started_at,
        2,
    )

    LAST_UTILITY_AI_RESULT = winner

    bd_client.log(
        f"{winner['engine_name']} won the "
        f"{task_name} race in "
        f"{winner['race_duration_seconds']}s"
    )

    return winner


# ============================================================
# Compatibility wrapper for existing JSON utility calls
# ============================================================

def run_chatgpt_without_web(
    prompt,
    timeout_seconds=900,
):
    """
    Compatibility name retained for the existing pipeline.

    Despite the historical function name, this now races Gemini and
    ChatGPT and returns the first parseable JSON object.
    """
    return race_utility_ai(
        prompt=prompt,
        validator=(
            validate_utility_json_answer
        ),
        timeout_seconds=timeout_seconds,
        task_name=(
            "JSON transformation"
        ),
    )


# ============================================================
# Raced final-report generator
# ============================================================

def generate_raced_utility_report(
    self,
    prompt,
    timeout_seconds=600,
):
    """
    Generate the final report by racing Gemini and ChatGPT.

    This method intentionally replaces the old ChatGPT-specific method
    while retaining its name at the call site for compatibility.
    """
    global LAST_UTILITY_REPORT_RESULT

    result = race_utility_ai(
        prompt=prompt,
        validator=(
            validate_utility_report_answer
        ),
        timeout_seconds=timeout_seconds,
        task_name="final report generation",
    )

    cleaned_answer = (
        remove_ai_boilerplate(
            result["answer"]
        )
    )

    report_result = {
        "engine": result[
            "engine"
        ],
        "engine_name": result[
            "engine_name"
        ],
        "snapshot_id": result[
            "snapshot_id"
        ],
        "all_snapshot_ids": result.get(
            "all_snapshot_ids",
            {},
        ),
        "answer": cleaned_answer,
        "record": result[
            "record"
        ],
        "validation": result.get(
            "validation",
            {},
        ),
        "duration_seconds": result.get(
            "race_duration_seconds",
            result.get(
                "duration_seconds"
            ),
        ),
    }

    LAST_UTILITY_REPORT_RESULT = (
        report_result
    )

    return report_result


# Preserve the existing call path while changing its implementation.
BrightDataClient.generate_chatgpt_report = (
    generate_raced_utility_report
)


console.print(
    "[bold green]✓ Gemini + ChatGPT utility race enabled[/bold green]"
)

console.print(
    "Utility behavior: first valid response wins"
)

console.print(
    "Visibility behavior: Google AI Mode, ChatGPT, and Gemini "
    "remain independently measured"
)


In [ ]:
#@title 4. Run Competitive Visibility Audit
#@markdown Run this cell to execute the complete live audit.

# Synchronize the current form settings with the API client.
DEBUG_MODE = bool(
    AUDIT_SETTINGS.get(
        "debug",
        False,
    )
)

bd_client.debug = DEBUG_MODE
bd_client.country = (
    AUDIT_SETTINGS["country"].upper()
)
bd_client.serp_zone = (
    AUDIT_SETTINGS["serp_zone"]
)

console.print(
    f"[bold cyan]Debug logging: "
    f"{'enabled' if DEBUG_MODE else 'disabled'}"
    f"[/bold cyan]"
)

AUDIT_RESULT = await run_competitive_visibility_audit(
    AUDIT_SETTINGS
)

display_audit(
    AUDIT_RESULT
)

if AUDIT_SETTINGS.get(
    "auto_download",
    False,
):
    download_audit(
        AUDIT_RESULT
    )
